# 第 6 步：從爬蟲輸出到釋出資料集

這份 notebook 把本 repository 第 5 步寫出的 `HPC_15to25_v2.csv` 轉成釋出資料集。輸入檔是 Google Patents 以 `HPC` 檢索得到的 21,947 件美國核准專利，申請年 2015 至 2025 年。整個流程在同一個工作階段內完成：移除 HPC 指其他意思的紀錄、套用語料規則、由語言模型寫出三層濃縮摘要、向量與主題、權屬證據、技術驗證，最後寫出釋出檔。

| | |
|---|---|
| 輸入 | `HPC_15to25_v2.csv`（第 5 步，33 欄）與 USPTO Patent Litigation Docket Reports 的 `patents.dta` |
| 輸出 | `SAVE_DIR/release/`（資料集本身）、`SAVE_DIR/figures/`、`SAVE_DIR/tables/`，以及各步驟的工作檔 |
| 硬體 | 生成步驟需要一張放得下 bf16 的 Qwen2.5-14B-Instruct 的 GPU（權重 29.5 GB；釋出版本用 A100 80 GB）；沿用先前執行（`REUSE_DIR`）時不載入生成模型，最大的模型是 BERTScore 用的 `microsoft/deberta-xlarge-mnli` |

| 節 | 步驟 | 筆數 | GPU |
|---|---|---|---|
| 1 | 環境、設定、預檢、沿用先前的輸出 | | |
| 2 | 撞字篩選：移除 HPC 不是高效能運算的紀錄 | 21,947 → 16,434 | 否 |
| 3 | 語料規則：五個段落齊全、全文出現展開語 | 16,434 → 7,158 → 6,535 | 否 |
| 4 | 分面：每筆三段短摘要（Llama-3.1-8B-Instruct，4-bit） | 6,535 | 是 |
| 5 | 三個摘要層：synthesized、direct-brief、direct-full（Qwen2.5-14B-Instruct） | 6,535 | 是 |
| 6 | 向量（SPECTER2）、分群（UMAP、HDBSCAN）與合併主題 | 6,535 | 部分 |
| 7 | 權屬證據與使用示範 | 6,535 | 否 |
| 8 | 摘要層的技術驗證 | 6,535 | 部分 |
| 9 | 釋出檔與計算成本 | | 否 |
| 10 | 執行情況報告 | | 否 |

### 執行方式

所有設定都在 1.3 這一格，其他格不需要改。

- **完整版，沿用釋出版本的生成結果**（Colab 預設）。`REUSE_DIR` 指向產生釋出資料集的那次執行的輸出資料夾。1.7 會把該次的生成結果複製過來，所以不會載入生成模型，其餘步驟全部重新計算。這些重算的步驟在釋出版本的執行時間紀錄中共 51 分鐘（A100 80 GB）；加上載入模型與寫檔，估計用一張 L4 或 A100 約需 1 到 1.5 小時。
- **完整版，全部重新生成。** 把 `REUSE_DIR` 設為 `None`。facet 與三個摘要層的生成在一張 A100 80 GB 上約需 20 小時（釋出版本 `runtime_log_release.csv` 中四個生成步驟合計 72,682 秒）。
- **抽樣版。** `step6_patentsum_pipeline_sample.ipynb` 用同一份程式，對隨機抽出的 400 筆輸入紀錄（固定種子）跑一遍。

在 Colab 上：從 GitHub 開啟 notebook（repository README 裡的 Colab 連結）或自行上傳，選 GPU 執行階段，全部執行；1.3 會掛載 Google Drive。在本機上：在 `colab/` 啟動 Jupyter。本機路徑也可以用環境變數 `HPC_V2_CSV`、`HPC_LITIG_DIR`、`HPC_STEP6_DIR`、`HPC_REUSE_DIR` 指定。每個耗時的步驟都會寫 checkpoint，中斷之後直接全部重跑即可：已完成的步驟會在幾秒到幾分鐘內讀回。

prompt、判定規則與所有模型設定都和釋出資料集相同。第 10 節會把完整版的關鍵數字和釋出值逐項對照。

## 1  環境與設定

1.1 安裝 Colab 沒有的套件；1.2 匯入套件並列出裝置。1.3 放所有路徑與執行設定：版本（完整版或抽樣版）、輸入檔、訴訟資料夾、輸出資料夾 `SAVE_DIR` 與沿用資料夾 `REUSE_DIR`。1.4 盡量在不詢問的情況下找到 Hugging Face 權杖。1.5 在幾秒內檢查套件、輸入欄位與訴訟檔是否齊全，缺了就立刻停下，免得跑了幾小時才失敗。1.6 定義共用的輔助函式：釋放 GPU 記憶體、執行時間紀錄、每一格的實際耗時、家族分組切分，以及圖表樣式。

In [ ]:
# ── 1.1  Install the dependencies (Colab or a new environment) ──────────────
import platform
%pip install -q "transformers>=4.45.0" accelerate sentencepiece
%pip install -q adapters umap-learn hdbscan bert-score minisom
%pip install -q tqdm pandas scikit-learn matplotlib seaborn plotly
%pip install -q sentence-transformers rouge-score pyarrow
if platform.system() == "Linux":
    %pip install -q bitsandbytes          # 4-bit loading of the facet model (needs a CUDA GPU)

In [ ]:
# ── 1.2  Imports and environment check ──────────────────────────────────────
import os, sys, gc, re, json, ast, platform, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.core.interactiveshell import InteractiveShell

warnings.filterwarnings("ignore")

# Show every bare expression of a cell, not only the last one, so that one cell can show several tables.
# Calls that are made only for a side effect are written as `_ = call(...)`.
InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

IN_COLAB = "google.colab" in sys.modules
HAS_CUDA = torch.cuda.is_available()
HAS_MPS  = (not HAS_CUDA) and torch.backends.mps.is_available()
DEVICE   = "cuda" if HAS_CUDA else ("mps" if HAS_MPS else "cpu")

print(f"Run environment : {'Google Colab' if IN_COLAB else 'local computer'} ({platform.system()} {platform.machine()})")
print(f"Python          : {sys.version.split()[0]}")
print(f"PyTorch         : {torch.__version__}")
print(f"pandas          : {pd.__version__}")
print(f"Device          : {DEVICE}")
if HAS_CUDA:
    print(f"GPU             : {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("No CUDA GPU. Sections 4 and 5 can only reuse existing outputs or use the small stand-in model (sample profile).")

In [ ]:
# ── 1.3  Configuration (change only this cell) ──────────────────────────────
# PROFILE "full"   : the whole input file (21,947 records).
# PROFILE "sample" : a seeded random subset of SAMPLE_N input records, for a test run of every cell.
PROFILE  = "full"
HF_TOKEN = ""            # Hugging Face token; leave empty to use the Colab secret HF_TOKEN, the environment
                         # variable HF_TOKEN or a prompt (needed only to download meta-llama/Llama-3.1-8B-Instruct)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR  = "/content/drive/MyDrive/patent_HPC"
    DATA_PATH = f"{BASE_DIR}/HPC_15to25_v2.csv"                # output of step 5 of the crawler
    LITIG_DIR = f"{BASE_DIR}/USPTO_litigation"                 # USPTO Patent Litigation Docket Reports (patents.dta)
    SAVE_DIR  = f"{BASE_DIR}/run_step6_{PROFILE}"
    # Output folder of an earlier run whose generated layers are reused (section 1.7); None generates everything.
    REUSE_DIR = f"{BASE_DIR}/run_v3_4" if PROFILE == "full" else None
else:
    REPO_DIR  = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "colab" else os.getcwd()
    DATA_PATH = os.environ.get("HPC_V2_CSV",    os.path.join(REPO_DIR, "data", "processed", "HPC_15to25_v2.csv"))
    LITIG_DIR = os.environ.get("HPC_LITIG_DIR", os.path.join(REPO_DIR, "data", "external", "USPTO_litigation"))
    SAVE_DIR  = os.environ.get("HPC_STEP6_DIR", os.path.join(REPO_DIR, "data", "step6", PROFILE))
    REUSE_DIR = os.environ.get("HPC_REUSE_DIR") or None

CLEAN_PATH = os.path.join(SAVE_DIR, "HPC_15to25_clean.csv")   # written by section 2, read by sections 3-9
FIG_DIR    = os.path.join(SAVE_DIR, "figures")
TBL_DIR    = os.path.join(SAVE_DIR, "tables")
PROV_DIR   = os.path.join(SAVE_DIR, "provenance")
for _d in (SAVE_DIR, FIG_DIR, TBL_DIR, PROV_DIR):
    os.makedirs(_d, exist_ok=True)

RANDOM_STATE = 42
MIN_LEN      = 100       # a section shorter than this number of characters is treated as missing

# ── Scale-dependent settings (the "full" values are the settings of the released dataset) ──
if PROFILE == "full":
    SAMPLE_N = None
    HDBSCAN_MIN_CLUSTER_SIZE, HDBSCAN_MIN_SAMPLES = 30, 10
    N_TARGET, MIN_CLASS_N = 25, 30          # merged topics; minimum records per merged topic
    MIN_LABEL_N, MIN_ASSIGNEE_N, N_BOOT = 20, 20, 1000
else:
    SAMPLE_N = 400                          # input records drawn at random from DATA_PATH
    HDBSCAN_MIN_CLUSTER_SIZE, HDBSCAN_MIN_SAMPLES = 8, 3
    N_TARGET, MIN_CLASS_N = 6, 5
    MIN_LABEL_N, MIN_ASSIGNEE_N, N_BOOT = 3, 3, 200

# ── Generation models ──
# The sample profile can replace both generation models by one small instruction model. That only tests that
# every cell runs; its text is not comparable with the released layers. "auto" uses the real models on a GPU
# with at least 32 GB (Qwen2.5-14B in bf16 needs about 30 GB) and the small model otherwise.
SAMPLE_MODELS = "auto"   # sample profile only: "auto", "real" or "small"
_GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if HAS_CUDA else 0.0
SMALL_MODELS = (PROFILE == "sample") and (SAMPLE_MODELS == "small" or (SAMPLE_MODELS == "auto" and _GPU_GB < 32))
if SMALL_MODELS:
    S1_MODEL_ID   = "Qwen/Qwen2.5-0.5B-Instruct"
    S2_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
    S1_4BIT, GEN_DTYPE = False, torch.float32
    S1_GPU_BATCH, S2_GPU_BATCH, BASE_BATCH, MATCHED_BATCH = 8, 8, 8, 4
else:
    S1_MODEL_ID   = "meta-llama/Llama-3.1-8B-Instruct"   # facets, 4-bit NF4
    S2_MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"          # synthesis, direct layers and topic labels, bf16
    S1_4BIT, GEN_DTYPE = True, torch.bfloat16
    # Batch sizes of the released run (A100 80 GB), halved on a smaller GPU: Qwen2.5-14B takes 29.5 GB in bf16,
    # and its cache needs about 0.2 MB per token, so 8 direct-full prompts of 5,000 tokens add another 8 GB.
    if _GPU_GB >= 70:
        S1_GPU_BATCH, S2_GPU_BATCH, BASE_BATCH, MATCHED_BATCH = 32, 24, 16, 8
    else:
        S1_GPU_BATCH, S2_GPU_BATCH, BASE_BATCH, MATCHED_BATCH = 16, 12, 8, 4

np.random.seed(RANDOM_STATE)
_ = torch.manual_seed(RANDOM_STATE)

print(f"PROFILE   : {PROFILE}" + (f" (SAMPLE_N = {SAMPLE_N} input records)" if SAMPLE_N else ""))
print(f"DATA_PATH : {DATA_PATH}  (exists: {os.path.exists(DATA_PATH)})")
print(f"LITIG_DIR : {LITIG_DIR}  (exists: {os.path.isdir(LITIG_DIR)})")
print(f"SAVE_DIR  : {SAVE_DIR}")
print(f"REUSE_DIR : {REUSE_DIR}" + ("" if REUSE_DIR is None else f"  (exists: {os.path.isdir(REUSE_DIR)})"))
print(f"Models    : facets {S1_MODEL_ID} | synthesis {S2_MODEL_NAME}"
      + ("  [small stand-in model: test run only]" if SMALL_MODELS else ""))

In [ ]:
# ── 1.4  Hugging Face token ─────────────────────────────────────────────────
# Only meta-llama/Llama-3.1-8B-Instruct (section 4) is a gated model; the other models download without a token.
# The token is looked up in this order: HF_TOKEN of 1.3, the Colab secret HF_TOKEN, the environment variable
# HF_TOKEN, an earlier `huggingface-cli login`. A prompt appears only when section 4 must download the gated model.
from huggingface_hub import login, get_token
from getpass import getpass

def _find_hf_token():
    if HF_TOKEN:
        return HF_TOKEN, "HF_TOKEN in 1.3"
    if IN_COLAB:
        try:
            from google.colab import userdata
            _t = userdata.get("HF_TOKEN")
            if _t:
                return _t, "Colab secret"
        except Exception:
            pass
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "environment variable"
    _t = get_token()
    if _t:
        return _t, "cached login"
    return None, None

def ensure_hf_login():
    """Log in before a gated model is downloaded; ask for the token only if no source has one."""
    token, source = _find_hf_token()
    if token is None:
        token, source = getpass("Hugging Face token: "), "prompt"
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
    print(f"Hugging Face login: token from {source}")

_tok, _src = _find_hf_token()
if _tok:
    ensure_hf_login()
else:
    print("No Hugging Face token found. That is fine unless section 4 has to generate facets with the gated model;"
          " in that case section 4.1 asks for the token.")

In [ ]:
# ── 1.5  Preflight: report missing packages and input columns before the long stages start ──
# Note: importlib.util.find_spec() fails for a module that is already imported and has __spec__ None
# (lazy modules such as adapters), so a loaded module is taken as available directly.
import importlib.util

def _module_available(name):
    if name in sys.modules:
        return sys.modules[name] is not None
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError, AttributeError):
        return False

REQUIRED_MODULES = {          # module -> first section that uses it
    "transformers": "4.1", "accelerate": "4.1", "sentencepiece": "4.1",
    "adapters": "6.1", "umap": "6.4", "hdbscan": "6.4", "minisom": "7.10",
    "sentence_transformers": "5.9", "bert_score": "8.14", "rouge_score": "8.4",
    "sklearn": "7.11", "scipy": "7.9", "joblib": "7.11", "pyarrow": "6.13",
    "matplotlib": "2.2", "seaborn": "7.9", "plotly": "7.6",
}
if S1_4BIT and HAS_CUDA:                  # 4-bit loading needs bitsandbytes and a CUDA GPU
    REQUIRED_MODULES["bitsandbytes"] = "4.1"
REQUIRED_COLS = [             # columns of the crawler output (step 5) that this notebook reads
    "patent_id", "title", "abstract", "BACKGROUND", "SUMMARY", "DETAILED DESCRIPTION",
    "claims_text", "claim_count", "main_ipc_4", "main_ipc_description_4", "ipc_description",
    "current_assignee", "assignee", "reassignment_assignee", "application_year", "application_date",
    "applications_claiming_priority", "legal_events", "citation_count", "citations",
]

_missing = [f"{m} (section {sec})" for m, sec in REQUIRED_MODULES.items() if not _module_available(m)]
_have = list(pd.read_csv(DATA_PATH, nrows=0).columns) if os.path.exists(DATA_PATH) else []
_lack = [c for c in REQUIRED_COLS if c not in _have]
_dta  = [p for p in (f"{LITIG_DIR}/patents.dta.zip", f"{LITIG_DIR}/patents.dta") if os.path.exists(p)]

print(f"Packages     : {len(REQUIRED_MODULES) - len(_missing)}/{len(REQUIRED_MODULES)} available")
for _m in _missing:
    print(f"  missing {_m}")
print(f"Input file   : {'found' if _have else 'NOT FOUND'} ({DATA_PATH})")
print(f"Input columns: {len(REQUIRED_COLS) - len(_lack)}/{len(REQUIRED_COLS)} present")
for _c in _lack:
    print(f"  missing column {_c}")
print(f"Docket file  : {_dta[0] if _dta else 'NOT FOUND in ' + LITIG_DIR}")
if _missing or _lack or not _have or not _dta:
    raise RuntimeError("Preflight failed: add the items listed above, then run this cell again.")
print("\nPreflight passed.")

In [ ]:
# ── 1.6  Helper functions: memory release, run-time log, family split, class level, figure style ──
import time
from contextlib import contextmanager
from IPython import get_ipython

def free_gpu(*var_names):
    """Delete the named global variables (if they exist) and empty the GPU cache.
    Called between stages so that only one large model is in GPU memory at a time."""
    g = globals()
    for name in var_names:
        if name in g:
            del g[name]
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU memory: allocated {torch.cuda.memory_allocated()/1e9:.2f} GB | "
              f"reserved {torch.cuda.memory_reserved()/1e9:.2f} GB | "
              f"free {total - torch.cuda.memory_reserved()/1e9:.2f} GB")
    elif HAS_MPS:
        torch.mps.empty_cache()

# ── Run-time log ─────────────────────────────────────────────────────────────
# Every expensive step is wrapped in `with timed("<section>  <name>"):`. The record is appended to
# SAVE_DIR/runtime_log.csv as soon as the step ends, so an interrupted session loses nothing; a rerun of
# a step replaces its own record. Section 9.2 turns the log into the cost table of the release.
RUNTIME_PATH = os.path.join(SAVE_DIR, "runtime_log.csv")
RUNTIME_LOG  = (pd.read_csv(RUNTIME_PATH).to_dict("records")
                if os.path.exists(RUNTIME_PATH) else [])

def _hardware_tag():
    if HAS_CUDA:
        p = torch.cuda.get_device_properties(0)
        return f"GPU {p.name} {p.total_memory/1e9:.0f}GB"
    return f"{'MPS' if HAS_MPS else 'CPU'} {platform.machine()} x{os.cpu_count()}"

def _hms(sec):
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h:d}:{m:02d}:{s:02d}"

HARDWARE = _hardware_tag()

class _Step:
    """Record made by timed(). Set n_items/note inside the with block; read .seconds after it."""
    def __init__(self, name, n_items=None):
        self.name, self.n_items = name, n_items
        self.note, self.seconds = "", None

@contextmanager
def timed(step, n_items=None, quiet=False):
    s  = _Step(step, n_items)
    t0 = time.perf_counter()
    try:
        yield s
    finally:
        s.seconds = time.perf_counter() - t0
        rec = {"step": s.name,
               "seconds": round(s.seconds, 2),
               "hms": _hms(s.seconds),
               "n_items": s.n_items,
               "ms_per_item": (round(s.seconds / s.n_items * 1000, 2) if s.n_items else None),
               "hardware": HARDWARE,
               "note": s.note}
        _prev = next((r for r in RUNTIME_LOG if r["step"] == s.name), None)
        if _prev is not None and float(_prev.get("seconds") or 0) > 60 and s.seconds < 5:
            # the step resumed from a complete checkpoint: keep the measured time and add a note
            _old_note = "" if pd.isna(_prev.get("note")) else str(_prev.get("note"))
            _prev["note"] = (_old_note + " | later run resumed from checkpoint").strip(" |")
            rec = _prev
        RUNTIME_LOG[:] = [r for r in RUNTIME_LOG if r["step"] != s.name] + [rec]
        pd.DataFrame(RUNTIME_LOG).to_csv(RUNTIME_PATH, index=False)
        if not quiet:
            extra = (f" | n={s.n_items:,} | {rec['ms_per_item']:,.1f} ms/item" if s.n_items else "")
            print(f"[time] {s.name}: {rec['hms']} ({s.seconds:,.1f} s){extra}")

# ── Wall time of every cell (for the run report of section 10) ───────────────
# The first non-empty line of each code cell is its title; the time of each executed cell is appended to cell_times.csv.
CELL_LOG_PATH = os.path.join(SAVE_DIR, "cell_times.csv")
_cell_state = {}

def _cell_pre(info=None):
    _cell_state["t0"] = time.perf_counter()
    _raw = getattr(info, "raw_cell", "") or ""
    _first = next((l for l in _raw.splitlines() if l.strip()), "")
    _cell_state["title"] = _first.strip().strip("#").strip().strip("─").strip()[:100]

def _cell_post(result=None):
    if "t0" not in _cell_state:
        return
    _sec = time.perf_counter() - _cell_state.pop("t0")
    _ok  = bool(getattr(result, "success", True))
    pd.DataFrame([{"cell": _cell_state.get("title", ""), "seconds": round(_sec, 2), "ok": _ok,
                   "finished": time.strftime("%Y-%m-%d %H:%M:%S"), "hardware": HARDWARE}]
                 ).to_csv(CELL_LOG_PATH, mode="a", header=not os.path.exists(CELL_LOG_PATH), index=False)

_ip = get_ipython()
if _ip is not None:
    for _ev, _fn in (("pre_run_cell", _cell_pre), ("post_run_cell", _cell_post)):
        for _old in [f for f in list(_ip.events.callbacks[_ev]) if getattr(f, "__name__", "") == _fn.__name__]:
            _ip.events.unregister(_ev, _old)
        _ip.events.register(_ev, _fn)

# ── Family-grouped split (prevents train/test leakage) ───────────────────────
# Continuations in one patent family have almost the same specification (for example, US10592275B2 and
# US12159161B2 belong to family US13/750696 and have a word-for-word identical background). A random split
# puts members of one family in both training and test sets and raises the scores. All supervised examples
# (7.11, 8.9, 8.10) therefore split by family. The family key is the root application number in the
# `family` column ("US13/750696 (n=2)" gives "US13/750696"); a record without a family is its own group.
import re as _re_fam

def build_family_groups(patent_ids, data_path=None):
    """Return the family keys aligned with patent_ids (reads the family column of CLEAN_PATH)."""
    fam_df = pd.read_csv(data_path or CLEAN_PATH,
                         usecols=["patent_id", "family"], dtype=str).fillna("")
    root = {}
    for pid, fam in zip(fam_df["patent_id"], fam_df["family"]):
        m = _re_fam.match(r"\s*(\S+)", fam)
        root[pid] = m.group(1) if m else ""
    return np.array([root.get(p) or f"self:{p}" for p in patent_ids])

def family_train_test_split(idx, y, groups, test_size=0.2):
    """Family-grouped train/test split (one GroupShuffleSplit split).
    A grouped split cannot also stratify; macro-F1 with zero_division=0 handles classes that are missing in one part."""
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)
    tr, te = next(gss.split(idx, y, groups=groups))
    return idx[tr], idx[te], y[tr], y[te]

def add_ipc3(frame, src_col="main_ipc_4", col="main_ipc_3"):
    """Add the three-character class (G06F -> G06) as a new column and return the same frame.
    The classification hierarchy is section G / class G06 / subclass G06F. The merged topics meet about
    40 three-character classes, a granularity closer to theirs than the hundreds of subclasses, so the
    three-character class is the scale of the external check; the subclass is kept for the finer examples."""
    if src_col in frame.columns:
        frame[col] = frame[src_col].astype(str).str.strip().str.upper().str[:3]
    return frame

# ── Figure style ─────────────────────────────────────────────────────────────
# One categorical order (checked for colour-vision deficiency), one blue ramp for magnitudes,
# grey for "other". Every figure is also saved to FIG_DIR as a 300 dpi PNG.
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap
PALETTE   = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, INK_2, MUTED, GRID, BASELINE, OTHER_GREY = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#d3d2cc"
SEQ_CMAP  = LinearSegmentedColormap.from_list(
    "seq_blue", ["#f4f8fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8,
    "axes.titlelocation": "left", "axes.titlecolor": INK, "axes.labelcolor": INK_2,
    "axes.edgecolor": BASELINE, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "grid.color": GRID, "grid.linewidth": 0.6, "legend.frameon": False,
    "axes.prop_cycle": cycler(color=PALETTE),
})

def save_fig(fig, name):
    """Save a figure as FIG_DIR/<name>.png (300 dpi) and return the path."""
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path)
    print(f"saved -> figures/{name}.png")
    return path

print(f"Helpers ready | hardware: {HARDWARE} | run-time log: {os.path.basename(RUNTIME_PATH)}")

### 1.7  沿用已生成的資料層

生成文字是整個流程最昂貴的部分，而且即使用貪婪解碼，換一張 GPU 或換套件版本也不一定得到一模一樣的文字。釋出資料集（包括人工稽核）對應的是某一次執行的文字。設定 `REUSE_DIR` 時，這一格會把該次執行的 checkpoint，以及結果受硬體影響的步驟輸出（向量、UMAP 座標、分群與主題標籤、稽核樣本）複製到 `SAVE_DIR`。下表列出每個檔與 SHA-256 前 12 碼。後面的格讀到這些檔時，都會和本次的語料逐筆比對，紀錄不同就停下。

In [ ]:
# ── 1.7  Reuse the generated layers of an earlier run (only when REUSE_DIR is set) ──
# The language-model outputs are the expensive part of this notebook (about 20 h on one A100 80 GB).
# When REUSE_DIR is the output folder of an earlier run, this cell copies into SAVE_DIR the checkpoints of
# the three generated layers and the outputs of steps whose result depends on the hardware or on a random
# initialisation: the SPECTER2 embeddings, the UMAP coordinates, the cluster assignment with its topic labels
# and the human-audit sample. The generation cells then find complete checkpoints and do not load a model;
# every other step is computed again. Each reused file is checked against the corpus of this run where it is read.
import shutil, hashlib, glob

REUSE_FILES = [   # (file in REUSE_DIR, file in SAVE_DIR, used in)
    ("stage1_checkpoint.csv",        "stage1_checkpoint.csv",          "4.3 facets"),
    ("stage2_checkpoint.csv",        "stage2_checkpoint.csv",          "5.6 synthesized summaries"),
    ("baseline_direct.csv",          "baseline_direct.csv",            "5.10 direct-brief summaries"),
    ("baseline_direct_matched.csv",  "baseline_direct_matched.csv",    "5.11 direct-full summaries"),
    ("specter2_embeddings.csv",      "specter2_embeddings.csv",        "6.2 synthesized embeddings"),
    ("cluster_results.csv",          "reused_cluster_results.csv",     "6.4 UMAP coordinates and clusters, 6.8 topic labels"),
    ("hdbscan_param_sweep.csv",      "reused_hdbscan_param_sweep.csv", "6.5 sweep that goes with the reused clusters"),
    ("umap2d_coords.csv",            "umap2d_coords.csv",              "7.7 two-dimensional map"),
    ("audit_primary_50.csv",         "audit_primary_50.csv",           "8.8 human-audit sample"),
    ("audit_edge_12.csv",            "audit_edge_12.csv",              "8.8 human-audit sample"),
    ("audit_sentences.csv",          "audit_sentences.csv",            "8.8 human-audit sample"),
    ("released_corpus_ids.csv",      "reused_released_corpus_ids.csv", "3.5 check of the corpus"),
    ("runtime_log.csv",              "reused_runtime_log.csv",         "9.2 cost of the reused steps"),
]
if REUSE_DIR:
    REUSE_FILES += [(os.path.join("embeddings", os.path.basename(p)),
                     os.path.join("reused_embeddings", os.path.basename(p)), "6.12 representations")
                    for p in sorted(glob.glob(os.path.join(REUSE_DIR, "embeddings", "*.parquet")))]

def _sha12(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()[:12]

reuse_report = pd.DataFrame(columns=["file", "status", "MB", "sha256_12", "used_in"])
if REUSE_DIR is None:
    print("REUSE_DIR is None: every layer is generated in this run.")
elif not os.path.isdir(REUSE_DIR):
    raise FileNotFoundError(f"REUSE_DIR does not exist: {REUSE_DIR} (set it to None to generate everything)")
else:
    os.makedirs(os.path.join(SAVE_DIR, "reused_embeddings"), exist_ok=True)
    _rows = []
    for _src_rel, _dst_rel, _use in REUSE_FILES:
        _src, _dst = os.path.join(REUSE_DIR, _src_rel), os.path.join(SAVE_DIR, _dst_rel)
        if not os.path.exists(_src):
            _rows.append({"file": _src_rel, "status": "not in REUSE_DIR (computed in this run)",
                          "MB": None, "sha256_12": "", "used_in": _use})
            continue
        _status = "already in SAVE_DIR" if os.path.exists(_dst) else "copied"
        if _status == "copied":
            _ = shutil.copy2(_src, _dst)
        _rows.append({"file": _src_rel, "status": _status, "MB": round(os.path.getsize(_dst) / 1e6, 2),
                      "sha256_12": _sha12(_dst), "used_in": _use})
    reuse_report = pd.DataFrame(_rows)
    reuse_report.to_csv(os.path.join(SAVE_DIR, "reuse_manifest.csv"), index=False)
    print(f"Reused from {REUSE_DIR}:")
reuse_report

## 2  撞字篩選：21,947 → 16,434 筆

縮寫 HPC 在運算類專利是高效能運算（high-performance computing），在藥學專利是羥丙基纖維素（hydroxypropyl cellulose），在渦輪專利是高壓壓縮機（high-pressure compressor），在生醫專利是造血前驅細胞（hematopoietic progenitor cell）。所以用 `HPC` 檢索會撈到許多和運算無關的紀錄。本節只在紀錄本身的文字顯示另一個意思時才移除：作者在括號定義裡寫了別的全稱，或全文出現別的意思的完整片語。沒有這類證據的紀錄一律保留。分類號只用來描述分布，不參與判定。

2.1 讀入爬蟲輸出，把五個段落欄位從網頁標題（例如 `DETAILED DESCRIPTION`）改成小寫欄名（`detailed_description`）。抽樣版也在這一格抽出子集。

In [ ]:
# ── 2.1  Load the crawler output and rename the section columns ──────────────
# dtype=str reads every column as text. Columns such as claim_count have missing values, and an inferred
# numeric type would turn 17 into 17.0; the screening does no arithmetic, so the output stays identical to
# the input file character by character.
from collections import Counter

# The section columns of the crawler output carry the heading on the Google Patents page (upper case with
# spaces). They are renamed here; every later cell and output file uses the new names.
RENAME_MAP = {
    "BACKGROUND": "background",
    "SUMMARY": "summary",
    "DETAILED DESCRIPTION": "detailed_description",
    "CROSS-REFERENCE TO RELATED APPLICATION": "cross-reference_to_related_application",
    "BRIEF DESCRIPTION OF THE DRAWINGS": "brief_description_of_drawings",
}
# The six text fields that are joined for the sense evidence: a "(HPC)" definition can be in any of them.
HOMONYM_TEXT_COLS = ["title", "abstract", "background", "summary", "detailed_description", "claims_text"]

print(f"input: {DATA_PATH} ({os.path.getsize(DATA_PATH)/1e9:.2f} GB)")
with timed("2.1  Load the crawler output"):
    df = pd.read_csv(DATA_PATH, dtype=str)
df = df.rename(columns=RENAME_MAP)
N_FILE = len(df)
if SAMPLE_N:
    # sample profile: a seeded random subset; the original row order is kept
    df = df.sample(min(SAMPLE_N, len(df)), random_state=RANDOM_STATE).sort_index().reset_index(drop=True)

# The decision steps add temporary columns to df; the output files keep only these original columns.
ORIG_COLS = df.columns.tolist()
N_RAW = len(df)
print(f"Records in the input file: {N_FILE:,}" + (f" | sample used in this run: {N_RAW:,}" if SAMPLE_N else ""))
print(f"Columns ({len(ORIG_COLS)}): {ORIG_COLS}")

# Records by application year
yr_df = (df["application_year"].value_counts().sort_index()
         .rename_axis("application_year").reset_index(name="records"))
yr_df

### 2.2  文字欄位的覆蓋率

第 4 步的解析器在網頁段落標題和標準標題對不上時會寫 `N/A`，所以下表的覆蓋率反映的是段落擷取的成功率，不是專利本身的性質。這會影響撞字判定：解析出來的文字若缺了出現「HPC」的段落，這筆紀錄就拿不出任何意思的證據，因此依規則保留。

In [ ]:
# ── 2.2  Coverage of the six text fields ────────────────────────────────────
# A field "has content" when it is not missing, not empty after stripping spaces and not the literal
# string "nan". pandas already reads the crawler placeholders "NA" and "N/A" as missing, so a section whose
# heading the parser did not recognise counts as missing here.
# Four tables: coverage per field, missing-content patterns, best available text, shortest abstracts.
present = {}                        # field -> bool Series (True = has content)
length  = {}                        # field -> characters (0 when missing)
for c in HOMONYM_TEXT_COLS:
    s = df[c].fillna("")
    ok = (s.str.strip() != "") & (s.str.lower() != "nan")
    present[c] = ok
    length[c]  = s.str.len().where(ok, 0)

# Table 1: coverage per field
rows = []
for c in HOMONYM_TEXT_COLS:
    ok, L = present[c], length[c]
    rows.append({
        "field": c,
        "has_content": int(ok.sum()),
        "coverage_percent": round(ok.mean() * 100, 1),
        "missing_content": int((~ok).sum()),
        # a title is short by nature, so the check for very short text does not apply to it
        "very_short_under_200_chars": "-" if c == "title" else int((ok & (L < 200)).sum()),
        "median_length": int(L[ok].median()) if ok.any() else 0,
    })
field_coverage = pd.DataFrame(rows)
field_coverage

# Figure 2.1: coverage of the six text fields
_qs = field_coverage.set_index("field")["coverage_percent"].sort_values()
fig, ax = plt.subplots(figsize=(6.4, 2.6))
_ = ax.barh(_qs.index, _qs.values, color=PALETTE[0], height=0.6)
for i, v in enumerate(_qs.values):
    _ = ax.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=8, color=INK_2)
_ = ax.set_xlim(0, 110)
_ = ax.set_xlabel("records with content (percent)")
_ = ax.set_title(f"Figure 2.1  Coverage of the six text fields ({N_RAW:,} records)")
fig.tight_layout()
_ = save_fig(fig, "fig_2_1_field_coverage")
plt.show()

# Table 2: missing-content patterns (one row per combination of present and missing fields)
pat = pd.DataFrame({c: present[c].map({True: "yes", False: "-"}) for c in HOMONYM_TEXT_COLS})
pat.columns = ["title", "abstract", "background", "summary", "detailed", "claims"]
combo = pat.value_counts().rename("records").reset_index()
combo["share_percent"] = (combo["records"] / len(df) * 100).round(1)
combo.head(12)

# Table 3: the most complete text that each record offers, in the order
# detailed_description > summary or background > claims > title and abstract only
best = pd.Series(np.select(
    [present["detailed_description"],
     present["summary"] | present["background"],
     present["claims_text"]],
    ["has detailed_description", "no detailed_description, has summary or background", "claims and title/abstract only"],
    default="title and abstract only"), index=df.index, name="best_available_text")
best_df = best.value_counts().rename_axis("best_available_text").reset_index(name="records")
best_df["share_percent"] = (best_df["records"] / len(df) * 100).round(1)
best_df

# Table 4: the five shortest non-empty abstracts (a visual check for junk values)
shortest = (df.loc[present["abstract"], ["patent_id", "abstract"]]
              .assign(n_chars=length["abstract"][present["abstract"]])
              .nsmallest(5, "n_chars"))
shortest["abstract"] = shortest["abstract"].str.slice(0, 100)
shortest.reset_index(drop=True)

deep4 = present["background"] | present["summary"] | present["detailed_description"] | present["claims_text"]
print(f"Records with none of background, summary, detailed_description and claims (title and abstract only): {int((~deep4).sum()):,}")
print(f"Records without title or abstract: {int((~(present['title'] & present['abstract'])).sum()):,}")

### 2.3–2.6  證據與判定

三層證據依序判斷：

1. **定義。** 以 Schwartz-Hearst 演算法取出作者寫在「(HPC)」前面的全稱。全稱屬於某個語意組，就由該組決定；同一筆紀錄的定義指向多個組時，取全文片語出現最多的組。
2. **片語。** 沒有定義時，取全文中完整片語出現最多的語意組。
3. **沒有證據。** 保留。

2.3 定義八個語意組。其中四組是運算以外的意思（纖維素、壓縮機、前驅細胞、混凝土）；三組是運算領域的術語，但其中的 HPC 不是高效能運算（硬體效能計數器、最高一致性點、高腳數封裝）。這七組都會移除。2.5 列出資料裡找到的每一個全稱，若規則漏了某個意思，從這張表就看得出來。

In [ ]:
# ── 2.3  Sense rules: what HPC means in other domains ───────────────────────
# In different domains, HPC means at least:
#   high-performance computing (the meaning that we want)
#   hydroxypropyl cellulose   (excipient in drugs/cosmetics)
#   high pressure compressor  (jet engines/gas turbines)
#   hematopoietic progenitor cell (blood-forming progenitor cell) … and others
# These patents have no relation to high-performance computing; they are noise from the literal match of the query term.

# ---- (1) Detection of parenthetical definitions ----
# find "(hpc)" or "(hpcs)" in the full text; the full text is already in lower case.
# \s* allows spaces inside the parentheses, for example "( HPC )".
DEF_PAT = re.compile(r"\(\s*(hpcs?)\s*\)")

def sh_longform(text_before, sf="hpc", max_words=8):
    """Schwartz–Hearst abbreviation algorithm: find the long form of the abbreviation in the text before the parenthesis.

    Principle: each letter of the abbreviation occurs "in order" in the long form (not necessarily as a first letter).
    Method: take at most max_words words before the parenthesis and match the abbreviation characters one by one, from last to first:

        Example: '... comprising hydroxypropyl cellulose (hpc) ...'
            search 'c' from right to left → matches the c in cellulose
            search 'p' further left     → matches the p of propyl
            search 'h' further left     → matches the h of hydroxypropyl
            all characters match → go back from the last match to the start of that word,
            result: long form = 'hydroxypropyl cellulose'

    Parameters
        text_before : a short text before the parenthesis (the caller passes only the 120 characters before it; the definition is always next to the parenthesis)
        sf          : the abbreviation (lower case)
        max_words   : look back only 8 words; a longer look-back takes ordinary sentences as definitions

    Returns the long form string; returns None if the match fails or the length is not plausible.
    """
    cand = " ".join(text_before.split()[-max_words:])
    i = len(cand) - 1
    for ch in reversed(sf):                 # match the abbreviation characters from last to first
        while i >= 0 and cand[i] != ch:
            i -= 1
        if i < 0:                           # a character is not found → not a definition
            return None
        i -= 1
    start = cand.rfind(" ", 0, i + 1) + 1 if i > 0 else 0   # go back to the start of the word
    lf = cand[start:].strip(" ,;:.")
    # Schwartz–Hearst plausibility check: the long form must not have more words than
    # min(|abbreviation|+5, |abbreviation|×2); a longer one is usually a false match, so drop it
    if not lf or len(lf.split()) > min(len(sf) + 5, len(sf) * 2):
        return None
    return lf

# ---- (2)(3) Sense groups ----
# key = internal code; value = (display name in reports, computing sense or not, regex for the full-text phrase)
#
# Notes on the regex:
#   [\- ]      hyphen or space → matches both high-performance and high performance
#   comput\w*  suffixes such as computing / computer / computation
#   hydroxyl?  both spellings hydroxy and hydroxyl (? = the previous character is optional)
#   [ \-]?     optional space or hyphen → also matches the joined form hydroxypropylcellulose
SENSE_GROUPS = {
    "computing":  ("Computing: high-performance computing",                         True,
                   r"high[\- ]performance comput\w*|supercomput\w*|high[\- ]performance cluster"),
    "cellulose":  ("Chemistry/pharma: hydroxypropyl cellulose",                     False,
                   r"hydroxyl?[ \-]?propyl[ \-]?cellulose"),
    "compressor": ("Aerospace/mechanical: high pressure compressor",                False,
                   r"high[\- ]pressure compressor"),
    "progenitor": ("Biomedical: hematopoietic progenitor cell",                     False,
                   r"hematopoietic progenitor"),
    "counter":    ("Computing peripheral (not HPC): hardware performance counter",  False,
                   r"hardware performance counter"),
    "coherency":  ("Computing term (not HPC): highest point of coherency (cache)",  False,
                   r"highest point of coherency"),
    "concrete":   ("Construction: high performance concrete",                       False,
                   r"high[\- ]performance concrete"),
    "pincount":   ("Electronics (not HPC): high pin count",                         False,
                   r"high[\- ]pin count"),
}
# Note: the counter and coherency groups belong to the computing "domain"
# (hardware performance counter = CPU performance counter;
#  highest point of coherency = an IBM term for cache coherency in multiprocessors),
# but "HPC" in these texts does not mean high-performance computing → it is still another meaning and is removed.

def defgroup(lf):
    """Map a long form found by SH to a sense group.

    Pass 1: exact match with the phrase regex of each group.
    Pass 2: loose match with one keyword, for definitions with suffixes or other forms
            (example: 'high performance computer system' contains 'comput' → computing).
    No match in both passes → 'other' (rare or one-time definitions; a manual look at the list in section 3 is enough).
    """
    for g, (_, _, rx) in SENSE_GROUPS.items():
        if re.search(rx, lf):
            return g
    for g, kw in [("computing", "comput"), ("cellulose", "cellulose"),
                  ("compressor", "compressor"), ("progenitor", "progenitor"),
                  ("counter", "counter"), ("coherency", "coheren"),
                  ("concrete", "concrete")]:
        if kw in lf:
            return g
    return "other"

In [ ]:
# ── 2.4  Evidence extraction (parenthetical definitions and domain phrases) ──
# 1. Join the six text fields of each record into one lower-case text (missing values count as empty).
# 2. Count the word "hpc" and the full phrase of each sense group with vectorised regular expressions.
# 3. Run the Schwartz-Hearst algorithm on every "(HPC)" to recover the long form that the authors wrote.
# The joined text is deleted afterwards to free memory.
with timed("2.4  Sense evidence extraction", n_items=len(df)):
    txt_parts = []
    for c in HOMONYM_TEXT_COLS:
        s = df[c].fillna("")
        s = s.mask((s.str.strip() == "") | (s.str.lower() == "nan"), "")
        txt_parts.append(s)
    txt = txt_parts[0].str.cat(txt_parts[1:], sep=" ").str.lower()
    del txt_parts

    # \b is a word boundary: only the separate words hpc and hpcs count, not strings such as "hpcl"
    df["hpc_mentions"] = txt.str.count(r"\bhpcs?\b")
    for g, (_, _, rx) in SENSE_GROUPS.items():
        df[f"n_{g}"] = txt.str.count(rx)

    # Only the 120 characters before the parenthesis go to sh_longform: a definition is always next to the
    # parenthesis, and 120 characters hold the max_words = 8 words that the algorithm looks back.
    defs = []
    for k, t in enumerate(txt):
        found = set()
        for m in DEF_PAT.finditer(t):
            lf = sh_longform(t[max(0, m.start() - 120):m.start()])
            if lf:
                found.add(lf)
        defs.append(";".join(sorted(found)))   # a record can define the abbreviation more than once; keep all
    df["sh_defs"] = defs

    # Section of the class code (first letter): G physics and computing, H electricity, A human necessities,
    # C chemistry, F mechanical engineering. Used only in the reports, never in the decision.
    df["ipc_section"] = df["main_ipc_4"].fillna("?").str[0]
    del txt
    _ = gc.collect()

# How many records each kind of evidence covers
evidence_summary = pd.DataFrame({
    "evidence": ['(1) "(HPC)" parenthetical definition', "(2) domain phrase (any group)", "word hpc occurs"],
    "records": [int((df["sh_defs"] != "").sum()),
                int((df[[f"n_{g}" for g in SENSE_GROUPS]].sum(axis=1) > 0).sum()),
                int((df["hpc_mentions"] > 0).sum())],
})
evidence_summary["share_percent"] = (evidence_summary["records"] / len(df) * 100).round(1)
evidence_summary

In [ ]:
# ── 2.5  The "(HPC)" definitions that occur in the data ────────────────────
# Collect the SH long forms of all patents and count them. This table has two uses:
#   1. Direct evidence: which other meanings HPC has in this corpus, and how many patents have each
#   2. Check whether SENSE_GROUPS misses a sense (rows with group = other need a manual look)
inv = Counter()
for ds in df["sh_defs"]:
    if ds:
        inv.update(ds.split(";"))              # several definitions in one patent count once each
inv_df = (pd.DataFrame(inv.most_common(), columns=["long form (HPC)", "patents"])
          .assign(group=lambda d: d["long form (HPC)"].map(defgroup)))

n_withdef = int((df["sh_defs"] != "").sum())
print(f'Patents with an explicit "…(HPC)" parenthetical definition: {n_withdef:,} / {N_RAW:,} ({n_withdef/N_RAW*100:.1f}%)')
print(f"Distinct long forms: {len(inv_df)}; top 30 below")
inv_df

In [ ]:
# ── 2.6  Sense of each record from three levels of evidence ─────────────────
# The three levels of evidence apply in order of priority:
#   (1) SH parenthetical definition present → the definition decides (the authors state what HPC means; most reliable)
#        - definitions in one patent map to several sense groups (very rare) → take the group with the most full-text phrases
#   (2) no definition, but sense phrases occur in the full text → take the group with the most occurrences
#   (3) no text evidence at all → always keep:
#        a removal needs "explicit text that shows the other meaning" (the parenthetical definition of (1) or the full phrase of (2));
#        without evidence, the IPC class is not used to infer it.
#        (missing evidence is mostly caused by incomplete full text in the CSV: the query term is in a section that was not parsed,
#          not because the patent does not mention HPC; there is no valid reason to remove such records.)
#
# Output columns:
#   sense        sense group code (unknown = no text evidence)
#   basis        decision basis ((1)/(2)/(3); for later spot checks or for the second-level validation)
#   sense_cat    display name in reports
#   is_hpc_sense True = keep (computing sense, or no evidence of another meaning) / False = other meaning (remove)
GLIST = list(SENSE_GROUPS)

def decide(row):
    # ---- (1) SH parenthetical definition ----
    dgs = {defgroup(x) for x in row["sh_defs"].split(";") if x} - {"other"}
    if len(dgs) == 1:
        return dgs.pop(), "(1) SH definition"
    if len(dgs) > 1:                       # one patent defines several senses (rare)
        return max(dgs, key=lambda g: row[f"n_{g}"]), "(1) SH definition (several senses: most phrases)"
    # ---- (2) Full-text phrase count ----
    counts = {g: row[f"n_{g}"] for g in GLIST}
    if max(counts.values()) > 0:
        return max(counts, key=counts.get), "(2) full-text phrase"
    # ---- (3) No text evidence → always keep (no inference from IPC) ----
    return "unknown", "(3) no text evidence (kept)"

with timed("2.6  Sense decision", n_items=len(df)):
    df[["sense", "basis"]] = df.apply(decide, axis=1, result_type="expand")

# sense code → (display name, keep or not); add the class "no evidence → keep" of (3)
CAT = {g: (c, h) for g, (c, h, _) in SENSE_GROUPS.items()}
CAT["unknown"] = ("No text evidence (cannot decide → keep)", True)
df["sense_cat"]    = df["sense"].map(lambda g: CAT[g][0])
df["is_hpc_sense"] = df["sense"].map(lambda g: CAT[g][1])

# ---- Report 1: counts per other meaning ----
print("=== Counts per other meaning ===")
homonym_tab = (df[~df.is_hpc_sense].groupby("sense_cat").size()
               .rename("patents").sort_values(ascending=False).to_frame())
homonym_tab["percent_of_all"] = (homonym_tab.patents / N_RAW * 100).round(1)
homonym_tab
n_unknown = int((df["sense"] == "unknown").sum())
print(f"Other meaning, total (explicit text evidence): {int((~df.is_hpc_sense).sum()):,} records"
      f" ({(~df.is_hpc_sense).mean()*100:.1f}%); "
      f"kept {int(df.is_hpc_sense.sum()):,} records"
      f" = computing sense {int(df.is_hpc_sense.sum()) - n_unknown:,} + no text evidence {n_unknown:,}")

# ---- Breakdown of the no-text-evidence group ((3))  ----
# "no evidence" is almost always caused by incomplete full text in the CSV: the word hpc is not in the available text at all
noev = df[df["sense"] == "unknown"]
noev_break = pd.DataFrame({
    "condition": ["the word hpc is not in the six text fields (query term in a section that was not parsed)",
             "hpc occurs, but no parenthetical definition and no known phrase"],
    "records": [int((noev["hpc_mentions"] == 0).sum()),
             int((noev["hpc_mentions"] > 0).sum())],
})
noev_break["percent_of_group3"] = (noev_break["records"] / max(len(noev), 1) * 100).round(1)
noev_break

# Class section of the kept records without evidence
noev_sec = (noev["ipc_section"].value_counts()
            .rename_axis("cpc_section").reset_index(name="records"))
noev_sec

# ---- Report 2: decision basis × computing or not ----
# shows how many decisions each level of evidence makes; the "(3) no text evidence (kept)" row has no clear basis for removal, so it is kept
print("\nDecision basis × computing sense or not:")
pd.crosstab(df["basis"], df["is_hpc_sense"], margins=True)

# Figure 2.2: records per sense class (blue = kept, orange = other sense, removed)
from matplotlib.patches import Patch
sc = df["sense_cat"].value_counts().sort_values()
ishpc_by_cat = {c: h for _, (c, h) in CAT.items()}
fig, ax = plt.subplots(figsize=(7.6, 3.6))
_ = ax.barh(sc.index, sc.values, height=0.6,
            color=[PALETTE[0] if ishpc_by_cat.get(c, False) else PALETTE[1] for c in sc.index])
for i, v in enumerate(sc.values):
    _ = ax.text(v + sc.max() * 0.01, i, f"{v:,}", va="center", fontsize=8, color=INK_2)
_ = ax.set_xlim(0, sc.max() * 1.15)
_ = ax.set_xlabel("records")
_ = ax.legend(handles=[Patch(color=PALETTE[0], label="kept (computing sense or no evidence)"),
                       Patch(color=PALETTE[1], label="removed (other sense)")], loc="lower right")
_ = ax.set_title(f"Figure 2.2  Sense of the abbreviation HPC in {N_RAW:,} records")
fig.tight_layout()
_ = save_fig(fig, "fig_2_2_sense_classes")
plt.show()

# Three titles of each removed class, for a visual check of the decision
samples = (df.loc[~df.is_hpc_sense, ["sense_cat", "patent_id", "title", "basis"]]
             .groupby("sense_cat", group_keys=False).head(3)
             .sort_values("sense_cat").reset_index(drop=True))
samples

### 2.7  移除

被移除的紀錄集中在其他意思所屬的部：A（生活必需品，含醫藥製劑）、C（化學）與 F（機械工程）。候選檔 `HPC_15to25_clean.csv` 保留原本的 33 欄。

In [ ]:
# ── 2.7  Remove the other senses and write the candidate file ───────────────
# The only removal condition is is_hpc_sense == False. If the removal is clean, the sections A (human
# necessities, including pharmaceuticals), C (chemistry) and F (mechanical engineering) shrink sharply.
keep_mask = df["is_hpc_sense"]
N_CLEAN = int(keep_mask.sum())
kept = df[keep_mask]

funnel = pd.DataFrame([
    {"stage": "records in the input",          "records": N_RAW,   "change": ""},
    {"stage": "removed: HPC has another sense", "records": N_RAW - N_CLEAN, "change": f"-{(N_RAW - N_CLEAN)/N_RAW*100:.1f}%"},
    {"stage": "candidates",                     "records": N_CLEAN, "change": f"kept {N_CLEAN/N_RAW*100:.1f}%"},
])
funnel

# Class section before and after the removal
sec_df = (pd.concat([df["ipc_section"].value_counts().rename("before_removal"),
                     kept["ipc_section"].value_counts().rename("after_removal")], axis=1)
          .fillna(0).astype(int).rename_axis("cpc_section").reset_index())
sec_df["removal_rate_percent"] = ((1 - sec_df["after_removal"] / sec_df["before_removal"]) * 100).round(1)
sec_df

# Figure 2.3: records per class section before and after the removal
_x = np.arange(len(sec_df))
fig, ax = plt.subplots(figsize=(7.2, 3.0))
_ = ax.bar(_x - 0.2, sec_df["before_removal"], width=0.38, color=OTHER_GREY, label=f"input ({N_RAW:,})")
_ = ax.bar(_x + 0.2, sec_df["after_removal"],  width=0.38, color=PALETTE[0], label=f"candidates ({N_CLEAN:,})")
_ = ax.set_xticks(_x, sec_df["cpc_section"])
_ = ax.set_xlabel("class section (first letter of the class code)")
_ = ax.set_ylabel("records")
_ = ax.legend(loc="upper left")
_ = ax.set_title("Figure 2.3  Class sections before and after the removal of the other senses")
fig.tight_layout()
_ = save_fig(fig, "fig_2_3_sections_before_after")
plt.show()

# Kept records outside sections G and H: computing evidence in another field (for example reservoir
# simulation or medical image computing), or no text evidence at all. This notebook does not filter by class.
nongh = (kept.loc[~kept["ipc_section"].isin(["G", "H"]), "ipc_section"]
         .value_counts().rename_axis("cpc_section").reset_index(name="records"))
print("Kept records outside sections G and H:")
nongh

# Only the original columns are written; the temporary decision columns stay in memory.
with timed("2.7  Write the candidate file", n_items=N_CLEAN):
    kept[ORIG_COLS].to_csv(CLEAN_PATH, index=False)
print(f"\nCandidates -> {CLEAN_PATH} ({N_CLEAN:,} records x {len(ORIG_COLS)} columns)")

### 2.8–2.11  家族與訴訟標註

新增兩欄註記，不移除任何紀錄。

- `family`：最早優先權申請案相同的紀錄視為同一家族（接近 simple family）。值是家族根申請號加上本語料中的家族成員數，例如 `US13/750696 (n=2)`；沒有其他家族成員的紀錄留空。後面的監督式示範以這個鍵切分，避免文字幾乎相同的延續案同時落在訓練集與測試集。
- `litigation`：USPTO docket 的地方法院案件（以專利號比對），以及 `legal_events` 裡的多方複審（IPR）、核准後複審（PGR）與再審查事件。docket 只收到 2020 年提起的案件，所以欄位空白不代表這件專利從未涉訟。

2.10 寫出兩個後面各節不會用到的衍生檔：每個家族只留一筆的版本，以及所有含涉訟成員的家族（附案件群組，把同一案件裡一起主張的專利連在一起）。2.11 各舉一個案件群組與一個家族為例，列出判定依據。

In [ ]:
# ── 2.8  Rules for the family and litigation marks ──────────────────────────
import ast

# ---- Family-root parsing ----
# in applications_claiming_priority, application_number is "application number + publication number" joined together,
# example: 'US15/235,004US10193762B2(en)' → the application number is only 'US15/235,004'
# Two rules split it:
#   PCT_PAT: catch PCT numbers (such as PCT/US2015/012345) first:
#            the middle part contains 'US' + digits, which would mislead the general rule and cut it to 'PCT/'
#   GEN_PAT: a general application number = 2–3 upper-case letters + digits/slashes/commas/hyphens,
#            and it stops at the next "two upper-case letters + digit" (= start of the publication number) or at '('
# Finally, remove commas and spaces to normalize ('US15/235,004' → 'US15/235004'),
# so that the same application number in a different format is not taken as a different family.
PCT_PAT = re.compile(r"^(PCT/[A-Z]{2}\d{4}/\d+)")
GEN_PAT = re.compile(r"^([A-Z]{2,3}[\d/,\-\. ]+?[A-Z]?)(?=[A-Z]{2}\d|\(|$)")

def clean_app_no(no):
    """Get the clean application number from the joined string and normalize it"""
    no = str(no).strip()
    m = PCT_PAT.match(no)
    if m:
        return m.group(1)
    m = GEN_PAT.match(no)
    s = m.group(1).strip() if m else no
    return s.replace(",", "").replace(" ", "")

def family_root(raw, own_id):
    """Return the family-root application number of this patent.

    applications_claiming_priority is a list[dict];
    a continuation lists the full chain of ancestors (itself included); each entry has application_number / filing_date.
    Rule: the application number of the entry with the earliest filing_date is the "family root";
          all members of a family give the same root, so grouping by the root gives the family.
    Parsing fails or the field is empty → the record's own patent_id is the root (singleton; not marked as a family).
    """
    try:
        apps = ast.literal_eval(raw) if isinstance(raw, str) and raw.strip() not in ("", "nan") else []
    except Exception:
        apps = []
    entries = []
    for a in apps:
        no = clean_app_no(a.get("application_number", ""))
        fd = str(a.get("filing_date", "") or "")
        if no:
            entries.append((fd, no))
    if not entries:
        return own_id
    dated = [e for e in entries if e[0]]              # entries with a filing_date first
    return min(dated)[1] if dated else entries[0][1]  # no dates at all → take the first entry (ancestors come first)

# ---- PTAB proceeding event extraction ----
# legal_events is a string of a list[dict]; scan each {...} event;
# an event whose title contains one of the keywords below is a proceeding event; extract its date for the note.
# The order is important: check 'inter partes review certificate' before 'inter partes review';
# otherwise a certificate event is marked as an ordinary IPR.
PTAB_KINDS = [
    ("inter partes review certificate", "IPR certificate"),   # certificate at the end of an IPR (the source text says whether claims were cancelled)
    ("inter partes review",             "IPR"),       # challenge to patent validity (often with an infringement suit)
    ("post-grant review",               "PGR"),       # validity challenge within 9 months after grant
    ("reexamination",                   "reexamination"),     # ex parte reexamination
]

def ptab_note(ev):
    """Get the PTAB proceeding note from the legal_events string, such as 'IPR 2021-12-21; IPR certificate 2025-11-04'"""
    if not isinstance(ev, str) or not ev:
        return ""
    out = []
    for m in re.finditer(r"\{[^{}]*\}", ev):          # take each event dict in turn (no eval, to avoid format traps)
        low = m.group(0).lower()
        for key, label in PTAB_KINDS:
            if key in low:
                dm = re.search(r"'date':\s*'(\d{4}-\d{2}-\d{2})'", m.group(0))
                out.append(f"{label} {dm.group(1)}" if dm else label)
                break                                  # one label per event
    return "; ".join(dict.fromkeys(out))               # remove duplicates, keep the order

# ---- District-court litigation match (built from the USPTO docket source file at run time; no fixed results) ----
# Source: patents.dta of the USPTO Patent Litigation Docket Reports:
#       patent litigation records of US district courts; each row = one case × one patent in suit;
#       three columns are used: case_number (case number), date_filed (filing date of the case), patent (patent number).
#       File: LITIG_DIR/patents.dta or patents.dta.zip (pandas reads the zip directly).
# Two cleaning steps; both are necessary:
#   1) Exclude values that contain "/": the patent column of the docket also has application "serial numbers" such as "11/082327";
#      without the non-digits this becomes 11082327, which collides with later patent numbers US11,0xx,xxx and gives false positives.
#   2) Make the join key "digits only, no leading zeros": the docket has values such as '5696571', 'RE38609',
#      and our patent_id is 'US10016403B2'; both sides must be reduced to the digit core to match.
# Note: the litigation status covers only the filing period of the data file (the load step prints the actual range); later cases are not found.
# The file name is chosen automatically: if the folder has the zip, read the zip (pandas supports it); otherwise read the original .dta file.
_cand = [f"{LITIG_DIR}/patents.dta.zip", f"{LITIG_DIR}/patents.dta"]
LITIG_DTA = next((p for p in _cand if os.path.exists(p)), None)
assert LITIG_DTA, f"Docket file not found: check that {LITIG_DIR} contains patents.dta or patents.dta.zip"
with timed("2.8  Read the litigation docket"):
    DOCKET = pd.read_stata(LITIG_DTA)
DOCKET["patent"] = DOCKET["patent"].astype(str)
DOCKET = DOCKET[~DOCKET["patent"].str.contains("/")]               # 1) exclude application serial numbers
DOCKET["_key"] = DOCKET["patent"].str.replace(r"\D", "", regex=True).str.lstrip("0")
DOCKET = (DOCKET[DOCKET["_key"] != ""]                             # 2) digit core as the join key
          .drop_duplicates(["_key", "case_number"])
          [["_key", "case_number", "date_filed"]])

def district_notes(pids):
    """For a Series of patent_id values, return the district-court litigation notes (no case → empty string); the index matches the input.

    Note format = "district court <n> cases (<filing years>): <first three case numbers>"; more than three case numbers end with "…".
    On the patent_id side, a regex takes the number after 'US'; do not keep all digits of the string:
    the 2 of the kind code (B2) would join the number, and then nothing matches.
    """
    keys = pids.str.extract(r"^US(\d+)", expand=False).str.lstrip("0")
    notes = {}
    for k, g in DOCKET[DOCKET["_key"].isin(keys)].groupby("_key"):
        cases = sorted(g["case_number"].unique())
        years = sorted(g["date_filed"].dt.year.unique())
        yr = str(years[0]) if years[0] == years[-1] else f"{years[0]}–{years[-1]}"
        notes[k] = (f"district court {len(cases)} cases ({yr}): "
                    + "; ".join(cases[:3]) + ("…" if len(cases) > 3 else ""))
    return keys.map(notes).fillna("")

print(f"Rules loaded: {os.path.basename(LITIG_DTA)} → docket {len(DOCKET):,} litigation rows, "
      f"{DOCKET['_key'].nunique():,} patents"
      f" (filed {DOCKET['date_filed'].min():%Y-%m} to {DOCKET['date_filed'].max():%Y-%m})")

In [ ]:
# ── 2.9  Apply the family and litigation marks ──────────────────────────────
# Two columns are added as notes; no record is removed. The candidate file is then written again with them.
# In a new session this cell reads the candidate file of 2.7 (sections 1, 2.8 and this cell are enough);
# a rerun computes both columns again and overwrites them.
if "kept" in globals() and isinstance(kept, pd.DataFrame):
    ann = kept.copy()
    base_cols = ORIG_COLS
else:
    ann = pd.read_csv(CLEAN_PATH, dtype=str)
    base_cols = [c for c in ann.columns if c not in ("family", "litigation")]
print(f"Records to mark: {len(ann):,}")

# ---- family column ----
# family root → number of records with the same root in "this corpus"; mark only if ≥2 (singletons stay empty).
# Note: n is the family size in the cleaned corpus, not the worldwide family size.
ann["_root"] = [family_root(r, p) for r, p in
                zip(ann["applications_claiming_priority"], ann["patent_id"])]
fam_size = ann.groupby("_root")["patent_id"].transform("size")
ann["family"] = np.where(fam_size >= 2,
                         ann["_root"] + " (n=" + fam_size.astype(str) + ")",
                         "")

n_fam_pat = int((ann["family"] != "").sum())
n_fam_grp = ann.loc[ann["family"] != "", "_root"].nunique()

# family mark statistics
fam_stats = pd.DataFrame([
    {"item": "families (≥2 records in this corpus)", "value": f"{n_fam_grp:,}"},
    {"item": "patents with a family mark",           "value": f"{n_fam_pat:,}"},
    {"item": "singletons (family empty)",            "value": f"{int((ann['family']=='').sum()):,}"},
])
fam_stats

# ---- litigation column ----
# merge the two sources: district-court docket match + PTAB events in legal_events, joined with " | "
dc   = district_notes(ann["patent_id"])
ptab = ann["legal_events"].map(ptab_note)
ann["litigation"] = [" | ".join(x for x in (a, b) if x) for a, b in zip(dc, ptab)]

n_dc, n_pt = int((dc != "").sum()), int((ptab != "").sum())
n_lit = int((ann["litigation"] != "").sum())

# litigation mark statistics
lit_stats = pd.DataFrame([
    {"source": "district court (docket lookup)",   "records": n_dc},
    {"source": "PTAB (IPR / PGR / reexamination)", "records": n_pt},
    {"source": "both",                             "records": n_dc + n_pt - n_lit},
    {"source": "litigation column marked (total)", "records": n_lit},
])
lit_stats

# ---- Sample view ----
print("\nSample of family marks (largest family):")
big = ann[ann["family"] != ""].copy()
big["_n"] = fam_size[big.index]
big.nlargest(5, "_n")[["patent_id", "title", "family"]]
print("All litigation marks:")
(ann.loc[ann["litigation"] != "", ["patent_id", "title", "litigation"]]
 .sort_values("patent_id").reset_index(drop=True))

# Write the candidate file again: original columns + family + litigation (helper columns such as _root are dropped)
with timed("2.9  Write the candidate file with the marks", n_items=len(ann)):
    ann[base_cols + ["family", "litigation"]].to_csv(CLEAN_PATH, index=False)
print(f"\nCandidates with marks -> {CLEAN_PATH} ({len(ann):,} records x {len(base_cols) + 2} columns)")

In [ ]:
# ── 2.10  Derived files: one record per family, and the litigated families ──
# Two derived files on the candidate level; neither is used by the later sections, and the candidate file is not changed.
#   HPC_15to25_clean_dedup.csv        one record per family (the earliest application), for analyses that count
#                                     each invention once
#   HPC_15to25_litigation_family.csv  every family with at least one litigated member, with case groups
if "ann" in globals():
    src_df   = ann
    out_cols = base_cols + ["family", "litigation"]
else:
    src_df = pd.read_csv(CLEAN_PATH, dtype=str)
    src_df[["family", "litigation"]] = src_df[["family", "litigation"]].fillna("")
    out_cols = [c for c in src_df.columns]
print(f"Source: {len(src_df):,} records × {len(out_cols)} columns")

# ============================================================
# Output A: family-deduplicated version
# ------------------------------------------------------------
# Group key: records with a family mark use family (= family root application number);
#         singletons (empty family) use their own patent_id: each singleton is its own group.
# Each group keeps the record with the "earliest application date" (on the same date, sort by patent_id for a stable result).
# Reason: continuations share the same description, so their embeddings are almost the same;
#       keeping the earliest record = keeping the first disclosure of the invention.
# ============================================================
grp_key = src_df["family"].where(src_df["family"] != "", src_df["patent_id"])
dedup = (src_df.assign(_grp=grp_key)
               .sort_values(["application_date", "patent_id"])   # earliest application first
               .drop_duplicates("_grp", keep="first"))

DEDUP_CSV = os.path.join(SAVE_DIR, "HPC_15to25_clean_dedup.csv")
dedup[out_cols].to_csv(DEDUP_CSV, index=False)

n_fam_grp2  = int((src_df["family"] != "").sum())                 # total number of family members
n_fam_root2 = src_df.loc[src_df["family"] != "", "family"].nunique()
dedup_stats = pd.DataFrame([
    {"item": "before deduplication",                            "n_rows": f"{len(src_df):,}"},
    {"item": "family members (reduced to 1 record per family)", "n_rows": f"{n_fam_grp2:,} → {n_fam_root2:,}"},
    {"item": "singletons (kept as is)",                         "n_rows": f"{int((src_df['family']=='').sum()):,}"},
    {"item": "after deduplication (output)",                    "n_rows": f"{len(dedup):,}"},
])
dedup_stats
print(f"Family-deduplicated version → {DEDUP_CSV}\n")

# ============================================================
# Output B: litigated-family subset
# ------------------------------------------------------------
# Selection rule: the record has a litigation mark, or any member of its family is litigated.
# Reason: litigation is about the "invention": continuations in a family share the same description and priority;
#       if any member is asserted or challenged, the whole family is relevant for litigation analysis.
# ============================================================
lit_mask = src_df["litigation"] != ""
lit_fams = set(src_df.loc[lit_mask & (src_df["family"] != ""), "family"])
sel      = lit_mask | src_df["family"].isin(lit_fams)
litfam   = src_df[sel].sort_values(["family", "application_date"])

# ============================================================
# Same-case mark (case_group) + shared-family mark (family_root)
# ------------------------------------------------------------
# Basis of case_group = case_number in the docket:
#   each docket row = one case × one patent in suit;
#   patents listed under the same case number = asserted together in the same lawsuit → same case group.
#   Transitive merge across cases (union-find): A and B in case 1, B and C in case 2
#   → A/B/C form one "case group" (one litigation campaign; common for serial NPE suits and multiple ANDA cases).
#   Group numbers L01, L02, … (groups with more members come first).
# PTAB proceedings (IPR/PGR/reexamination) have no district-court case number → the record's own case_group stays empty;
#   if a family member is in a district-court case, the record inherits that family's case group; pure PTAB families are grouped by family_root.
# family_root = shared family root application number (the family column without the " (n=N)" suffix; singleton = own patent_id)
# ============================================================
if "DOCKET" not in globals():        # load it here when this cell runs alone (same cleaning rules as 2.8)
    _cand = [f"{LITIG_DIR}/patents.dta.zip", f"{LITIG_DIR}/patents.dta"]
    _p = next((p for p in _cand if os.path.exists(p)), None)
    assert _p, f"Docket file not found: {LITIG_DIR}"
    DOCKET = pd.read_stata(_p)
    DOCKET["patent"] = DOCKET["patent"].astype(str)
    DOCKET = DOCKET[~DOCKET["patent"].str.contains("/")]
    DOCKET["_key"] = DOCKET["patent"].str.replace(r"\D", "", regex=True).str.lstrip("0")
    DOCKET = (DOCKET[DOCKET["_key"] != ""].drop_duplicates(["_key", "case_number"])
              [["_key", "case_number", "date_filed"]])

_keys = litfam["patent_id"].str.extract(r"^US(\d+)", expand=False).str.lstrip("0")
_sub  = DOCKET[DOCKET["_key"].isin(set(_keys.dropna()))]

# --- union-find: connect patents into case groups, with "same case number" as the edge ---
_parent = {}
def _find(x):
    while _parent.setdefault(x, x) != x:
        _parent[x] = _parent[_parent[x]]   # path halving, to avoid long chains
        x = _parent[x]
    return x
def _union(a, b):
    _parent[_find(a)] = _find(b)
for _, _grp in _sub.groupby("case_number"):
    _ks = _grp["_key"].tolist()
    for _k in _ks[1:]:
        _ = _union(_ks[0], _k)                 # connect all patents with the same case number

# --- Group numbers: groups with more members first (L01, L02, …) ---
_comp = {}
for _k in _sub["_key"].unique():
    _comp.setdefault(_find(_k), []).append(_k)
_order = sorted(_comp.items(), key=lambda kv: (-len(kv[1]), kv[0]))
_label = {k: f"L{i+1:02d}" for i, (_, ks) in enumerate(_order) for k in ks}

_case_list = _sub.groupby("_key")["case_number"].apply(lambda s: "; ".join(sorted(set(s))))
litfam["case_numbers"] = _keys.map(_case_list).fillna("")   # all case numbers that list this record (not cut)
litfam["case_group"]   = _keys.map(_label).fillna("")       # case group of this record

# --- Family members that are not in a district-court case inherit the case group of the litigated family members ---
_grpkey = litfam["family"].where(litfam["family"] != "", litfam["patent_id"])
_has = litfam["case_group"] != ""
_fam2grp = (litfam.loc[_has].assign(_g=_grpkey[_has])
            .groupby("_g")["case_group"]
            .agg(lambda s: "; ".join(sorted(set(s)))))
litfam["case_group"] = litfam["case_group"].where(_has, _grpkey.map(_fam2grp).fillna(""))

# --- Shared-family mark ---
litfam["family_root"] = (litfam["family"].str.split(" (", regex=False).str[0]
                         .where(litfam["family"] != "", litfam["patent_id"]))

# --- Case group summary ---
# the case_group of an inherited family member can be a combined value (a family in several case groups, such as the two Sysmex waves of suits),
# so split it (explode) before the summary; each group counts as one row
_cg = pd.DataFrame()
if (litfam["case_group"] != "").any():
    _cg = (litfam[litfam["case_group"] != ""]
           .assign(case_group=lambda d: d["case_group"].str.split("; "))
           .explode("case_group")
           .groupby("case_group")
           .agg(n_patents=("patent_id", "nunique"),
                n_case_numbers=("case_numbers", lambda s: len({c for x in s if x for c in x.split("; ")})),
                family_summary=("family_root", lambda s: "; ".join(sorted(set(s))[:3])))
           .sort_values("n_patents", ascending=False))
print("Case group summary (same group = same litigation campaign):")
_cg.head(15)

LITFAM_CSV = os.path.join(SAVE_DIR, "HPC_15to25_litigation_family.csv")
litfam[out_cols + ["family_root", "case_numbers", "case_group"]].to_csv(LITFAM_CSV, index=False)

litfam_stats = pd.DataFrame([
    {"item": "litigated patents (litigation not empty)",      "n_rows": int(lit_mask.sum())},
    {"item": "families of the litigated patents in a family", "n_rows": len(lit_fams)},
    {"item": "non-litigated members added by the family",     "n_rows": int((sel & ~lit_mask).sum())},
    {"item": "litigated-family subset (output)",              "n_rows": int(sel.sum())},
])
litfam_stats

# family-level summary table: size and litigation summary of each litigated family
roll = (litfam.assign(litigation_summary=lit_mask[litfam.index])
        .groupby(litfam["family"].where(litfam["family"] != "", litfam["patent_id"]))
        .agg(n_members=("patent_id", "size"),
             litigation_members=("litigation_summary", "sum"),
             earliest_filing=("application_date", "min"),
             litigation_note=("litigation", lambda s: next((x for x in s if x), "")[:60]),
             case_group=("case_group", lambda s: "; ".join(sorted({g for x in s if x
                                                             for g in x.split("; ")}))))
        .sort_values(["litigation_members", "n_members"], ascending=False))
roll.index.name = "family_root_or_patent_id"
roll
print(f"Litigated-family subset → {LITFAM_CSV}")

In [ ]:
# ── 2.11  How a case group and a shared family are decided (two examples) ──
# Case group (case_group)
#   Basis = case_number in the USPTO docket. Each docket row = one case × one patent in suit,
#   so "patents listed under the same case number" were asserted together in the same lawsuit → same case group.
#   Transitive across cases: patents A and B in case 1, B and C in case 2 → A/B/C form one case group
#   (union-find connected components; serial suits of one plaintiff against several defendants are merged correctly).
#   PTAB (IPR/PGR/reexamination) has no district-court case number → not part of the merge; the shared family does the grouping.
# Shared family (family_root)
#   Basis = the priority chain in applications_claiming_priority. Under 37 CFR 1.78, a continuation
#   must state its parent application in the description and claim the same priority date, so the chain of each member contains
#   the same "earliest application"; it is the family root, and the same root = shared family (≈ simple family).
# Needs 2.8 (family_root) and 2.10 (_sub, _keys, litfam).
rowsA, evidence_b_family_root = None, None
if len(_sub):
    # Evidence A: the case that lists the most records of this corpus, with its docket rows
    top_case = _sub.groupby("case_number")["_key"].nunique().idxmax()
    rowsA = (_sub[_sub["case_number"] == top_case]
             .merge(litfam[["patent_id", "title"]].assign(_key=_keys.values), on="_key"))
    rowsA["date_filed"] = pd.to_datetime(rowsA["date_filed"]).dt.date
    rowsA["title"] = rowsA["title"].str.slice(0, 60)
    rowsA = rowsA[["case_number", "date_filed", "patent_id", "title"]].reset_index(drop=True)
    print(f"Evidence A | records listed under the same case number {top_case} (one case group):")
else:
    print("Evidence A | no record of this run is in the docket.")
rowsA

# Evidence B: the litigated family with the most members; the root is parsed for each member separately
_fams = litfam.loc[litfam["family"] != "", "family"]
if len(_fams):
    top_fam = _fams.value_counts().idxmax()
    memb = litfam[litfam["family"] == top_fam].copy()
    memb["family_root_parsed"] = [family_root(r, p) for r, p in
                                  zip(memb["applications_claiming_priority"], memb["patent_id"])]
    evidence_b_family_root = (memb[["patent_id", "application_date", "family_root_parsed", "litigation"]]
                              .sort_values("application_date").reset_index(drop=True))
    print(f"Evidence B | family {top_fam}: every member, parsed separately, gives the same root application number:")
else:
    print("Evidence B | no litigated family with two or more members in this run.")
evidence_b_family_root

### 2.12–2.13  釋出用清單

`provenance/removed_homonym.csv` 列出每一筆被移除紀錄的語意組、證據類型與找到的全稱，使用者可以逐筆查核，或改用自己的規則。`provenance/candidate_ids.csv` 列出所有候選紀錄。2.13 最後釋放本節占用的記憶體，第 3 節會重新讀入候選檔。

In [ ]:
# ── 2.12  List of the removed records with the reason for each ─────────────
# Patent number and decision columns only, no full text, so that users can check each removal or decide again with their own rule.
REMOVED_CSV = os.path.join(PROV_DIR, "removed_homonym.csv")
removed = df.loc[~df["is_hpc_sense"]].copy()

# the display name is "domain: English name"; the English name after the colon is the sense_label
SENSE_LABEL_EN = {g: v[0].split(": ", 1)[-1] for g, v in SENSE_GROUPS.items()}

def basis_code(b):
    if b.startswith("(1)"):
        return "definition_multi" if "several senses" in b else "definition"
    if b.startswith("(2)"):
        return "phrase"
    return "none"

removed_out = pd.DataFrame({
    "patent_id":        removed["patent_id"],
    "sense_group":      removed["sense"],
    "sense_label":      removed["sense"].map(SENSE_LABEL_EN),
    "evidence_basis":   removed["basis"].map(basis_code),
    "sh_long_forms":    removed["sh_defs"],
    "phrase_count":     removed.apply(lambda r: int(r[f"n_{r['sense']}"]), axis=1),
    "hpc_mentions":     removed["hpc_mentions"].astype(int),
    "ipc_section":      removed["ipc_section"],
    "main_ipc_4":       removed["main_ipc_4"],
    "application_year": removed["application_year"],
    "title":            removed["title"],
}).sort_values(["sense_group", "patent_id"]).reset_index(drop=True)
removed_out.to_csv(REMOVED_CSV, index=False)
print(f"List of homonym removals → {REMOVED_CSV} ({len(removed_out):,} records × {removed_out.shape[1]} columns)")
print(f"Check: removed {len(removed_out):,} + kept {N_CLEAN:,} = {len(removed_out) + N_CLEAN:,}; raw {N_RAW:,}")

# Sense group x decision basis
removed_tab = pd.crosstab(removed_out["sense_label"], removed_out["evidence_basis"], margins=True, margins_name="total")
removed_tab

In [ ]:
# ── 2.13  Candidate identifiers and the hand-off to section 3 ──────────────
# Writes the identifiers of the candidates (no full text) and counts the records with all five required
# fields, the rule that section 3.3 applies. When an earlier run is reused, its released identifiers must
# all be in this five-field subset.
CANDIDATE_IDS_CSV = os.path.join(PROV_DIR, "candidate_ids.csv")
cand_ids = kept["patent_id"]
cand_ids.to_frame().to_csv(CANDIDATE_IDS_CSV, index=False)
print(f"Candidate identifiers -> {CANDIDATE_IDS_CSV} ({len(cand_ids):,} records)")

REQUIRED_5 = ["title", "abstract", "background", "detailed_description", "claims_text"]
complete_mask = np.logical_and.reduce([kept[c].fillna("").str.strip() != "" for c in REQUIRED_5])
complete_set  = set(kept.loc[complete_mask, "patent_id"])
print(f"Candidates with all five required fields: {len(complete_set):,}; without: {len(cand_ids) - len(complete_set):,}")

handoff_df = None
_rel_path = os.path.join(SAVE_DIR, "reused_released_corpus_ids.csv")
if os.path.exists(_rel_path):
    _rel = set(pd.read_csv(_rel_path, dtype=str)["patent_id"])
    handoff_df = pd.DataFrame([{
        "check": "released identifiers of the reused run", "records": len(_rel),
        "reference": "five-field subset of this run", "reference_records": len(complete_set),
        "only_in_reused": len(_rel - complete_set), "result": "subset" if _rel <= complete_set else "NOT a subset"}])
handoff_df

# Free the memory of the screening step: section 3 reads the candidate file again
for _n in ("txt", "pat", "combo", "present", "length", "best", "removed", "src_df", "dedup", "ann", "kept"):
    _ = globals().pop(_n, None)
del df
_ = gc.collect()

### 第 2 節結果（釋出版本）

21,947 筆中有 5,513 筆（25.1%）顯示 HPC 是其他意思而被移除，留下 16,434 筆候選。被移除最多的是羥丙基纖維素（3,901 筆）與高壓壓縮機（1,407 筆），其次是造血前驅細胞（116 筆）。被移除的紀錄中，2,555 筆依據明確的定義，2,958 筆依據全文片語。圖 2.3 顯示移除集中在哪裡：F 部移除 82.0%，A 部 47.2%，C 部 38.9%，G 部與 H 部只移除 3.1% 與 4.0%。16,434 筆候選中有 7,998 筆找不到任何意思的文字證據，依規則保留；這些紀錄多半缺少出現縮寫的那幾個段落。

## 3  語料規則：16,434 → 7,158 → 6,535 筆

兩條規則把候選紀錄變成釋出語料。第一條要求 title、abstract、background、detailed description 與 claims 都有文字（3.3）。第二條要求全文出現縮寫的展開語 `high performance comput*`，連字號與空白的寫法都算（3.5）。兩條規則每次執行都重新判定，不需要任何中間語料檔。

summary 段不列為必要欄位：發明概要不是核准說明書必備的部分，本語料大多數紀錄也沒有這一段。3.7 會在 summary 缺漏或過短時，以 detailed description 的前 2,000 個字元補上，並以 `summary_is_substituted` 標記。

In [ ]:
# ── 3.1  Load the candidates (with a check against section 2) ─────────────────
with timed("3.1  Load the candidates"):
    df_raw = pd.read_csv(CLEAN_PATH, low_memory=False)

n_dup = df_raw["patent_id"].duplicated().sum()
if n_dup:
    print(f"WARNING: dropping {n_dup} duplicated patent_id rows")
    df_raw = df_raw.drop_duplicates(subset="patent_id").reset_index(drop=True)

# The candidate file must hold exactly the records that section 2 kept.
if "N_CLEAN" in globals() and len(df_raw) != N_CLEAN:
    raise RuntimeError(f"The candidate file has {len(df_raw):,} records, but section 2 kept {N_CLEAN:,}. "
                       "Run section 2 again with the same configuration.")
N_INPUT = len(df_raw)
print(f"Candidates: {N_INPUT:,} records x {len(df_raw.columns)} columns ({os.path.basename(CLEAN_PATH)})")
df = df_raw

In [ ]:
# ── 3.2  Descriptive statistics of the candidates (before the corpus rules) ──

# Safety: make sure that patent_id is unique (all downstream merges assume that it is unique)
n_dup = df["patent_id"].duplicated().sum()
if n_dup:
    print(f"WARNING: dropping {n_dup} duplicated patent_id rows")
    df = df.drop_duplicates(subset="patent_id").reset_index(drop=True)

print(f"records : {len(df):,}")
print(f"columns : {len(df.columns)}")

print(f"\n{'='*68}")
print("per-column validity")
print(f"{'='*68}")
for col in df.columns:
    if df[col].dtype == object:
        valid = (df[col].fillna("").astype(str).str.strip() != "").sum()
    else:
        valid = df[col].notna().sum()
    print(f"  {col:42s} {valid:>7,}  ({valid/len(df)*100:5.1f}%)")

TEXT_STAT_COLS = [c for c in ["abstract","background","summary","detailed_description","claims_text"] if c in df.columns]
print(f"\n{'='*68}")
print("word-count statistics (non-empty rows)")
print(f"{'='*68}")
print(f"  {'column':26s} {'n':>7} {'mean':>8} {'median':>8} {'min':>6} {'max':>9}")
for col in TEXT_STAT_COLS:
    filled = df[col].fillna("").astype(str).str.strip()
    filled = filled[filled != ""]
    w = filled.str.split().str.len()
    print(f"  {col:26s} {len(w):>7,} {w.mean():>8.0f} {w.median():>8.0f} {w.min():>6,} {w.max():>9,}")

print(f"\n{'='*68}")
print("application_year distribution")
print(f"{'='*68}")
for yr, cnt in df["application_year"].value_counts().sort_index().items():
    print(f"  {yr}  {cnt:>5,}  {'█' * (cnt // 40)}")

print(f"\n{'='*68}")
print("main_ipc_4 top 10")
print(f"{'='*68}")
for ipc, cnt in df["main_ipc_4"].value_counts().head(10).items():
    print(f"  {ipc:8s} {cnt:>5,} ({cnt/len(df)*100:4.1f}%)  {'█' * (cnt // 40)}")

print(f"\n{'='*68}")
print("current_assignee top 10")
print(f"{'='*68}")
for co, cnt in df["current_assignee"].value_counts().head(10).items():
    print(f"  {co[:40]:42s} {cnt:>5,}")

In [ ]:
# ── 3.3  Five-field completeness filter ──────────────────────────────────────
# A record stays when title, abstract, background, detailed_description and claims_text all have text.
# summary is exempt by design: a summary of the invention is not a required part of a granted
# specification, and requiring it would discard records that are otherwise complete. Because the parser
# writes "N/A" when a section heading does not match one of its standard headings, this filter measures
# the success of section extraction, not the completeness of the patents themselves.
REQUIRED_FULL = ["title", "abstract", "background",
                 "detailed_description", "claims_text"]
_complete = pd.Series(True, index=df_raw.index)
for _c in REQUIRED_FULL:
    _complete &= df_raw[_c].fillna("").astype(str).str.strip().str.len() > 0

# For 3.4: keep a snapshot of the minimum columns before the filter and a matrix of the missing fields (selection-bias comparison)
_BIAS_COLS = [c for c in ("patent_id", "main_ipc_4", "application_year",
                          "current_assignee") if c in df_raw.columns]
_bias_src = df_raw[_BIAS_COLS].copy()
_bias_src["_kept"] = _complete.values
_bias_miss = pd.DataFrame(
    {c: df_raw[c].fillna("").astype(str).str.strip().str.len().eq(0).values
     for c in REQUIRED_FULL})

print(f"before completeness filter  : {len(df_raw):>7,}")
print(f"  − one of five fields empty: {(~_complete).sum():>7,}")
df_raw = df_raw[_complete].reset_index(drop=True)
print(f"after completeness filter   : {len(df_raw):>7,}")

N_COMPLETE = len(df_raw)
df = df_raw

### 3.4  完整性過濾移除了哪些紀錄

完整性過濾移除超過一半的候選，因此在往下走之前，先比較被移除與被保留的紀錄。由於解析器在標題對不上標準標題時寫 `N/A`，移除率反映的是段落擷取：段落標題採用其他慣例的說明書（藥學與化學專利常見，有些公司的撰寫格式也是如此）比較容易被移除。下列各表依缺漏欄位、分類部與子類、申請年、權利人集中度做比較，只描述分布，不做檢定。

In [ ]:
# ── 3.4  Who the completeness filter removes (CPU only, read-only) ──────────
# The filter removes more than half of the candidates, so the removed and the kept records are compared on
# the missing field, class section and subclass, application year and assignee. Distributions only: no
# statistical test and no inference. All tables are written to tables/.
import os
import numpy as np, pandas as pd

for _n in ("_bias_src", "_bias_miss"):
    if _n not in globals():
        raise RuntimeError("Run 3.3 first (this cell uses the snapshot that 3.3 keeps before the filter).")

_b      = _bias_src
_keep   = _b["_kept"].to_numpy(dtype=bool)
_n_all  = len(_b); _n_keep = int(_keep.sum()); _n_drop = _n_all - _n_keep
print(f"Completeness filter: {_n_all:,} → {_n_keep:,}"
      f" (removed {_n_drop:,} = {_n_drop / _n_all * 100:.1f}%)\n")

# ── Table A  Which column is missing: the structure of the removal reasons ───
_rows = []
for _c in REQUIRED_FULL:
    _m = _bias_miss[_c].to_numpy(dtype=bool)
    _rows.append({"required_field": _c,
                  "records_missing": int(_m.sum()),
                  "share_of_all_pct": round(_m.mean() * 100, 1),
                  "share_of_removed_pct": round(_m[~_keep].mean() * 100, 1)})
_tabA = pd.DataFrame(_rows).sort_values("records_missing", ascending=False)
print("Table A  Missing values of each required field")
print(_tabA.to_string(index=False))
_nmiss = _bias_miss.sum(axis=1).to_numpy()
_dist  = pd.Series(_nmiss[~_keep]).value_counts().sort_index()
print("\n  Removed records by the number of missing fields:",
      " | ".join(f"{k} field(s) {v:,} records" for k, v in _dist.items()))
_tabA.to_csv(os.path.join(TBL_DIR, "completeness_missing_by_field.csv"), index=False)

# ── Table B  Removal rate by patent classification section (first letter of the CPC) ───
def _rate_table(key, label, min_n=0):
    _t = (pd.DataFrame({label: key, "kept": _keep})
          .groupby(label, dropna=False)
          .agg(before=("kept", "size"), after=("kept", "sum")))
    _t = _t[_t["before"] >= min_n].copy()
    _t["removed"]  = _t["before"] - _t["after"]
    _t["removal_rate_pct"] = (_t["removed"] / _t["before"] * 100).round(1)
    _t["share_before_pct"] = (_t["before"] / _n_all * 100).round(1)
    _t["share_after_pct"] = (_t["after"] / max(_n_keep, 1) * 100).round(1)
    return _t.sort_values("before", ascending=False)

if "main_ipc_4" in _b.columns:
    _sec = _b["main_ipc_4"].fillna("").astype(str).str[:1].replace("", "(missing)")
    _tabB = _rate_table(_sec, "cpc_section")
    print("\nTable B  Removal rate by class section (first letter of the CPC code)")
    print(_tabB.to_string())
    _spread = _tabB.loc[_tabB["before"] >= 100, "removal_rate_pct"]
    print(f"\n  Range of the removal rate for sections with ≥100 records: {_spread.min():.1f}% – {_spread.max():.1f}%")
    _tabB.to_csv(os.path.join(TBL_DIR, "completeness_bias_by_ipc_section.csv"))

    _tabB4 = _rate_table(_b["main_ipc_4"].fillna("(missing)").astype(str),
                         "cpc_subclass", min_n=100)
    print("\n  Removal rate of the 12 largest four-character subclasses")
    print(_tabB4.head(12).to_string())
    _tabB4.to_csv(os.path.join(TBL_DIR, "completeness_bias_by_ipc4.csv"))

# ── Table C  Removal rate by application year ───────────────────────────────
if "application_year" in _b.columns:
    _yr = pd.to_numeric(_b["application_year"], errors="coerce")
    _tabC = _rate_table(_yr.fillna(-1).astype(int), "application_year").sort_index()
    print("\nTable C  Removal rate by application year")
    print(_tabC.to_string())
    _y = _tabC.loc[_tabC.index >= 2015, "removal_rate_pct"]
    if len(_y):
        print(f"\n  Range of the yearly removal rate from 2015 on: {_y.min():.1f}% – {_y.max():.1f}%")
    _tabC.to_csv(os.path.join(TBL_DIR, "completeness_bias_by_year.csv"))

# ── Table D  Comparison of assignee concentration ───────────────────────────
if "current_assignee" in _b.columns:
    _a = _b["current_assignee"].fillna("").astype(str).str.strip()
    _rows = []
    for _name, _mask in ((f"before filter ({_n_all:,})",  np.ones(_n_all, bool)),
                         (f"kept ({_n_keep:,})",  _keep),
                         (f"removed ({_n_drop:,})",  ~_keep)):
        _s = _a[_mask]; _s = _s[_s != ""]
        _vc = _s.value_counts()
        _rows.append({"subset": _name, "records": int(_mask.sum()),
                      "distinct_assignees": int(_vc.size),
                      "top10_share_pct": round(_vc.head(10).sum() / max(len(_s), 1) * 100, 1),
                      "single_record_assignees_pct": round((_vc == 1).mean() * 100, 1)})
    _tabD = pd.DataFrame(_rows)
    print("\nTable D  Comparison of assignee concentration")
    print(_tabD.to_string(index=False))
    _tabD.to_csv(os.path.join(TBL_DIR, "completeness_bias_assignee.csv"), index=False)

    _top = _a[_keep].value_counts().head(10).index
    _cmp = pd.DataFrame({
        "records_kept": _a[_keep].value_counts().reindex(_top).fillna(0).astype(int),
        "records_removed": _a[~_keep].value_counts().reindex(_top).fillna(0).astype(int)})
    _cmp["assignee_removal_rate_pct"] = (_cmp["records_removed"] /
                          (_cmp["records_kept"] + _cmp["records_removed"]) * 100).round(1)
    print("\n  Removal rate of each of the 10 largest assignees of the kept corpus")
    print(_cmp.to_string())
    _cmp.to_csv(os.path.join(TBL_DIR, "completeness_bias_top_assignees.csv"))

print("\nsaved -> tables/completeness_*.csv")
# ── Everpure Inc (formerly Pure Storage): its specifications have no standard section headings ───
_e = _bias_src[_bias_src["current_assignee"].astype(str).str.strip() == "Everpure Inc"].copy()
_e["cpc_section"] = _e["main_ipc_4"].fillna("").astype(str).str[:1].replace("", "(missing)")
print("Everpure Inc:", len(_e), "records")
if len(_e):
    print(pd.crosstab(_e["cpc_section"], _e["_kept"].map({True: "kept", False: "removed"}), margins=True))
print("\nFour-character class distribution of the kept records:")
print(_e.loc[_e["_kept"], "main_ipc_4"].value_counts().head(8).to_string())
print("\nFour-character class distribution of the removed records:")
print(_e.loc[~_e["_kept"], "main_ipc_4"].value_counts().head(8).to_string())

# Also check whether the year gradient comes from A/C contamination
_x = _bias_src.copy()
_x["section"] = _x["main_ipc_4"].fillna("").astype(str).str[:1]
_x["computing"] = _x["section"].isin(["G", "H"])
print("\nComputing vs non-computing classes, removal rate by year:")
print(_x.pivot_table(index=pd.to_numeric(_x["application_year"], errors="coerce"),
                     columns="computing", values="_kept",
                     aggfunc=lambda s: round((1 - s.mean()) * 100, 1)).to_string())

# ── Yearly removal rate of the computing sections (G, H) and of the other sections, written to a file ───
(_x.pivot_table(index=pd.to_numeric(_x["application_year"], errors="coerce"),
                columns="computing", values="_kept",
                aggfunc=lambda s: round((1 - s.mean()) * 100, 1))
   .rename(columns={True: "computing_GH_removed_pct", False: "other_sections_removed_pct"})
   .rename_axis("application_year")
   .to_csv(os.path.join(TBL_DIR, "completeness_bias_by_year_computing.csv")))
print("saved -> tables/completeness_bias_by_year_computing.csv")

### 3.5–3.6  收錄規則與收錄證據

檢索用的是縮寫，規則要求的是展開語，所以每一筆釋出紀錄兩者都有。兩項輸出說明這條規則移除的是其他意思，而不是隨機的紀錄。`evidence_flags.csv` 記錄每一筆釋出紀錄的文字是否正式把縮寫和展開語綁在一起（`sense_basis = "definition"`），或只出現展開語（`"phrase"`）；這個檔只釋出、不套用，使用者可以自行取更嚴格的子集。若規則只是比對表面字串，各分類部的排除率應該差不多；實際上 A 部與 C 部的排除率遠高於 G 部與 H 部。

In [ ]:
# ── 3.5  Inclusion rule: the expanded phrase must occur ──────────────────────
# One rule defines the released corpus: the text must contain the expanded phrase of the abbreviation,
# "high performance comput*" (hyphen and space variants, endings compute, computer, computing, computation).
# The search itself used the abbreviation HPC, so a released record contains both the abbreviation (somewhere
# in the Google Patents record) and the expanded phrase (in the parsed text). The rule runs again in every run.
EXPANSION_RE = re.compile(r"high[\s\-]{0,2}performance[\s\-]{0,2}comput", re.I)
TEXT_FIELDS  = ["title", "abstract", "background", "summary",
                "detailed_description", "claims_text"]
_fields = [c for c in TEXT_FIELDS if c in df_raw.columns]

with timed("3.5  Inclusion rule (expanded phrase)", n_items=len(df_raw)):
    _joined = df_raw[_fields].fillna("").astype(str).agg(" ".join, axis=1)
    has_expansion = _joined.str.contains(EXPANSION_RE)

df = df_raw[has_expansion].reset_index(drop=True)
N_RELEASED, N_EXCLUDED = len(df), int((~has_expansion).sum())
print(f"records with five complete fields : {len(df_raw):>6,}")
print(f"  without the expanded phrase     : {N_EXCLUDED:>6,}")
print(f"released corpus                   : {N_RELEASED:>6,}")

# The complete records that the rule excludes are released as well (full text, normalised again in 9.3)
_excl = add_ipc3(df_raw[~has_expansion].reset_index(drop=True))
_excl.to_csv(os.path.join(SAVE_DIR, "excluded_no_expansion.csv"), index=False)
print(f"saved -> excluded_no_expansion.csv ({len(_excl):,} records x {len(_excl.columns)} columns)")

# Identifiers of the released corpus: the fingerprint that all later steps check against
pd.DataFrame({"patent_id": df["patent_id"]}).to_csv(os.path.join(SAVE_DIR, "released_corpus_ids.csv"), index=False)
print(f"saved -> released_corpus_ids.csv ({N_RELEASED:,} records)")

# With a reused run: the corpus of this run must be the same set of records
_rel_path = os.path.join(SAVE_DIR, "reused_released_corpus_ids.csv")
if os.path.exists(_rel_path):
    _old, _new = set(pd.read_csv(_rel_path, dtype=str)["patent_id"]), set(df["patent_id"])
    print(f"\nCheck against the reused run: {len(_new):,} here, {len(_old):,} there, "
          f"only here {len(_new - _old)}, only there {len(_old - _new)} -> "
          f"{'identical' if _new == _old else 'DIFFERENT'}")
    if _new != _old:
        raise RuntimeError("The released corpus differs from the reused run; its generated layers cannot be reused. "
                           "Set REUSE_DIR = None or use the input file of that run.")

In [ ]:
# ── 3.6  Inclusion evidence of each record and exclusion rate by section (CPU only) ──
# Two pieces of evidence that the rule removes the other senses of the abbreviation and not random records:
# (1) evidence_flags.csv records why each record is included. It is released, not applied: users who want
#     only records that define the abbreviation as high-performance computing keep sense_basis == "definition".
# (2) Exclusion rate by class section. A rule that matched a surface string would exclude every section at
#     the same rate; a rule that matches the computing sense must exclude much more in sections C and A
#     (hydroxypropyl cellulose, hematopoietic progenitor cell) than in G and H.
import os
import numpy as np, pandas as pd

# Definition form: the document ties the abbreviation to the expanded phrase ("high-performance computing (HPC)" or the reverse)
DEFINITION_RE = re.compile(
    r"high[\s\-]{0,2}performance[\s\-]{0,2}comput\w*[^()\n]{0,60}\(\s*HPC\s*\)"
    r"|\bHPC\b\s*[\(,:]\s*(?:i\.e\.,?\s*|that is,?\s*|or\s*)?"
    r"high[\s\-]{0,2}performance[\s\-]{0,2}comput", re.I)
HPC_ABBR_RE = re.compile(r"\bHPC\b")

with timed("3.6  Inclusion evidence flags", n_items=len(df)):
    _txt = df[_fields].fillna("").astype(str).agg(" ".join, axis=1)
    _exp_n  = _txt.map(lambda t: len(EXPANSION_RE.findall(t)))
    _abb_n  = _txt.map(lambda t: len(HPC_ABBR_RE.findall(t)))
    _hasdef = _txt.map(lambda t: bool(DEFINITION_RE.search(t)))
    _basis  = np.where(_hasdef, "definition",
              np.where(_exp_n.values > 0, "phrase", "no_evidence"))

ev = pd.DataFrame({"patent_id": df["patent_id"].values,
                   "sense_basis": _basis,
                   "expansion_mentions": _exp_n.values,
                   "hpc_mentions": _abb_n.values})
ev.to_csv(os.path.join(SAVE_DIR, "evidence_flags.csv"), index=False)

print("Inclusion evidence (sense_basis)")
print("-" * 62)
for _b in ("definition", "phrase", "no_evidence"):
    _m = ev["sense_basis"] == _b
    if not _m.any():
        print(f"  {_b:12} {0:>6,}  ( 0.0%)"); continue
    _e = ev.loc[_m, "expansion_mentions"]
    print(f"  {_b:12} {int(_m.sum()):>6,}  ({_m.mean()*100:>4.1f}%)"
          f"  expanded-phrase mentions median {_e.median():>4.0f} | mean {_e.mean():>5.2f}"
          f" | three or more {(_e >= 3).mean()*100:>4.1f}% | exactly one {int((_e == 1).sum()):>5,}")
_one = ev[ev["expansion_mentions"] == 1]
if len(_one):
    _top = _one["sense_basis"].value_counts()
    print(f"\n  {len(_one):,} records in the whole corpus mention it only once; of these, "
          f"{_top.index[0]} has {int(_top.iloc[0]):,}")

# ── Differential exclusion rate: if the rule caught a surface string, the exclusion rates of all sections would be the same ───
_bef = df_raw["main_ipc_4"].astype(str).str[0].replace("", "?")
_aft = df["main_ipc_4"].astype(str).str[0].replace("", "?")
_b, _a = _bef.value_counts(), _aft.value_counts()
_rows = []
for _s in sorted(set(_b.index)):
    _nb, _na = int(_b.get(_s, 0)), int(_a.get(_s, 0))
    _rows.append({"ipc_section": _s, "before": _nb, "after": _na,
                  "removed": _nb - _na,
                  "exclusion_rate_pct": round((_nb - _na) / _nb * 100, 1) if _nb else None})
t_excl = pd.DataFrame(_rows).sort_values("before", ascending=False)
t_excl.to_csv(os.path.join(SAVE_DIR, "differential_exclusion.csv"), index=False)

print(f"\nDifferential exclusion rate ({len(df_raw):,} -> {len(df):,})")
print("-" * 62)
print(f"  {'section':<6}{'before':>8}{'after':>8}{'removed':>9}{'rate (%)':>9}")
for _r in t_excl.itertuples():
    print(f"  {_r.ipc_section:<8}{_r.before:>8,}{_r.after:>8,}{_r.removed:>9,}"
          f"{_r.exclusion_rate_pct:>8.1f}%")
_noncomp_b = int(_b[[s for s in _b.index if s not in ("G", "H")]].sum())
_noncomp_a = int(_a[[s for s in _a.index if s not in ("G", "H")]].sum())
print(f"\n  Share of the non-computing sections (other than G, H): {_noncomp_b/len(df_raw)*100:.1f}% "
      f"-> {_noncomp_a/len(df):.1%}")
print("saved -> evidence_flags.csv, differential_exclusion.csv")

### 3.7–3.8  工作表與涉訟家族

`m1` 存放生成步驟要讀的欄位、補齊後的 summary，以及可用段落數（0 到 3）。3.8 從釋出語料推導涉訟家族：凡是家族中至少一筆有訴訟註記，整個家族都納入，並依註記中的案號分組。

In [ ]:
# ── 3.7  Working table m1 for the generation steps ───────────────────────────
KEEP_COLS = ["patent_id","title","abstract","current_assignee",
             "main_ipc_4","main_ipc_description_4","application_year",
             "background","summary","detailed_description","claims_text"]
TEXT_COLS = ["title","abstract","background","summary","detailed_description"]

df = add_ipc3(df)                      # add the three-character class to the released corpus
m1 = df[[c for c in KEEP_COLS if c in df.columns]].copy()
m1 = add_ipc3(m1)
for col in TEXT_COLS:
    if col in m1.columns:
        m1[col] = m1[col].fillna("").astype(str).str.strip()

# summary fill: when `summary` is missing or too short, take the first
# 2,000 characters of `detailed_description` as a pseudo-summary.
# Many records have no summary section, so the values are converted to strings before len() is applied.
_s = m1["summary"].fillna("").astype(str) if "summary" in m1.columns else pd.Series("", index=m1.index)
_d = m1["detailed_description"].fillna("").astype(str)
m1["summary_filled"] = np.where(
    (_s.str.len() < MIN_LEN) & (_d.str.len() >= MIN_LEN),
    _d.str[:2000], _s)
m1["summary_is_substituted"] = (m1["summary_filled"] != _s)

# sections_usable: the number of usable sections among the three sections (0-3)
m1["sections_usable"] = (
    (m1["background"].str.len()           >= MIN_LEN).astype(int) +
    (m1["summary_filled"].str.len()       >= MIN_LEN).astype(int) +
    (m1["detailed_description"].str.len() >= MIN_LEN).astype(int)
)

print(f"cohort size          : {len(m1):,}")
print(f"summary before fill  : {(m1['summary'].str.len()>=MIN_LEN).sum():,}")
print(f"summary after  fill  : {(m1['summary_filled'].str.len()>=MIN_LEN).sum():,}")
print("\nsections_usable distribution (3 = all sections usable):")
print(m1["sections_usable"].value_counts().sort_index().to_string())

In [ ]:
# ── 3.8  Litigated families in the released corpus (derived at run time) ──────
# The litigation (court/review records) and family (patent family) columns of the corpus are value-added fields that only this dataset has.
# The subset is derived from the candidate file in every run, like the released corpus in 3.5.
#
# Derivation rules
#   seed         litigation column not empty: the patent itself has a record of a district-court case, of an inter partes review
#                (inter partes review, IPR) or of a reexamination
#   family_root  the family root number in the family column ("US13/750696 (n=2)" gives "US13/750696");
#                a record without a family is a family of its own, written as self:<patent_id>
#   subset       all members of each family that contains a seed. A continuation and its parent application have
#                almost the same specification text and fall at the same position of the semantic landscape, so the whole family is included
#   case_group   the district-court case number parsed from the litigation text (for example 3:20-cv-03137).
#                One case number can cover more than one family (several patents of the same defendant are litigated together); family_root
#                cannot give this information. A record that has only an inter partes review/reexamination and no case number uses the proceeding type plus
#                the family root as its group key
CASE_NO_RE  = re.compile(r"\d+:\d{2}-cv-\d+", re.I)
PROCEEDINGS = (("district court", "DC"), ("IPR", "IPR"), ("reexamination", "REEX"))

with timed("3.8  Litigated-family subset", n_items=len(df)):
    _lit_src = pd.read_csv(CLEAN_PATH, usecols=["patent_id", "family", "litigation"],
                           dtype=str).fillna("")
    _lit_src = _lit_src[_lit_src["patent_id"].isin(set(df["patent_id"]))].copy()

    _root = _lit_src["family"].str.extract(r"^\s*(\S+)", expand=False).fillna("")
    _lit_src["family_root"]  = _root.where(_root != "", "self:" + _lit_src["patent_id"])
    _lit_src["case_numbers"] = _lit_src["litigation"].map(
        lambda t: ";".join(sorted(set(CASE_NO_RE.findall(t)))))

    _seed = _lit_src[_lit_src["litigation"].str.strip() != ""]
    _cg = {}
    for _r, _g in _seed.groupby("family_root"):
        _nums = sorted({n for s in _g["case_numbers"] if s for n in s.split(";")})
        if _nums:
            _cg[_r] = ";".join(_nums)
        else:
            _tags = [tag for kw, tag in PROCEEDINGS if any(kw in t for t in _g["litigation"])]
            _cg[_r] = f"{'/'.join(_tags) or 'UNKNOWN'}:{_r}"

    lit_family_df = _lit_src[_lit_src["family_root"].isin(set(_seed["family_root"]))].copy()
    lit_family_df["case_group"] = lit_family_df["family_root"].map(_cg)
    lit_family_df["is_seed"]    = lit_family_df["patent_id"].isin(set(_seed["patent_id"]))
    lit_family_df = (lit_family_df[["patent_id", "family_root", "case_numbers",
                                    "case_group", "is_seed", "litigation"]]
                     .sort_values(["case_group", "family_root", "patent_id"])
                     .reset_index(drop=True))

lit_family_df.to_csv(os.path.join(SAVE_DIR, "litigation_family.csv"), index=False)
print(f"seed (own record)         : {len(_seed):>4,}  ({len(_seed)/len(df)*100:.2f}% of the released corpus)")
print(f"subset with family members: {len(lit_family_df):>4,}  "
      f"({len(lit_family_df)/len(df)*100:.2f}% of the released corpus)")
print(f"families / case groups    : {lit_family_df['family_root'].nunique()} / "
      f"{lit_family_df['case_group'].nunique()}")
print("Note: the litigation column records the 'known' court and review events. An empty field does not mean that the patent was not litigated.")
print("\nsaved -> litigation_family.csv")

### 3.9–3.10  敘述統計表與漏斗圖

七張表描述釋出語料（建構漏斗、欄位完整性、文字長度、申請年、分類分布、權利人集中度、引用與請求項），每張都寫到 `tables/`。圖 3.1 呈現從爬蟲輸出到釋出語料的完整漏斗。

In [ ]:
# ── 3.9  Descriptive tables of the released corpus (CPU only) ────────────────
# Seven tables, each saved to tables/ as a CSV.
_rel = df


def _save(frame: pd.DataFrame, name: str) -> None:
    frame.to_csv(os.path.join(TBL_DIR, name), index=False)


def _wordcount(col: pd.Series) -> pd.Series:
    s = col.fillna("").astype(str).str.strip()
    return s[s != ""].str.split().str.len()


print(f"Released corpus: {len(_rel):,} records × {len(_rel.columns)} columns\n")

# ── Table 1  Corpus construction funnel ──────────────────────────────────────
_stages = [("candidates after the homonym screening", N_INPUT, ""),
           ("five-field completeness filter", N_COMPLETE,
            "title, abstract, background, detailed_description and claims_text all non-empty"),
           ("inclusion rule (expanded phrase)", len(_rel), "the text contains a variant of high performance comput*")]
t1 = pd.DataFrame(_stages, columns=["stage", "records", "criterion"])
t1["removed"] = [""] + [f"−{t1['records'][i-1] - t1['records'][i]:,}" for i in range(1, len(t1))]
print("Table 1  Corpus construction funnel"); print(t1.to_string(index=False)); _save(t1, "t1_funnel.csv")

# ── Table 2  Field completeness ──────────────────────────────────────────────
t2 = pd.DataFrame({
    "field": _rel.columns,
    "records_non_empty": [int((_rel[c].fillna("").astype(str).str.strip() != "").sum()) for c in _rel.columns],
})
t2["non_empty_pct"] = (t2["records_non_empty"] / len(_rel) * 100).round(1)
print(f"\nTable 2  Field completeness ({len(t2)} fields; complete fields: "
      f"{int((t2['non_empty_pct'] == 100).sum())})")
print(t2[t2["non_empty_pct"] < 100].to_string(index=False))
_save(t2, "t2_field_completeness.csv")

# ── Table 3  Text length ─────────────────────────────────────────────────────
_rows = []
for c in ["title", "abstract", "background", "summary",
          "detailed_description", "claims_text"]:
    if c not in _rel.columns:
        continue
    w = _wordcount(_rel[c])
    _rows.append([c, len(w), round(w.mean()), int(w.median()),
                  int(w.quantile(.10)), int(w.quantile(.90)), int(w.max())])
t3 = pd.DataFrame(_rows, columns=["field", "records_non_empty", "mean", "median", "P10", "P90", "max"])
print("\nTable 3  Text length (words)"); print(t3.to_string(index=False)); _save(t3, "t3_text_length.csv")

# ── Table 4  Application year distribution ───────────────────────────────────
_yr = pd.to_numeric(_rel["application_year"], errors="coerce")
t4 = _yr.value_counts().sort_index().rename_axis("application_year").reset_index(name="records")
t4["share_pct"] = (t4["records"] / len(_rel) * 100).round(1)
t4["note"] = np.where(t4["application_year"] >= 2023, "right-truncated (applications still under examination are not yet granted)", "")
print("\nTable 4  Application year distribution"); print(t4.to_string(index=False)); _save(t4, "t4_application_year.csv")

# ── Table 5  Patent classification distribution (CPC; the column name keeps ipc) ──────────────
_sec = _rel["main_ipc_4"].astype(str).str[0]
t5a = _sec.value_counts().rename_axis("cpc_section").reset_index(name="records")
t5a["share_pct"] = (t5a["records"] / len(_rel) * 100).round(1)
print("\nTable 5a  International Patent Classification section"); print(t5a.to_string(index=False))
_save(t5a, "t5a_ipc_section.csv")

_desc = (_rel.drop_duplicates("main_ipc_4")
         .set_index("main_ipc_4")["main_ipc_description_4"].astype(str))
t5b = _rel["main_ipc_4"].value_counts().head(15).rename_axis("cpc_subclass").reset_index(name="records")
t5b["share_pct"] = (t5b["records"] / len(_rel) * 100).round(1)
t5b["cpc_description"] = t5b["cpc_subclass"].map(_desc).str[:60]
print("\nTable 5b  The 15 largest four-character classes"); print(t5b.to_string(index=False))
_save(t5b, "t5b_ipc_top15.csv")

# ── Table 6  Assignee concentration ──────────────────────────────────────────
_ca = _rel["current_assignee"]
_vc = _ca.value_counts()
print(f"\nTable 6  Assignee concentration")
print(f"  distinct assignees        : {_ca.nunique():,}")
print(f"  share of the top 10       : {_vc.head(10).sum()/len(_rel)*100:.1f}%")
print(f"  share of the top 20       : {_vc.head(20).sum()/len(_rel)*100:.1f}%")
print(f"  companies with 1 record   : {int((_vc == 1).sum()):,}"
      f" ({(_vc == 1).sum()/_ca.nunique()*100:.1f}% of all companies)")
t6 = _vc.head(20).rename_axis("assignee").reset_index(name="records")
t6["share_pct"] = (t6["records"] / len(_rel) * 100).round(1)
t6.insert(0, "rank", range(1, len(t6) + 1))
print(t6.to_string(index=False)); _save(t6, "t6_top20_assignee.csv")

# ── Table 7  Citations, claims and value-added fields ────────────────────────
_rows = []
for c in ["citation_count", "claim_count"]:
    v = pd.to_numeric(_rel[c], errors="coerce")
    _rows.append([c, int(v.notna().sum()), round(v.mean(), 1), int(v.median()),
                  int(v.quantile(.90)), int(v.max())])
t7 = pd.DataFrame(_rows, columns=["field", "records_non_empty", "mean", "median", "P90", "max"])
print("\nTable 7  Citations and claims"); print(t7.to_string(index=False)); _save(t7, "t7_citation_claim.csv")

print(f"\nsaved -> {TBL_DIR}/ {len(os.listdir(TBL_DIR))} table files in total")

In [ ]:
# ── 3.10  Figure 3.1: from the crawler output to the released corpus ─────────
from matplotlib.ticker import MaxNLocator, StrMethodFormatter

_funnel = pd.DataFrame([
    ("crawler output (step 5)",             N_RAW,      ""),
    ("candidates",                          N_CLEAN,    "HPC has another sense"),
    ("five complete fields",                N_COMPLETE, "a required section was not extracted"),
    ("released corpus",                     N_RELEASED, "no \"high performance comput*\" in the text"),
], columns=["stage", "records", "removed_because"])
_funnel["removed"] = [0] + [int(_funnel["records"][i - 1] - _funnel["records"][i]) for i in range(1, len(_funnel))]
_funnel["kept_percent_of_input"] = (_funnel["records"] / N_RAW * 100).round(1)
_funnel.to_csv(os.path.join(TBL_DIR, "funnel_from_crawler_output.csv"), index=False)
_funnel

fig, ax = plt.subplots(figsize=(7.2, 2.6))
_y = np.arange(len(_funnel))[::-1]
_ = ax.barh(_y, _funnel["records"], color=PALETTE[0], height=0.62)
for yy, (_, r) in zip(_y, _funnel.iterrows()):
    _ = ax.text(r["records"] + N_RAW * 0.01, yy, f"{r['records']:,}", va="center", fontsize=8, color=INK)
    if r["removed"]:
        _ = ax.text(N_RAW * 1.17, yy, f"-{r['removed']:,}  {r['removed_because']}", va="center",
                    fontsize=7.5, color=INK_2)
_ = ax.set_yticks(_y, _funnel["stage"])
_ = ax.set_xlim(0, N_RAW * 1.15)
_ = ax.xaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 5, 10]))
_ = ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
_ = ax.set_xlabel("records")
_ = ax.set_title("Figure 3.1  Records at each step of the corpus construction")
fig.tight_layout()
_ = save_fig(fig, "fig_3_1_corpus_funnel")
plt.show()

### 第 3 節結果（釋出版本）

完整性過濾在 16,434 筆候選中保留 7,158 筆，移除 9,276 筆（56.4%）。最常缺漏的是 detailed description（占候選的 48.0%），其次是 background（39.5%）與 claims（8.6%）。移除並不平均：G 部被移除 40.1%、H 部 42.3%，A 部則是 90.1%、C 部 94.6%；依申請年看，2024 年升到 71.8%、2025 年 86.7%。權利人 Everpure Inc（前身為 Pure Storage）的 758 筆候選中有 588 筆（77.6%）被移除；在 Google Patents 頁面上查過的 US11128448B2，說明書完全沒有段落標題。

收錄規則接著移除 623 筆，留下釋出語料的 6,535 筆。排除率在 G 部是 5.1%、H 部 4.5%，在 A 部是 57.2%、C 部 84.4%，可見規則移除的是 HPC 其他意思所屬的部。釋出紀錄中，2,831 筆（43.3%）把縮寫定義為高效能運算，3,704 筆只出現展開語而沒有正式定義；4,958 筆只提到展開語一次。G 部與 H 部合計占語料的 95.4%，G06F 一個子類就占 42.1%，前十大權利人合計占 57.0%。2023 年起申請的案件偏少，因為其中許多仍在審查中；這個語料的任何逐年圖表都帶有這個右截斷。

## 4  分面：每筆三段短摘要

detailed description 的中位長度約 15,000 個 token，遠超過一個 prompt 能容納的長度，所以第一個生成步驟把各段落分開摘要。`meta-llama/Llama-3.1-8B-Instruct` 以 4-bit NF4 載入，三個段落各讀到 token 預算為止，各寫一段短文：

| 輸入段落 | 預算（token） | 輸出欄位 | 目標長度 |
|---|---|---|---|
| `background` | 1,500 | `bg_summary` | 2–3 句 |
| `summary_filled` | 1,500 | `inv_summary` | 2–3 句 |
| `detailed_description` | 2,000 | `tech_summary` | 3–4 句 |

解碼採貪婪解碼（`do_sample=False`、`repetition_penalty=1.15`、最多 200 個新 token）。少於 100 個字元的段落得到空的分面。checkpoint 每 50 筆寫一次，而且必須是目前語料、依相同順序的前綴，所以別的語料版本留下的 checkpoint 會讓執行停下。只有在分面還沒生成完時才載入模型。

In [ ]:
# ── 4.1  Facet model (loaded only when facets are still missing) ─────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CHECKPOINT_S1 = os.path.join(SAVE_DIR, "stage1_checkpoint.csv")
STAGE1_OUTPUT = os.path.join(SAVE_DIR, "stage1_summaries.csv")
_done_s1 = (len(pd.read_csv(CHECKPOINT_S1, usecols=["patent_id"])) if os.path.exists(CHECKPOINT_S1) else 0)
S1_TODO  = max(len(m1) - _done_s1, 0)

tokenizer, model = None, None
if S1_TODO:
    if not (HAS_CUDA or SMALL_MODELS):
        raise RuntimeError(f"{S1_TODO:,} facets are missing and this machine has no CUDA GPU. "
                           "Run on a GPU, set REUSE_DIR to a run with complete facets, or use PROFILE = 'sample'.")
    if S1_MODEL_ID.startswith("meta-llama/"):
        ensure_hf_login()                        # gated model
    tokenizer = AutoTokenizer.from_pretrained(S1_MODEL_ID)
    if S1_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            S1_MODEL_ID, quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.bfloat16,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(S1_MODEL_ID, torch_dtype=GEN_DTYPE).to(DEVICE)
    model = model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    print(f"{S1_MODEL_ID} loaded | {model.get_memory_footprint()/1e9:.2f} GB | "
          f"{S1_TODO:,} of {len(m1):,} records still need facets")
else:
    print(f"All {len(m1):,} facets are in {os.path.basename(CHECKPOINT_S1)}; the facet model is not loaded.")

In [ ]:
# ── 4.2  Facet prompts and batch generation ─────────────────────────────────
def truncate_text(text: str, max_tokens: int = 1500) -> str:
    toks = tokenizer.encode(text, add_special_tokens=False)
    if len(toks) > max_tokens:
        text = tokenizer.decode(toks[:max_tokens], skip_special_tokens=True)
    return text

# The prompts are part of the method and are released as they are (9.1 writes them to release_assets/prompts/).
SECTION_PROMPTS = {
    "BACKGROUND": (
        "You are a patent analyst specializing in High-Performance Computing (HPC). "
        "Summarize the BACKGROUND section below in 2-3 sentences as a single "
        "continuous paragraph. Do NOT use bullet points, headers, bold text, "
        "or markdown formatting. Do NOT start with phrases like 'Here is' or "
        "'This section describes'. Just write the summary directly.\n\n"
        "BACKGROUND:\n{text}\n\nSUMMARY:"
    ),
    "SUMMARY": (
        "You are a patent analyst specializing in High-Performance Computing (HPC). "
        "Condense the following patent SUMMARY into 2-3 sentences as a single "
        "continuous paragraph covering the core invention and key technical "
        "contribution. Do NOT use bullet points, headers, bold text, or markdown "
        "formatting. Do NOT start with phrases like 'Here is' or 'This patent'. "
        "Just write the summary directly.\n\nPATENT SUMMARY:\n{text}\n\nCONDENSED:"
    ),
    "DETAILED DESCRIPTION": (
        "You are a patent analyst specializing in High-Performance Computing (HPC). "
        "Summarize the DETAILED DESCRIPTION below in 3-4 sentences as a single "
        "continuous paragraph covering the architecture or method, key components, "
        "and the novel technical mechanism. Omit implementation-specific details. "
        "Do NOT use bullet points, headers, bold text, or markdown formatting. "
        "Do NOT start with phrases like 'Here is' or 'This section'. "
        "Just write the summary directly.\n\nDETAILED DESCRIPTION:\n{text}\n\nSUMMARY:"
    ),
}

def generate_batch(prompts, max_new_tokens=256):
    """Batch greedy generation (left padding, with the chat template)."""
    if len(prompts) == 0:
        return []
    tokenizer.padding_side = "left"
    chats = [tokenizer.apply_chat_template(
                [{"role":"user","content":p}],
                tokenize=False, add_generation_prompt=True) for p in prompts]
    inputs = tokenizer(chats, return_tensors="pt", padding=True,
                       truncation=True, max_length=4096).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[:, inputs["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]

def generate_summary(prompt, max_new_tokens=256):
    return generate_batch([prompt], max_new_tokens)[0]

# Check on the first record (only when the model is loaded)
if model is not None:
    row = m1.iloc[0]
    print(f"Title: {row['title']}\n")
    print("Background facet:")
    print(generate_summary(SECTION_PROMPTS["BACKGROUND"].format(
        text=truncate_text(row["background"], 1500)), 150))
else:
    print("Prompts ready (the facet model is not loaded; see 4.1).")

In [ ]:
# ── 4.3  Facet generation (checkpoint every 50 records) ──────────────────────
# Each section that has at least MIN_LEN characters is cut to its token budget and summarised; a shorter
# section gives an empty facet. The checkpoint must be a prefix of the current input in the same order.

def run_stage1(df_input, gpu_batch=S1_GPU_BATCH, save_every=50):
    results = {"patent_id": [], "bg_summary": [], "inv_summary": [], "tech_summary": []}
    start_idx = 0
    if os.path.exists(CHECKPOINT_S1):
        existing = pd.read_csv(CHECKPOINT_S1).fillna("")
        start_idx = len(existing)
        # A row count alone would accept the checkpoint of another corpus version, so the identifiers are compared in order.
        _prev = existing["patent_id"].astype(str).tolist()
        _curr = df_input["patent_id"].astype(str).tolist()
        if start_idx > len(_curr) or _prev != _curr[:start_idx]:
            raise RuntimeError(
                f"The facet checkpoint does not match the current corpus"
                f" (checkpoint {start_idx} records, input {len(_curr)} records). "
                "SAVE_DIR or REUSE_DIR belongs to another corpus version: use a new SAVE_DIR, "
                "or a REUSE_DIR whose corpus is the same.")
        for col in results:
            results[col] = existing[col].tolist()
        print(f"checkpoint found: {start_idx}/{len(df_input)} done, resuming")
    else:
        print(f"starting from scratch: {len(df_input)} records")

    end_idx = len(df_input)
    if start_idx >= end_idx:
        stage1_df = pd.DataFrame(results)
        if not os.path.exists(STAGE1_OUTPUT):
            stage1_df.to_csv(STAGE1_OUTPUT, index=False)
        print(f"All {end_idx:,} facets are in the checkpoint; nothing to generate.")
        return stage1_df
    print(f"range {start_idx} -> {end_idx} | gpu_batch={gpu_batch}\n")

    def _collect_prompts(rows, section_key, field, in_trunc):
        """Collect the prompts only for the rows whose section length passes MIN_LEN."""
        prompts, valid_idx = [], []
        for i, r in enumerate(rows):
            text = r.get(field, "")
            if len(text) >= MIN_LEN:
                prompts.append(
                    SECTION_PROMPTS[section_key].format(text=truncate_text(text, in_trunc))
                )
                valid_idx.append(i)
        return prompts, valid_idx

    pbar = tqdm(total=end_idx - start_idx, desc="facets")
    for chunk_start in range(start_idx, end_idx, gpu_batch):
        chunk_end = min(chunk_start + gpu_batch, end_idx)
        rows = [df_input.iloc[i] for i in range(chunk_start, chunk_end)]
        n = len(rows)

        # Join the three sections into one large batch for better GPU use
        bg_prompts,   bg_idx   = _collect_prompts(rows, "BACKGROUND",           "background",           1500)
        inv_prompts,  inv_idx  = _collect_prompts(rows, "SUMMARY",              "summary_filled",       1500)
        tech_prompts, tech_idx = _collect_prompts(rows, "DETAILED DESCRIPTION", "detailed_description", 2000)

        all_prompts = bg_prompts + inv_prompts + tech_prompts
        all_gens    = generate_batch(all_prompts, max_new_tokens=200) if all_prompts else []

        b1 = len(bg_prompts); b2 = b1 + len(inv_prompts)
        bg_gens, inv_gens, tech_gens = all_gens[:b1], all_gens[b1:b2], all_gens[b2:]

        bgs, invs, techs = [""]*n, [""]*n, [""]*n
        for k, g in enumerate(bg_gens):   bgs[bg_idx[k]]     = g
        for k, g in enumerate(inv_gens):  invs[inv_idx[k]]   = g
        for k, g in enumerate(tech_gens): techs[tech_idx[k]] = g

        for i, r in enumerate(rows):
            results["patent_id"].append(r["patent_id"])
            results["bg_summary"].append(bgs[i])
            results["inv_summary"].append(invs[i])
            results["tech_summary"].append(techs[i])

        pbar.update(chunk_end - chunk_start)
        processed = chunk_end - start_idx
        if processed % save_every == 0 or chunk_end == end_idx:
            pd.DataFrame(results).to_csv(CHECKPOINT_S1, index=False)
    pbar.close()

    stage1_df = pd.DataFrame(results)
    stage1_df.to_csv(CHECKPOINT_S1, index=False)
    stage1_df.to_csv(STAGE1_OUTPUT, index=False)
    m1.merge(stage1_df, on="patent_id", how="left").to_csv(
        os.path.join(SAVE_DIR, "patent_with_stage1.csv"), index=False)

    print(f"\nFacets complete: {len(stage1_df)} records")
    for col in ["bg_summary", "inv_summary", "tech_summary"]:
        empty = (stage1_df[col] == "").sum()
        pct   = empty / len(stage1_df) * 100
        print(f"  {col:15s}: empty {empty:>5}/{len(stage1_df)} ({pct:.1f}%)")
    print(f"saved -> {STAGE1_OUTPUT}")
    return stage1_df

_done_before = len(pd.read_csv(CHECKPOINT_S1)) if os.path.exists(CHECKPOINT_S1) else 0
with timed(f"4.3  Facet generation ({S1_MODEL_ID.split('/')[-1]})") as t:
    stage1_df = run_stage1(m1, gpu_batch=S1_GPU_BATCH, save_every=50)
    t.n_items = max(len(stage1_df) - _done_before, 0) or None
    t.note    = f"{t.n_items or 0} records generated in this run; batch={S1_GPU_BATCH}"

In [ ]:
# ── 4.4  Reload the facets and check them against the corpus (also for a new session) ──
STAGE1_OUTPUT = os.path.join(SAVE_DIR, "stage1_summaries.csv")
assert os.path.exists(STAGE1_OUTPUT), f"missing {STAGE1_OUTPUT}"
stage1_df = pd.read_csv(STAGE1_OUTPUT).fillna("")
print(f"stage1_df: {len(stage1_df)} records | columns: {list(stage1_df.columns)}")

# Compare the patent_id sets (not only the row counts): a mismatch means that this output belongs to another corpus version
_s1, _m1 = set(stage1_df["patent_id"].astype(str)), set(m1["patent_id"].astype(str))
if _s1 != _m1:
    raise RuntimeError(
        f"stage1_summaries.csv does not match the current corpus: stage1={len(stage1_df)} records, "
        f"m1={len(m1)} records; only in stage1: {len(_s1 - _m1)}, "
        f"only in m1: {len(_m1 - _s1)}. "
        "SAVE_DIR or REUSE_DIR belongs to another corpus version: use a new SAVE_DIR and run 4.3 again.")
print("patent_id alignment check passed")

### 4.5–4.6  分面的量測

檢查空白與過短的分面、30 個字元以上的重複區塊（生成迴圈的跡象），並在隨機抽出的 100 筆上檢查四條規則：輸出夾帶 prompt 的字句、迴圈、少於 30 個詞、markdown 格式。同一批紀錄的 GPT-2 perplexity 描述這些文字對一般語言模型而言有多平常，不是正確性的量測。

In [ ]:
# ── 4.5  Completeness, length and repetition of the facets ───────────────────
S1_COLS = ["bg_summary", "inv_summary", "tech_summary"]
S1_NAMES = {"bg_summary": "background facet", "inv_summary": "invention facet", "tech_summary": "technical facet"}

facet_check = pd.DataFrame([{
    "facet": col,
    "empty": int((stage1_df[col] == "").sum()),
    "empty_percent": round((stage1_df[col] == "").mean() * 100, 1),
    "shorter_than_50_chars": int((stage1_df[col].str.len() < 50).sum()),
    "mean_words": round(stage1_df[col].str.split().str.len().mean(), 1),
    # a repeated block of 30 or more characters is a sign of a generation loop
    "suspected_loops": int(stage1_df[col].str.contains(r"(.{30,})\1", regex=True).sum()),
} for col in S1_COLS])
print(f"records: {len(stage1_df):,}")
facet_check

print("Five facets drawn at random, for reading:")
for _, row in stage1_df.sample(min(5, len(stage1_df)), random_state=RANDOM_STATE).iterrows():
    print(f"\n  {row['patent_id']}")
    print(f"  background : {row['bg_summary'][:140]}...")
    print(f"  invention  : {row['inv_summary'][:140]}...")
    print(f"  technical  : {row['tech_summary'][:140]}...")

# Figure 4.1: length of the three facets
fig, axes = plt.subplots(1, 3, figsize=(9.6, 2.8), sharey=True)
for ax, col in zip(axes, S1_COLS):
    lengths = stage1_df[col].str.split().str.len()
    lengths = lengths[lengths > 0]
    _ = ax.hist(lengths, bins=30, color=PALETTE[0], edgecolor="white", linewidth=0.5)
    _ = ax.axvline(lengths.mean(), color=INK_2, linestyle="--", linewidth=1)
    _ = ax.text(lengths.mean(), ax.get_ylim()[1] * 0.92, f" mean {lengths.mean():.0f}", fontsize=8, color=INK_2)
    _ = ax.set_title(S1_NAMES[col])
    _ = ax.set_xlabel("words")
_ = axes[0].set_ylabel("records")
_ = fig.suptitle("Figure 4.1  Length of the facets (non-empty facets only)", x=0.01, ha="left", fontsize=10)
fig.tight_layout()
_ = save_fig(fig, "fig_4_1_facet_length")
plt.show()

In [ ]:
# ── 4.6  Rule checks and GPT-2 perplexity of the facets (100 records) ──────
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

gpt2_tok   = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").eval().to(DEVICE)

def compute_perplexity(text: str) -> float:
    if not text.strip():
        return float("inf")
    inputs = gpt2_tok(text, return_tensors="pt", truncation=True,
                      max_length=512).to(DEVICE)
    with torch.no_grad():
        loss = gpt2_model(**inputs, labels=inputs["input_ids"]).loss
    return torch.exp(loss).item()

PROMPT_LEAKAGE = ["Just write", "Do NOT", "You are a", "SUMMARY:", "CONDENSED:",
                  "Here is", "This section", "I'm happy to help", "I must point out",
                  "This background assumes that unless explicitly stated",
                  "unless explicitly stated as published prior art"]

def check_rule_violations(text: str) -> dict:
    """Rule-based violation check: prompt leftovers, loops, too short, markdown traces."""
    issues = []
    if any(p.lower() in text.lower() for p in PROMPT_LEAKAGE):
        issues.append("prompt_leak")
    if re.search(r"(.{30,})\1", text):
        issues.append("loop")
    if len(text.split()) < 30:
        issues.append("too_short")
    if re.search(r"[\*\#\-]{2,}|^\s*[\*\-]\s", text, re.MULTILINE):
        issues.append("markdown")
    return {"is_pure": len(issues) == 0, "issues": issues}

QC_N = min(100, len(stage1_df))
qc_sample = stage1_df.sample(QC_N, random_state=RANDOM_STATE)
qc = {col: {"ppls": [], "pure": 0, "issues": {}} for col in S1_COLS}

for _, row in tqdm(qc_sample.iterrows(), total=QC_N, desc="facet checks"):
    for col in S1_COLS:
        text = row[col]
        qc[col]["ppls"].append(compute_perplexity(text))
        pur = check_rule_violations(text)
        if pur["is_pure"]:
            qc[col]["pure"] += 1
        for issue in pur["issues"]:
            qc[col]["issues"][issue] = qc[col]["issues"].get(issue, 0) + 1

print(f"\n{'='*64}")
print(f"Facet measurements (n={QC_N} records drawn at random)")
print(f"{'='*64}")
print(f"{'facet':<16} {'rule_pass':>8} {'PPL mean':>10} {'PPL med':>9} {'PPL p90':>9}")
for col in S1_COLS:
    ppls = [p for p in qc[col]["ppls"] if p != float("inf")]
    rule_pass_pct = qc[col]["pure"] / QC_N * 100
    print(f"{col:<16} {rule_pass_pct:>7.1f}% {np.mean(ppls):>10.1f} "
          f"{np.median(ppls):>9.1f} {np.percentile(ppls, 90):>9.1f}")
    for issue, cnt in qc[col]["issues"].items():
        print(f"    - {issue}: {cnt}")
print("\nreference values used during development: rule_pass >= 90% | PPL mean < 80 | PPL p90 < 150")

free_gpu("gpt2_model", "gpt2_tok")

In [ ]:
# ── 4.7  Release the facet model before section 5 ─────────────────────────
free_gpu("model", "tokenizer")

### 第 4 節結果（釋出版本）

6,535 筆全部都有 invention 分面與 technical 分面；12 筆的 background 分面是空的，因為它們的 background 段少於 100 個字元。沒有任何分面含重複區塊。隨機抽出的 100 筆中，background、invention、technical 三種分面分別有 91%、97%、100% 通過全部四條規則，違規的全是分面裡重複了 prompt 的字句。三種分面的平均 GPT-2 perplexity 分別是 33.9、41.4、33.5。釋出的分面分兩個工作階段生成：前 4,000 筆來自較早的一次執行，其餘 2,535 筆在續跑時生成；兩部分的分面長度非常接近（Kolmogorov-Smirnov 距離最大 0.031，見釋出資料的 `provenance/checks/facet_sessions.csv`）。

## 5  三個摘要層：synthesized、direct-brief 與 direct-full

三個摘要層並列釋出。它們最後一次生成呼叫的設定相同：`Qwen/Qwen2.5-14B-Instruct`（bf16）、貪婪解碼、`repetition_penalty=1.15`、最多 300 個新 token、目標五到七句。差別在於模型讀了什麼，以及最後一次呼叫之前做了哪些步驟。

| 資料層 | 輸入 | 步驟 | checkpoint |
|---|---|---|---|
| synthesized | title、abstract 與三個分面（分面預算：原文約 5,000 個 Llama token） | 先做一次推理呼叫（問題、解法、機制、優點；最多 350 個 token），再做合成呼叫；兩次呼叫都列出最多十個要保留的詞 | `stage2_checkpoint.csv` |
| direct-brief | title、abstract、background 800、summary 400、detailed description 800 個 token | 單一 prompt | `baseline_direct.csv` |
| direct-full | title、abstract 與分面預算（1,500、1,500、2,000 個 token） | 與 direct-brief 相同的 prompt | `baseline_direct_matched.csv` |

direct-full 與 direct-brief 只差在讀取深度；synthesized 與 direct-full 只差在處理流程。要保留的詞（`must_keep`，只在 5.3 定義一次）取自領域白名單、標題中有資訊量的詞與較長的技術詞，每筆最多十個。生成函式不會截斷 prompt：超過 8,192 個 token 的 prompt 會直接讓執行停下（語料中最長的 prompt 是 5,304 個 token）。

In [ ]:
# ── 5.1  Synthesis model (the model is loaded only when summaries are still missing) ──
# The tokenizer is needed in every run: the token budgets of 5.10, 5.11, 8.1 and 8.6 are counted with it.
from transformers import AutoModelForCausalLM, AutoTokenizer

CHECKPOINT_S2 = os.path.join(SAVE_DIR, "stage2_checkpoint.csv")
FULL_OUTPUT   = os.path.join(SAVE_DIR, "patent_full_pipeline.csv")
_done_s2 = (len(pd.read_csv(CHECKPOINT_S2, usecols=["patent_id"])) if os.path.exists(CHECKPOINT_S2) else 0)
S2_TODO  = max(len(stage1_df) - _done_s2, 0)

tokenizer_s2 = AutoTokenizer.from_pretrained(S2_MODEL_NAME, trust_remote_code=True, padding_side="left")
if tokenizer_s2.pad_token is None:
    tokenizer_s2.pad_token = tokenizer_s2.eos_token

if S2_TODO and "model_s2" not in globals():
    if not (HAS_CUDA or SMALL_MODELS):
        raise RuntimeError(f"{S2_TODO:,} synthesized summaries are missing and this machine has no CUDA GPU.")
    model_s2 = AutoModelForCausalLM.from_pretrained(
        S2_MODEL_NAME, torch_dtype=GEN_DTYPE, trust_remote_code=True,
        device_map="auto" if HAS_CUDA else None)
    if not HAS_CUDA:
        model_s2 = model_s2.to(DEVICE)
    # On a GPU with less than 32 GB, load in 4-bit instead:
    #   from transformers import BitsAndBytesConfig
    #   bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    #                            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    #   model_s2 = AutoModelForCausalLM.from_pretrained(S2_MODEL_NAME, quantization_config=bnb, device_map="auto")
    model_s2 = model_s2.eval()
    print(f"{S2_MODEL_NAME} loaded | {S2_TODO:,} of {len(stage1_df):,} records still need a synthesized summary")
    if HAS_CUDA:
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU memory: allocated {torch.cuda.memory_allocated()/1e9:.1f} GB | "
              f"free {total - torch.cuda.memory_reserved()/1e9:.1f} GB")
else:
    print(f"Tokenizer of {S2_MODEL_NAME} loaded; the model is not loaded "
          f"({'already in memory' if 'model_s2' in globals() else 'all synthesized summaries are in the checkpoint'}).")

In [ ]:
# ── 5.2  Generation helpers of the synthesis model ───────────────────────────
# The prompt is never cut silently: a prompt longer than MAX_INPUT_TOKENS_S2 stops the run with an error.
def truncate_text_s2(text: str, max_tokens: int = 1500) -> str:
    toks = tokenizer_s2.encode(str(text), add_special_tokens=False)
    if len(toks) > max_tokens:
        text = tokenizer_s2.decode(toks[:max_tokens], skip_special_tokens=True)
    return text

MAX_INPUT_TOKENS_S2 = 8192   # the longest prompt of the full corpus has 5,304 tokens with the chat format; model limit 32,768
def generate_batch_s2(prompts, max_new_tokens=256):
    if not prompts:
        return []
    tokenizer_s2.padding_side = "left"
    if tokenizer_s2.pad_token is None:
        tokenizer_s2.pad_token = tokenizer_s2.eos_token
    chats = [tokenizer_s2.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True) for p in prompts]
    _n_in = [len(x) for x in tokenizer_s2(chats)["input_ids"]]
    if max(_n_in) > MAX_INPUT_TOKENS_S2:   # no silent cut: stop when the input exceeds the limit
        raise ValueError(f"prompt of {max(_n_in)} tokens exceeds MAX_INPUT_TOKENS_S2 = {MAX_INPUT_TOKENS_S2}")
    inputs = tokenizer_s2(chats, return_tensors="pt", padding=True, truncation=False).to(model_s2.device)
    with torch.no_grad():
        out = model_s2.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer_s2.eos_token_id,
        )
    gen = out[:, inputs["input_ids"].shape[1]:]
    return [tokenizer_s2.decode(g, skip_special_tokens=True).strip() for g in gen]

print("generate_batch_s2 ready")

In [ ]:
# ── 5.3  Terms to keep (must_keep): the only definition of the extractor ─────
# The whitelist and the extractor are defined only here; 5.6 and the checks of section 8 use them.
HPC_WHITELIST = {
    # architecture
    "parallel","distributed","concurrent","pipelined","multiprocessor",
    "multi-core","many-core","heterogeneous","accelerator","coprocessor",
    "superscalar","out-of-order","speculative","vectorized",
    # memory
    "cache","bandwidth","latency","throughput","prefetch",
    "coherence","scratchpad","hierarchy","non-volatile","volatile",
    "high-bandwidth","near-memory","shared-memory","distributed-memory",
    "dram","sram","hbm","nvm","nvram","flash","persistent",
    # interconnect
    "interconnect","topology","routing","infiniband","fabric",
    "switched","mesh","torus","hypercube","crossbar","bisection",
    # computing models
    "sparse","stencil","convolution","scheduler","workload",
    "dataflow","pipeline","streaming","vectorization","parallelism",
    "simd","mimd","spmd","openmp","cuda","opencl","mpi",
    # semiconductors
    "transistor","semiconductor","photonic","cmos","wafer",
    "monolithic","chiplet","interposer","substrate",
    # networks
    "network","multicast","broadcast","unicast","packet","switch",
    # storage
    "storage","filesystem","checkpoint","journal","snapshot",
    "erasure","replication","deduplication","compression",
    # algorithms
    "gradient","optimization","quantization","pruning","sparsity",
    "pooling","normalization","backpropagation",
    # mathematics/geometry
    "manifold","diffeomorphic","topological","differentiable",
    "euclidean","riemannian","voxel","quaternion","tensor",
    # security
    "encryption","decryption","authentication","cryptographic",
    # HPC core
    "scalable","bottleneck","multi-level","high-performance",
    "fault-tolerant","load-balancing","real-time","multi-threaded",
    "multi-node","multi-gpu","multi-chip","many-to-many",
    # memory types
    "memory","buffer","register",
    # other
    "high","multi",
}

GENERIC_SUFFIXES = re.compile(
    r'(ing|tions|sion|tion|ments|ment|ness|ities|ity|ally|arly|onal|ical|ival|inal|ive|ous|ful|less|ary|ery|ory|ular|ated|ence|ance|ward|wise|ently|ingly|able|ible|ship|hood|ture|work|tics|ors|ers|ists|isms)$',
    re.I
)

TITLE_STOPWORDS = {
    "method","system","apparatus","device","process",
    "technique","approach","based","using","without","within",
    "through","between","among","their","these","those","which",
    "that","with","from","into","onto","upon","over","under",
    "about","after","before","during","while","when","where",
    "assessment","selection","image","media","level","type",
    "large","small","general","specific","various","certain",
    "given","known","existing","current","conventional",
    "proposed","present","consequently","therefore","however",
    "moreover","furthermore","meshes","methods","systems",
    "devices","approaches","results","issues","challenges",
    "problems","solutions","aspects","generation","serial",
    "digital","virtual","logical","physical","efficient",
}

def extract_tech_terms(title, *texts):
    """Return at most 10 must-keep technical terms:
    priority order: whitelist hits > informative title words > long words (>= 12 characters)."""
    combined = " ".join(texts)
    whitelist_hits, seen_wl = [], set()
    for w in re.findall(r'\b[a-zA-Z][a-zA-Z-]{2,}\b', title + " " + combined):
        lw = w.lower()
        if lw in HPC_WHITELIST and lw not in seen_wl:
            seen_wl.add(lw); whitelist_hits.append(lw)
    title_words, seen_tw = [], set()
    for w in re.findall(r'\b[a-zA-Z][a-zA-Z-]{4,}\b', title):
        lw = w.lower()
        if (lw not in TITLE_STOPWORDS and not GENERIC_SUFFIXES.search(lw)
                and lw not in seen_wl and lw not in seen_tw):
            seen_tw.add(lw); title_words.append(lw)
    long_words, seen_lw = [], set()
    for w in re.findall(r'\b[a-zA-Z][a-zA-Z-]{11,}\b', combined):
        lw = w.lower()
        if (lw not in TITLE_STOPWORDS and not GENERIC_SUFFIXES.search(lw)
                and lw not in seen_wl and lw not in seen_tw and lw not in seen_lw):
            seen_lw.add(lw); long_words.append(lw)
    return ", ".join((whitelist_hits + title_words + long_words)[:10])

print("extract_tech_terms ready |", len(HPC_WHITELIST), "whitelist terms")

In [ ]:
# ── 5.4  Prompt templates of the synthesized layer ──────────────────────────
# The prompts are part of the method and are released as they are (9.1).
COT_REASONING_PROMPT = (
    "You are an expert patent analyst specializing in computing and "
    "engineering technologies.\n\n"
    "STRICT RULES:\n"
    "- Use ONLY information explicitly stated in the Title, Abstract, "
    "and Section Summaries below.\n"
    "- Do NOT infer, assume, or add any applications, use cases, or "
    "benefits not directly mentioned in the input.\n"
    "- You MUST include key terms from the Title verbatim in your output.\n"
    "- If the Background Summary merely describes existing technology "
    "without stating a problem, derive the problem from the Abstract "
    "or Invention Summary instead.\n\n"
    "Given the following section summaries of a patent, output EXACTLY "
    "4 steps:\n\n"
    "Step 1 – The core technical PROBLEM or LIMITATION (1 sentence). "
    "Focus on what is broken, missing, or insufficient — "
    "NOT a description of existing technology.\n"
    "Step 2 – The proposed solution or key invention (1 sentence).\n"
    "Step 3 – The 2-3 most technically significant mechanisms. "
    "Write full descriptive sentences, do NOT just list keywords.\n"
    "Step 4 – The primary technical advantage (1 sentence, "
    "no inferred applications).\n\n"
    "IMPORTANT: Your output must naturally incorporate these technical "
    "terms where relevant: {must_keep}.\n\n"
    "Title: {title}\n"
    "Abstract: {abstract}\n"
    "Background Summary: {bg_summary}\n"
    "Invention Summary: {inv_summary}\n"
    "Technical Details Summary: {tech_summary}\n\n"
    "REASONING:"
)

FINAL_SYNTHESIS_PROMPT_V2 = (
    "You are a scientific writer producing a patent summary "
    "for a research journal audience.\n\n"
    "STRICT RULES:\n"
    "- Summarize ONLY what is explicitly stated in the inputs below.\n"
    "- Do NOT add applications, industries, or use cases not mentioned.\n"
    "- You MUST preserve these exact technical terms verbatim in your output: "
    "{must_keep}.\n"
    "- Every term listed above MUST appear at least once in your paragraph.\n"
    "\n"
    "Using the structured reasoning below, produce a single coherent "
    "paragraph of 5-7 sentences covering: the technical problem, "
    "proposed solution, key mechanisms, and technical advantages.\n\n"
    "Do NOT use bullet points, headers, bold, or markdown. "
    "Do NOT start with 'Here is', 'This patent', or 'This summary'.\n\n"
    "Title: {title}\n"
    "Abstract: {abstract}\n"
    "Structured Reasoning:\n{cot_reasoning}\n\n"
    "FINAL SUMMARY:"
)

# Optional third refinement pass (off by default, use_compression=False)
COMPRESSION_PROMPT = (
    "You are a technical editor. Refine the following patent summary to "
    "be concise and fluent. The refined summary must:\n"
    "- Be exactly 4-5 sentences\n"
    "- Preserve ALL key technical terms from the original\n"
    "- Remove redundancy and improve sentence transitions\n"
    "- Contain NO bullet points, headers, or markdown formatting\n\n"
    "ORIGINAL SUMMARY:\n{raw_summary}\n\n"
    "REFINED SUMMARY:"
)

print("Prompt templates ready")

In [ ]:
# ── 5.5  Check on one record: reasoning call and synthesis call (only when the model is loaded) ──
if "model_s2" in globals():
    test_row = stage1_df.iloc[0]
    meta_row = m1[m1["patent_id"] == test_row["patent_id"]].iloc[0]
    must_keep = extract_tech_terms(
        meta_row["title"],
        test_row["bg_summary"], test_row["inv_summary"], test_row["tech_summary"])
    print(f"patent_id : {test_row['patent_id']}")
    print(f"title     : {meta_row['title']}")
    print(f"must_keep : {must_keep}\n")

    cot_out = generate_batch_s2([COT_REASONING_PROMPT.format(
        title=meta_row["title"],
        abstract=truncate_text_s2(meta_row["abstract"], 300),
        bg_summary=test_row["bg_summary"],
        inv_summary=test_row["inv_summary"],
        tech_summary=test_row["tech_summary"],
        must_keep=must_keep)], max_new_tokens=350)[0]
    print("Reasoning:\n" + cot_out)

    final_out = generate_batch_s2([FINAL_SYNTHESIS_PROMPT_V2.format(
        title=meta_row["title"],
        abstract=truncate_text_s2(meta_row["abstract"], 300),
        cot_reasoning=cot_out,
        must_keep=must_keep)], max_new_tokens=300)[0]
    print("\nSynthesized summary:\n" + final_out)

    _terms = [t.strip() for t in must_keep.split(",") if t.strip()]
    _kept_terms = [t for t in _terms if t.lower() in final_out.lower()]
    print(f"\nmust_keep terms in the summary: {len(_kept_terms)}/{len(_terms)}")
else:
    print("The synthesis model is not loaded (all synthesized summaries exist); the check on one record is skipped.")

In [ ]:
# ── 5.6  Synthesized summaries: reasoning call, then synthesis call (checkpoint every 50) ──
# The output file patent_full_pipeline.csv joins m1, the facets and the two generated texts.

def run_stage2(stage1_df, meta_df, gpu_batch=S2_GPU_BATCH, save_every=50,
               use_compression=False):
    df = stage1_df.merge(
        meta_df[["patent_id","title","abstract"]],
        on="patent_id", how="left").fillna("")

    results = {"patent_id":[],"cot_reasoning":[],"raw_summary":[],"final_summary":[]}
    start_idx = 0
    if os.path.exists(CHECKPOINT_S2):
        existing  = pd.read_csv(CHECKPOINT_S2).fillna("")
        start_idx = len(existing)
        # The checkpoint must be a prefix of the same input, in the same order (as in 4.3)
        _prev = existing["patent_id"].astype(str).tolist()
        _curr = df["patent_id"].astype(str).tolist()
        if start_idx > len(_curr) or _prev != _curr[:start_idx]:
            raise RuntimeError(
                f"The synthesis checkpoint does not match the current corpus"
                f" (checkpoint {start_idx} records, input {len(_curr)} records). "
                "SAVE_DIR or REUSE_DIR belongs to another corpus version: use a new SAVE_DIR, "
                "or a REUSE_DIR whose corpus is the same.")
        for col in results:
            results[col] = existing[col].tolist()
        print(f"checkpoint found: {start_idx}/{len(df)} done, resuming")
    else:
        print(f"starting from scratch: {len(df)} records")

    if start_idx >= len(df):
        stage2_df = pd.DataFrame(results)
        if not os.path.exists(FULL_OUTPUT):
            meta_df \
                .merge(stage1_df, on="patent_id", how="left") \
                .merge(stage2_df[["patent_id","cot_reasoning","raw_summary","final_summary"]],
                       on="patent_id", how="left") \
                .to_csv(FULL_OUTPUT, index=False)
        print(f"All {len(df):,} synthesized summaries are in the checkpoint; nothing to generate.")
        return stage2_df
    print(f"range {start_idx} -> {len(df)} | batch={gpu_batch} | "
          f"compression={'ON' if use_compression else 'OFF'}\n")

    pbar = tqdm(total=len(df) - start_idx, desc="synthesis")
    for chunk_start in range(start_idx, len(df), gpu_batch):
        chunk_end = min(chunk_start + gpu_batch, len(df))
        rows = [df.iloc[i] for i in range(chunk_start, chunk_end)]

        must_keeps = [
            extract_tech_terms(r["title"], r["bg_summary"],
                               r["inv_summary"], r["tech_summary"])
            for r in rows]

        cot_outs = generate_batch_s2([
            COT_REASONING_PROMPT.format(
                title=r["title"],
                abstract=truncate_text_s2(r["abstract"], 300),
                bg_summary=r["bg_summary"],
                inv_summary=r["inv_summary"],
                tech_summary=r["tech_summary"],
                must_keep=must_keeps[i],
            ) for i, r in enumerate(rows)], max_new_tokens=350)

        raw_outs = generate_batch_s2([
            FINAL_SYNTHESIS_PROMPT_V2.format(
                title=rows[i]["title"],
                abstract=truncate_text_s2(rows[i]["abstract"], 300),
                cot_reasoning=cot_outs[i],
                must_keep=must_keeps[i],
            ) for i in range(len(rows))], max_new_tokens=300)

        final_outs = generate_batch_s2([
            COMPRESSION_PROMPT.format(raw_summary=raw_outs[i])
            for i in range(len(rows))], max_new_tokens=200) if use_compression else raw_outs

        for i, r in enumerate(rows):
            results["patent_id"].append(r["patent_id"])
            results["cot_reasoning"].append(cot_outs[i])
            results["raw_summary"].append(raw_outs[i])
            results["final_summary"].append(final_outs[i])

        pbar.update(chunk_end - chunk_start)
        processed = chunk_end - start_idx
        if processed % save_every == 0 or chunk_end == len(df):
            pd.DataFrame(results).to_csv(CHECKPOINT_S2, index=False)
    pbar.close()

    stage2_df = pd.DataFrame(results)
    stage2_df.to_csv(CHECKPOINT_S2, index=False)

    meta_df \
        .merge(stage1_df, on="patent_id", how="left") \
        .merge(stage2_df[["patent_id","cot_reasoning","raw_summary","final_summary"]],
               on="patent_id", how="left") \
        .to_csv(FULL_OUTPUT, index=False)

    print(f"\nSynthesized summaries complete: {len(stage2_df)} records")
    for col in ["cot_reasoning", "raw_summary", "final_summary"]:
        empty = (stage2_df[col] == "").sum()
        print(f"  {col:20s}: empty {empty:>5}/{len(stage2_df)} ({empty/len(stage2_df)*100:.1f}%)")
    print(f"saved -> {FULL_OUTPUT}")
    return stage2_df

_done_before = len(pd.read_csv(CHECKPOINT_S2)) if os.path.exists(CHECKPOINT_S2) else 0
with timed(f"5.6  Synthesized summary generation ({S2_MODEL_NAME.split('/')[-1]})") as t:
    stage2_df = run_stage2(stage1_df, m1, gpu_batch=S2_GPU_BATCH, save_every=50)
    t.n_items = max(len(stage2_df) - _done_before, 0) or None
    t.note    = f"{t.n_items or 0} records generated in this run; batch={S2_GPU_BATCH}"

In [ ]:
# ── 5.7  Reload the synthesized layer and check it against the corpus ────────
FULL_OUTPUT = os.path.join(SAVE_DIR, "patent_full_pipeline.csv")
pipeline_df = pd.read_csv(FULL_OUTPUT).fillna("")
print(f"pipeline_df: {len(pipeline_df)} records")
for col in ["patent_id", "title", "abstract", "final_summary"]:
    print(f"  {col:15s} empty: {(pipeline_df[col]=='').sum()}")

# The identifiers must be those of the current corpus (see 4.4)
if "m1" in globals():
    _p, _m = set(pipeline_df["patent_id"].astype(str)), set(m1["patent_id"].astype(str))
    if _p != _m:
        raise RuntimeError(
            f"patent_full_pipeline.csv does not match the current corpus ({len(_p)} vs {len(_m)} records). "
            "SAVE_DIR belongs to another corpus version. Use a new SAVE_DIR.")
    print("patent_id alignment check passed")

### 5.8–5.9  synthesized 層的量測

隨機抽 20 筆，報告空白與過短的輸出、長度，以及 `must_keep` 詞出現在摘要中的比例。隨機抽 100 筆，報告 GPT-2 perplexity，以及每篇摘要和它所依據的三個分面之間的餘弦相似度（all-MiniLM-L6-v2）。

In [ ]:
# ── 5.8  Completeness, length and term retention of the synthesized layer (20 records) ──
# (a) completeness and length
test_stage2 = pipeline_df.sample(min(20, len(pipeline_df)), random_state=RANDOM_STATE).reset_index(drop=True)
for col in ["cot_reasoning", "raw_summary", "final_summary"]:
    empty = (test_stage2[col] == "").sum()
    short = (test_stage2[col].str.split().str.len() < 30).sum()
    print(f"  {col:20s}: empty {empty} | short(<30 words) {short}")
for col in ["cot_reasoning", "final_summary"]:
    lengths = test_stage2[col].str.split().str.len()
    print(f"  {col:20s}: mean={lengths.mean():.0f} | min={lengths.min()} | max={lengths.max()} words")

# (b) must_keep retention on the same records
hit, total_kw = 0, 0
for _, row in test_stage2.iterrows():
    s1_row = stage1_df[stage1_df["patent_id"] == row["patent_id"]]
    mrow   = m1[m1["patent_id"] == row["patent_id"]]
    if len(s1_row) == 0 or len(mrow) == 0:
        continue
    s1 = s1_row.iloc[0]
    mk = extract_tech_terms(mrow.iloc[0]["title"],
                            s1["bg_summary"], s1["inv_summary"], s1["tech_summary"])
    terms = [t.strip().lower() for t in mk.split(",") if t.strip()]
    hit      += sum(1 for t in terms if t in row["final_summary"].lower())
    total_kw += len(terms)
print(f"\nmust_keep retention ({len(test_stage2)} records): {hit}/{total_kw} ({hit/max(total_kw, 1)*100:.1f}%)")

In [ ]:
# ── 5.9  Perplexity and similarity to the source facets (100 records) ───────
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

gpt2_tok   = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").eval().to(DEVICE)

def compute_perplexity(text: str) -> float:
    if not text.strip():
        return float("inf")
    inputs = gpt2_tok(text, return_tensors="pt", truncation=True,
                      max_length=512).to(DEVICE)
    with torch.no_grad():
        loss = gpt2_model(**inputs, labels=inputs["input_ids"]).loss
    return torch.exp(loss).item()

sbert = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

def cosine_to_facets(summary: str, reference: str) -> float:
    """Cosine similarity between the synthesized summary and the three facets that it was made from."""
    if not summary.strip() or not reference.strip():
        return 0.0
    embs = sbert.encode([summary, reference])
    return float(cosine_similarity([embs[0]], [embs[1]])[0][0])

QC_N = min(100, len(pipeline_df))
qc_sample = pipeline_df.sample(QC_N, random_state=RANDOM_STATE)

ppl_list, cos_to_facets_list = [], []
for _, row in tqdm(qc_sample.iterrows(), total=QC_N, desc="synthesis checks"):
    ppl_list.append(compute_perplexity(row["final_summary"]))
    reference = " ".join([row.get("bg_summary",""), row.get("inv_summary",""),
                          row.get("tech_summary","")])
    cos_to_facets_list.append(cosine_to_facets(row["final_summary"], reference))

ppl_arr    = np.array([p for p in ppl_list if p != float("inf")])
cos_to_facets_arr = np.array(cos_to_facets_list)

print(f"\n{'='*56}")
print(f"Synthesized layer, measurements (n={QC_N} records drawn at random)")
print(f"{'='*56}")
print(f"Perplexity : mean {ppl_arr.mean():.2f} | median {np.median(ppl_arr):.2f} | "
      f"p90 {np.percentile(ppl_arr, 90):.2f}")
print(f"Cosine to facets : mean {cos_to_facets_arr.mean():.4f} | median {np.median(cos_to_facets_arr):.4f} | "
      f"p10 {np.percentile(cos_to_facets_arr, 10):.4f}")
print("reference values used during development: PPL mean < 80 | PPL p90 < 150 | cos_to_facets mean > 0.75")

low_cos_to_facets = [(qc_sample.iloc[i]["patent_id"], cos_to_facets_list[i])
              for i in range(QC_N) if cos_to_facets_list[i] < 0.60]
print(f"\nrecords with cos_to_facets < 0.60: {len(low_cos_to_facets)}")
for pid, pur in low_cos_to_facets[:5]:
    print(f"  {pid}: {pur:.4f}")

free_gpu("gpt2_model", "gpt2_tok", "sbert")

### 5.10–5.11  兩個 direct 資料層

兩層用同一個 prompt，只差在 token 預算。prompt 與預算都會在 9.1 寫進釋出資產。每層各有自己的 checkpoint，只有在還有紀錄未生成時才載入模型。

In [ ]:
# ── 5.10  direct-brief layer: one prompt over a short reading of the sections ──
# The same model reads the main fields directly with one common prompt. The target length is the same as for
# the synthesized layer (5-7 sentences), so that length-sensitive measures compare layers of similar length.
# Budgets (tokens of the synthesis model): background 800, summary 400, detailed description 800.
# The prompt is part of the method and is released as it is (9.1). Checkpoint: baseline_direct.csv.
DIRECT_PROMPT = (
    "You are a patent analyst specializing in High-Performance Computing (HPC).\n\n"
    "Read the following patent text and write a concise technical summary "
    "of 5-7 sentences as a single paragraph.\n"
    "Cover: the core technical problem, proposed solution, and key mechanisms.\n"
    "Do NOT use bullet points, headers, or markdown.\n\n"
    "Title: {title}\n"
    "Abstract: {abstract}\n\n"
    "Background:\n{background}\n\n"
    "Summary:\n{summary}\n\n"
    "Detailed Description:\n{detail}\n\n"
    "SUMMARY:"
)

DIRECT_PATH = os.path.join(SAVE_DIR, "baseline_direct.csv")
_SUM_COL = "summary_filled" if "summary_filled" in pipeline_df.columns else "summary"
_BATCH = BASE_BATCH

def _direct_brief_prompt(row):
    return DIRECT_PROMPT.format(
        title      = str(row["title"]),
        abstract   = str(row["abstract"]),
        background = truncate_text_s2(row["background"],           800),
        summary    = truncate_text_s2(row[_SUM_COL],               400),
        detail     = truncate_text_s2(row["detailed_description"], 800),
    )

pids = pipeline_df["patent_id"].tolist()
if os.path.exists(DIRECT_PATH):
    done_df  = pd.read_csv(DIRECT_PATH).fillna("")
    done_ids = set(done_df["patent_id"])
    remain   = [i for i, p in enumerate(pids) if p not in done_ids]
    results  = done_df.to_dict("records")
    print(f"checkpoint: {len(done_ids)} done, {len(remain)} remaining")
else:
    remain, results = list(range(len(pids))), []
# prompts are built only for the records that still need a summary (tokenising long sections takes minutes)
prompts = {i: _direct_brief_prompt(pipeline_df.iloc[i]) for i in tqdm(remain, desc="build prompts")}

if remain and "model_s2" not in globals():
    model_s2 = AutoModelForCausalLM.from_pretrained(
        S2_MODEL_NAME, torch_dtype=GEN_DTYPE, trust_remote_code=True,
        device_map="auto" if HAS_CUDA else None)
    if not HAS_CUDA:
        model_s2 = model_s2.to(DEVICE)
    model_s2 = model_s2.eval()

with timed(f"5.10  direct-brief generation ({S2_MODEL_NAME.split('/')[-1]}, single prompt)",
           n_items=len(remain) or None) as t:
    t.note = f"{len(remain)} records generated in this run; batch={_BATCH}"
    pbar = tqdm(range(0, len(remain), _BATCH), desc="direct-brief")
    for b in pbar:
        chunk = remain[b : b + _BATCH]
        outs = generate_batch_s2([prompts[i] for i in chunk], max_new_tokens=300)
        for i, out in zip(chunk, outs):
            results.append({"patent_id": pids[i], "baseline_summary": out})
        pbar.set_description(f"direct-brief ({len(results)}/{len(pids)})")
        if len(results) % 100 < _BATCH:
            pd.DataFrame(results).to_csv(DIRECT_PATH, index=False)
pd.DataFrame(results).to_csv(DIRECT_PATH, index=False)
_wl = pd.DataFrame(results)["baseline_summary"].astype(str).str.split().str.len()
print(f"saved -> {DIRECT_PATH} ({len(results):,} records) | mean {_wl.mean():.0f} words | empty {(_wl == 0).sum()}")

In [ ]:
# ── 5.11  direct-full layer: the same prompt with the input budget of the facets ──
# direct-brief reads about 2,000 tokens, while the facet step reads about 5,000. This layer gives the single
# prompt the facet budgets section by section (see _collect_prompts in 4.3) and changes nothing else: the same
# model, precision, greedy decoding, repetition penalty, output limit, target length and prompt template.
# The three layers then form a simple factor structure:
#   direct-brief  short reading, single prompt   direct-full  full reading, single prompt
#   synthesized   full reading, facets and synthesis
#   direct-full - direct-brief = reading-depth difference; synthesized - direct-full = process difference
# The generation helper of 5.2 never cuts a prompt (the longest prompt of the full corpus has 5,304 tokens).
# Checkpoint: baseline_direct_matched.csv. About 5-6 hours for the full corpus on one A100 80 GB.
MATCHED_BUDGETS = {"background": 1500, "summary": 1500, "detail": 2000}
MATCHED_PATH = os.path.join(SAVE_DIR, "baseline_direct_matched.csv")
_SUM_COL = "summary_filled" if "summary_filled" in pipeline_df.columns else "summary"
_BATCH   = MATCHED_BATCH   # the prompts are longer; reduce it when the GPU memory is not sufficient

def _direct_full_prompt(row):
    return DIRECT_PROMPT.format(
        title      = str(row["title"]),
        abstract   = str(row["abstract"]),
        background = truncate_text_s2(row["background"],           MATCHED_BUDGETS["background"]),
        summary    = truncate_text_s2(row[_SUM_COL],               MATCHED_BUDGETS["summary"]),
        detail     = truncate_text_s2(row["detailed_description"], MATCHED_BUDGETS["detail"]),
    )

pids = pipeline_df["patent_id"].tolist()
_first = [_direct_full_prompt(pipeline_df.iloc[i]) for i in range(min(200, len(pipeline_df)))]
_ntok = [len(tokenizer_s2.encode(p, add_special_tokens=False)) for p in _first]
DIRECT_FULL_PROMPT_MEDIAN = int(np.median(_ntok))      # written to release_assets/config/generation.json (9.1)
print(f"Prompt length with the facet budgets (first {len(_first)} records): median {DIRECT_FULL_PROMPT_MEDIAN} tokens"
      f" | 90th percentile {int(np.percentile(_ntok, 90))} | max {int(max(_ntok))}")

if os.path.exists(MATCHED_PATH):
    done_df  = pd.read_csv(MATCHED_PATH).fillna("")
    done_ids = set(done_df["patent_id"])
    remain   = [i for i, p in enumerate(pids) if p not in done_ids]
    results  = done_df.to_dict("records")
    print(f"checkpoint: {len(done_ids)} done, {len(remain)} remaining")
else:
    remain, results = list(range(len(pids))), []
# prompts are built only for the records that still need a summary (tokenising long sections takes minutes)
prompts = {i: (_first[i] if i < len(_first) else _direct_full_prompt(pipeline_df.iloc[i]))
           for i in tqdm(remain, desc="build prompts (matched)")}

if remain and "model_s2" not in globals():
    model_s2 = AutoModelForCausalLM.from_pretrained(
        S2_MODEL_NAME, torch_dtype=GEN_DTYPE, trust_remote_code=True,
        device_map="auto" if HAS_CUDA else None)
    if not HAS_CUDA:
        model_s2 = model_s2.to(DEVICE)
    model_s2 = model_s2.eval()

with timed(f"5.11  direct-full generation ({S2_MODEL_NAME.split('/')[-1]}, facet budgets)",
           n_items=len(remain) or None) as t:
    t.note = f"{len(remain)} records generated in this run; batch={_BATCH}; budgets={MATCHED_BUDGETS}"
    pbar = tqdm(range(0, len(remain), _BATCH), desc="direct-full")
    for b in pbar:
        chunk = remain[b : b + _BATCH]
        outs = generate_batch_s2([prompts[i] for i in chunk], max_new_tokens=300)
        for i, out in zip(chunk, outs):
            results.append({"patent_id": pids[i], "baseline_matched_summary": out})
        pbar.set_description(f"direct-full ({len(results)}/{len(pids)})")
        if len(results) % 100 < _BATCH:
            pd.DataFrame(results).to_csv(MATCHED_PATH, index=False)
pd.DataFrame(results).to_csv(MATCHED_PATH, index=False)
_wl = pd.DataFrame(results)["baseline_matched_summary"].astype(str).str.split().str.len()
print(f"saved -> {MATCHED_PATH} ({len(results):,} records) | mean {_wl.mean():.0f} words"
      f" | median {_wl.median():.0f} | empty {(_wl == 0).sum()}")

In [ ]:
# ── 5.12  Release the synthesis model before section 6 ─────────────────────
free_gpu("model_s2", "tokenizer_s2")

### 第 5 節結果（釋出版本）

synthesized 摘要與推理文字都沒有空白。抽出的 20 筆中，synthesized 摘要平均 165 個詞（117 到 219），`must_keep` 詞有 84.5% 出現在摘要裡。抽出的 100 筆中，平均 perplexity 是 40.8，與來源分面的平均餘弦相似度是 0.79；有 3 筆低於 0.60，列在人工稽核的邊界案例中（8.8）。在分面預算下，direct-full 的 prompt 在前 200 筆的中位數是 2,936 個 token，最長 5,217。direct-brief 與 direct-full 摘要平均 132 與 133 個詞。

第一次執行時，生成函式會從右側把 prompt 截在 4,096 個 token，因此有 536 筆 direct-full 的 prompt 失去了結尾的指示，模型接著續寫說明書，而不是寫摘要。現在的生成函式不再截斷 prompt，這 536 筆已於 2026-10-11 重新生成；其餘 5,999 筆未變。

## 6  向量、分群與主題

SPECTER2（`allenai/specter2_base` 加 proximity adapter）把每筆紀錄的 title 與 synthesized 摘要編碼成 768 維向量（6.2）。UMAP 把向量降到三維（cosine 距離、`n_neighbors=15`、`min_dist=0.0`、種子 42），再以 HDBSCAN 分群（`min_cluster_size=30`、`min_samples=10`；6.4）。這組設定取自 6.5 的六組掃描，是唯一一組以密度為基礎的分群有效性指標（DBCV）高於 0.3 的設定。沿用先前執行（1.7）時，6.4 讀入該次的 UMAP 座標並重新分群。換了函式庫版本或處理器，HDBSCAN 可能把少數邊界紀錄分到不同的群，所以這格會報告重算結果與沿用分群的一致程度（adjusted Rand index），兩者不同時保留沿用的分群；6.5 則把沿用執行的掃描表寫成 `hdbscan_param_sweep.csv`，本次重算的寫成 `hdbscan_param_sweep_recomputed.csv`。每個群以關鍵詞（TF-IDF 與 KeyBERT 式排序，6.7）和合成模型給的標籤（6.8）描述。6.9 依群中心的餘弦相似度把群合併成 25 個主題，每個主題以其最大成員群的標籤命名；合併對照表寫在 `merge_map_worksheet.csv`，可供複核。

主題層在 7.16 與專利分類對照。在同一組向量上另外配適主題模型不算獨立的檢核，所以不使用。

In [ ]:
# ── 6.1  SPECTER2 encoder with the proximity adapter ────────────────────────
from adapters import AutoAdapterModel
# from_pretrained prints "There are adapters available but none are activated" before the adapter is loaded;
# the adapter is activated by load_adapter(set_active=True) and set_active_adapters below.
from transformers import AutoTokenizer

SPECTER2_MODEL = "allenai/specter2_base"

specter_tokenizer = AutoTokenizer.from_pretrained(SPECTER2_MODEL)
specter_model     = AutoAdapterModel.from_pretrained(SPECTER2_MODEL)
_ = specter_model.load_adapter("allenai/specter2", source="hf",
                               load_as="proximity", set_active=True)
specter_model.set_active_adapters("proximity")
specter_model = specter_model.eval().to(DEVICE)

dummy = specter_tokenizer("test", return_tensors="pt").to(specter_model.device)
with torch.no_grad():
    out = specter_model(**dummy)
print(f"SPECTER2 ready | dim = {out.last_hidden_state[:, 0, :].shape[-1]} | max 512 tokens")

In [ ]:
# ── 6.2  Embedding of title + synthesized summary ────────────────────────────
# With a reused run (1.7) the embeddings of that run are read, so that the clusters, topics and figures stay
# the same; the first 256 records are encoded again to show how close this hardware comes to the reused values.
def embed_texts(texts, batch_size=64):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        inputs = specter_tokenizer(batch, return_tensors="pt", truncation=True,
                                   padding=True, max_length=512).to(specter_model.device)
        with torch.no_grad():
            out = specter_model(**inputs)
        all_embeddings.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_embeddings)

texts = (pipeline_df["title"] + " " + pipeline_df["final_summary"]).tolist()
embed_path = os.path.join(SAVE_DIR, "specter2_embeddings.csv")
EMBEDDING_CHECK = None
if os.path.exists(embed_path):
    embed_df = pd.read_csv(embed_path)
    if list(embed_df["patent_id"].astype(str)) != list(pipeline_df["patent_id"].astype(str)):
        raise RuntimeError(f"{os.path.basename(embed_path)} does not follow the record order of the corpus; "
                           "delete it to encode again.")
    embeddings = embed_df.drop(columns=["patent_id"]).values.astype(np.float32)
    _n = min(256, len(texts))
    with timed("6.2  SPECTER2 check of the reused embeddings", n_items=_n):
        _new = embed_texts(texts[:_n], batch_size=64)
    _a, _b = _new, embeddings[:_n]
    _cos = (_a * _b).sum(1) / (np.linalg.norm(_a, axis=1) * np.linalg.norm(_b, axis=1))
    EMBEDDING_CHECK = {"records": _n, "max_abs_diff": float(np.abs(_a - _b).max()),
                       "min_cosine": float(_cos.min()), "mean_cosine": float(_cos.mean())}
    print(f"read {embed_path} ({embeddings.shape[0]:,} x {embeddings.shape[1]})")
    print(f"encoded again on {HARDWARE}: {_n} records | max |difference| {EMBEDDING_CHECK['max_abs_diff']:.2e} | "
          f"cosine min {EMBEDDING_CHECK['min_cosine']:.6f}, mean {EMBEDDING_CHECK['mean_cosine']:.6f}")
else:
    with timed("6.2  SPECTER2 embedding (title + final_summary)", n_items=len(texts)):
        embeddings = embed_texts(texts, batch_size=64)
    embed_df = pd.DataFrame(embeddings, columns=[f"dim_{i}" for i in range(embeddings.shape[1])])
    embed_df.insert(0, "patent_id", pipeline_df["patent_id"].values)
    embed_df.to_csv(embed_path, index=False)
    print(f"embeddings: {embeddings.shape} | saved -> {embed_path} ({os.path.getsize(embed_path)/1e6:.1f} MB)")

In [ ]:
# ── 6.3  Reload the embeddings and check them against the corpus ─────────────
embed_df   = pd.read_csv(os.path.join(SAVE_DIR, "specter2_embeddings.csv"))
patent_ids = embed_df["patent_id"].values
embeddings = embed_df.drop(columns=["patent_id"]).values.astype(np.float32)
print(f"embeddings: {embeddings.shape}")

# The identifiers must be those of the current corpus (see 4.4)
if "m1" in globals():
    _e, _m = set(embed_df["patent_id"].astype(str)), set(m1["patent_id"].astype(str))
    if _e != _m:
        raise RuntimeError(
            f"specter2_embeddings.csv does not match the current corpus ({len(_e)} vs {len(_m)} records). "
            "SAVE_DIR belongs to another corpus version. Use a new SAVE_DIR.")
    print("patent_id alignment check passed")

In [ ]:
# ── 6.4  UMAP (768 -> 3 dimensions), then HDBSCAN ───────────────────────────
# UMAP depends on the hardware and the library version even with a fixed seed. With a reused run, its
# three-dimensional coordinates are read and the clusters are computed again on them. HDBSCAN can still place
# a few boundary records differently under another library version or processor, so the result is compared
# with the reused assignment, and the reused assignment is kept when the two differ.
import umap
import hdbscan
from sklearn.metrics import silhouette_score, calinski_harabasz_score, adjusted_rand_score

_reused_clu = os.path.join(SAVE_DIR, "reused_cluster_results.csv")
REUSED_CLUSTERS = None
if os.path.exists(_reused_clu):
    REUSED_CLUSTERS = (pd.read_csv(_reused_clu, usecols=["patent_id", "cluster_id", "umap_3d_x", "umap_3d_y", "umap_3d_z"])
                       .astype({"patent_id": str}).set_index("patent_id")
                       .reindex(pd.Series(patent_ids).astype(str)))
    if REUSED_CLUSTERS["cluster_id"].isna().any():
        raise RuntimeError("The reused cluster file does not cover every record of this corpus.")
    # UMAP returns float32; the CSV holds their shortest decimal form, which converts back to the same float32 values
    embed_3d = REUSED_CLUSTERS[["umap_3d_x", "umap_3d_y", "umap_3d_z"]].values.astype(np.float32)
    umap_3d  = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.0, metric="cosine",
                         random_state=RANDOM_STATE, low_memory=False)   # settings only, for 9.1 (not fitted)
    print(f"UMAP coordinates read from the reused run: {embed_3d.shape}")
else:
    print("UMAP 768d -> 3d ...")
    with timed("6.4  UMAP 768d -> 3d", n_items=len(embeddings)):
        umap_3d = umap.UMAP(
            n_components=3, n_neighbors=15, min_dist=0.0,
            metric="cosine", random_state=RANDOM_STATE, low_memory=False,
        )
        embed_3d = umap_3d.fit_transform(embeddings)
    print(f"embed_3d: {embed_3d.shape}")

print("\nHDBSCAN ...")
with timed("6.4  HDBSCAN clustering", n_items=len(embed_3d)):
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES, metric="euclidean",
        cluster_selection_method="eom", prediction_data=True, gen_min_span_tree=True,
    )
    cluster_labels = clusterer.fit_predict(embed_3d)
REUSED_KEPT = False
dbcv_refit = clusterer.relative_validity_
if REUSED_CLUSTERS is not None:
    _reused_ids = REUSED_CLUSTERS["cluster_id"].astype(int).values
    _same = int((_reused_ids == cluster_labels).sum())
    # Cluster numbers follow the order in which HDBSCAN finds the clusters, so one changed boundary renumbers
    # the clusters after it. The adjusted Rand index and the matched count do not depend on the numbers:
    # a record counts as matched when it lies in the new cluster that takes most of its reused cluster
    # (noise counted as one group).
    _ari = adjusted_rand_score(_reused_ids, cluster_labels)
    _matched = int(pd.crosstab(_reused_ids, cluster_labels).max(axis=1).sum())
    print(f"clusters computed again on the reused coordinates: {_same:,} of {len(cluster_labels):,} records keep"
          f" the same cluster_id; {_matched:,} lie in the matching cluster (adjusted Rand index {_ari:.4f})")
    if _same != len(cluster_labels):
        print("  -> the reused assignment is kept, so that topics and audit stay those of the released dataset")
        cluster_labels = _reused_ids
        REUSED_KEPT = True

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_pct  = (cluster_labels == -1).sum() / len(cluster_labels) * 100
mask    = cluster_labels != -1
sil  = silhouette_score(embed_3d[mask], cluster_labels[mask], sample_size=min(500, int(mask.sum())),
                        random_state=RANDOM_STATE)
ch   = calinski_harabasz_score(embed_3d[mask], cluster_labels[mask])
dbcv = dbcv_refit
_dbcv_source = ""
_reused_sweep_path = os.path.join(SAVE_DIR, "reused_hdbscan_param_sweep.csv")
if REUSED_KEPT and os.path.exists(_reused_sweep_path):
    # DBCV needs the cluster tree, which exists only for the clusters computed here; the value of the kept
    # assignment is the one its own run recorded in the sweep.
    _row = pd.read_csv(_reused_sweep_path).query(
        "min_cluster_size == @HDBSCAN_MIN_CLUSTER_SIZE and min_samples == @HDBSCAN_MIN_SAMPLES")
    if len(_row):
        dbcv = float(_row["dbcv"].iloc[0])
        _dbcv_source = f"  (reused run; {dbcv_refit:.4f} for the clusters computed again)"

print(f"\n{'='*56}")
print("clustering metrics")
print(f"{'='*56}")
print(f"  Silhouette        : {sil:.4f}")
print(f"  DBCV              : {dbcv:.4f}{_dbcv_source}")
print(f"  Calinski-Harabasz : {ch:.2f}")
print(f"  noise share       : {noise_pct:.1f}%")
print(f"  clusters          : {n_clusters}")
print("reference values: Silhouette > 0.3 | DBCV > 0.3 | noise < 20%")
if PROFILE == "full":
    print("selection rule: of the six settings of the sweep in 6.5, the only one with DBCV > 0.3 is (30, 10); "
          "Silhouette and noise share are secondary criteria (full sweep in hdbscan_param_sweep.csv)")
else:
    print(f"sample profile: smaller settings ({HDBSCAN_MIN_CLUSTER_SIZE}, {HDBSCAN_MIN_SAMPLES}) for the small number of records")

In [ ]:
# ── 6.5  HDBSCAN parameter sweep (six settings) ─────────────────────────────
param_grid = [
    {"min_cluster_size": 10, "min_samples": 3},
    {"min_cluster_size": 15, "min_samples": 5},
    {"min_cluster_size": 20, "min_samples": 5},
    {"min_cluster_size": 20, "min_samples": 10},
    {"min_cluster_size": 25, "min_samples": 5},
    {"min_cluster_size": 30, "min_samples": 10},
]

print(f"  {'min_cluster':>12} {'min_samples':>12} {'n_cluster':>10} "
      f"{'noise%':>8} {'Silhouette':>12} {'DBCV':>8}")
sweep_rows = []
_sweep_timer = timed(f"6.5  HDBSCAN parameter sweep ({len(param_grid)} settings)",
                     n_items=len(param_grid))
_ = _sweep_timer.__enter__()
for params in tqdm(param_grid, desc="param sweep"):
    clf = hdbscan.HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        metric="euclidean", cluster_selection_method="eom",
        gen_min_span_tree=True, prediction_data=True)
    labels = clf.fit_predict(embed_3d)
    n_clust = len(set(labels)) - (1 if -1 in labels else 0)
    npct    = (labels == -1).sum() / len(labels) * 100
    msk     = labels != -1
    if msk.sum() < 10:
        continue
    if len(set(labels[msk])) < 2:
        continue
    s = silhouette_score(embed_3d[msk], labels[msk], sample_size=min(500, int(msk.sum())),
                         random_state=RANDOM_STATE)
    d = clf.relative_validity_
    sweep_rows.append({**params, "n_cluster": n_clust, "noise_pct": round(npct,1),
                       "silhouette": round(s,4), "dbcv": round(d,4)})
    print(f"  {params['min_cluster_size']:>12} {params['min_samples']:>12} "
          f"{n_clust:>10} {npct:>7.1f}% {s:>12.4f} {d:>8.4f}")

_ = _sweep_timer.__exit__(None, None, None)

sweep_df   = pd.DataFrame(sweep_rows)
sweep_path = os.path.join(SAVE_DIR, "hdbscan_param_sweep.csv")
if REUSED_KEPT and os.path.exists(_reused_sweep_path):
    # The kept clusters come from the reused run, so its sweep is the one that describes them; the sweep of
    # this run is written next to it.
    _recomputed_path = os.path.join(SAVE_DIR, "hdbscan_param_sweep_recomputed.csv")
    sweep_df.to_csv(_recomputed_path, index=False)
    _ = shutil.copy2(_reused_sweep_path, sweep_path)
    _sweep_cmp = sweep_df.merge(pd.read_csv(_reused_sweep_path), on=["min_cluster_size", "min_samples"],
                                suffixes=("_this_run", "_reused"))
    print("\nthis run against the reused run:")
    print(_sweep_cmp[["min_cluster_size", "min_samples", "n_cluster_this_run", "n_cluster_reused",
                      "dbcv_this_run", "dbcv_reused"]].to_string(index=False))
    for _name, _tab in (("this run", sweep_df), ("reused run", pd.read_csv(_reused_sweep_path))):
        _above = [(int(a), int(b)) for a, b in _tab.loc[_tab["dbcv"] > 0.3, ["min_cluster_size", "min_samples"]].values]
        print(f"settings with DBCV > 0.3, {_name}: {_above}")
    print(f"\nsaved -> {sweep_path} (reused run) and {_recomputed_path} (this run)")
else:
    sweep_df.to_csv(sweep_path, index=False)
    print(f"\nsaved -> {sweep_path}")

In [ ]:
# ── 6.6  Cluster table cluster_results.csv ──────────────────────────────────
result_df = pd.DataFrame({
    "patent_id"  : patent_ids,
    "cluster_id" : cluster_labels,
    "umap_3d_x"  : embed_3d[:, 0],
    "umap_3d_y"  : embed_3d[:, 1],
    "umap_3d_z"  : embed_3d[:, 2],
})
result_df = result_df.merge(
    pipeline_df[["patent_id", "title", "final_summary"]],
    on="patent_id", how="left")
result_df = result_df.merge(
    add_ipc3(m1)[["patent_id", "current_assignee", "main_ipc_4", "main_ipc_3"]],
    on="patent_id", how="left")

result_path = os.path.join(SAVE_DIR, "cluster_results.csv")
result_df.to_csv(result_path, index=False)
print(f"saved -> {result_path} | columns: {list(result_df.columns)}")

# Overview of each cluster
cluster_df = result_df[result_df["cluster_id"] != -1]
n_noise = (result_df["cluster_id"] == -1).sum()
print(f"\nnoise: {n_noise} ({n_noise/len(result_df)*100:.1f}%)")
print(f"\n  {'cluster':>8} {'n':>6} {'share':>7}  {'top IPC':10} top company")
for cid in sorted(cluster_df["cluster_id"].unique()):
    rows = cluster_df[cluster_df["cluster_id"] == cid]
    top_ipc = rows["main_ipc_4"].value_counts().index[0]
    top_co  = rows["current_assignee"].value_counts().index[0][:32]
    print(f"  {cid:>8} {len(rows):>6} {len(rows)/len(result_df)*100:>6.1f}%  {top_ipc:10} {top_co}")

In [ ]:
# ── 6.7  Keywords of each cluster (TF-IDF and KeyBERT style) ────────────────
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cluster_df = result_df[result_df["cluster_id"] != -1]
cluster_ids = sorted(cluster_df["cluster_id"].unique())

# TF-IDF (1-2 grams) on the joined document of each cluster
cluster_docs = {cid: " ".join(cluster_df[cluster_df["cluster_id"] == cid]["final_summary"])
                for cid in cluster_ids}
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english",
                             ngram_range=(1, 2), min_df=2)
tfidf_matrix  = vectorizer.fit_transform([cluster_docs[c] for c in cluster_ids])
feature_names = vectorizer.get_feature_names_out()

tfidf_keywords = {}
print("TF-IDF keywords (top 8 per cluster):")
for i, cid in enumerate(cluster_ids):
    scores  = tfidf_matrix[i].toarray()[0]
    top_idx = scores.argsort()[::-1][:8]
    kws     = [feature_names[j] for j in top_idx if scores[j] > 0]
    tfidf_keywords[cid] = kws
    print(f"  C{cid:>2} ({(cluster_df['cluster_id']==cid).sum():>4}): {', '.join(kws)}")

# KeyBERT style: SPECTER2 similarity between the cluster text and the candidate n-grams
def embed_texts_simple(texts):
    embs = []
    for t in texts:
        inputs = specter_tokenizer(t, return_tensors="pt", truncation=True,
                                   max_length=512, padding=True).to(specter_model.device)
        with torch.no_grad():
            out = specter_model(**inputs)
        embs.append(out.last_hidden_state[:, 0, :].cpu().numpy()[0])
    return np.array(embs)

def extract_keywords_keybert(text, top_n=5):
    count = CountVectorizer(ngram_range=(1,2), stop_words="english",
                            max_features=50).fit([text])
    candidates = count.get_feature_names_out().tolist()
    if not candidates:
        return []
    doc_emb  = embed_texts_simple([text[:1000]])
    cand_emb = embed_texts_simple(candidates)
    sims = cosine_similarity(doc_emb, cand_emb)[0]
    return [candidates[i] for i in sims.argsort()[::-1][:top_n]]

keybert_keywords = {}
print("\nKeyBERT-style keywords (top 5 per cluster):")
for cid in tqdm(cluster_ids, desc="KeyBERT"):
    rows = cluster_df[cluster_df["cluster_id"] == cid]
    text = " ".join(rows["final_summary"].head(10).tolist())
    keybert_keywords[cid] = extract_keywords_keybert(text, top_n=5)
    print(f"  C{cid:>2}: {', '.join(keybert_keywords[cid])}")

free_gpu("specter_model", "specter_tokenizer")

In [ ]:
# ── 6.8  Topic label of each cluster by the synthesis model ──────────────────
# Ten records of each cluster (seeded draw) go into one prompt; the model returns a label and five keywords.
# With a reused run and the same cluster assignment, the labels of that run are read and no model is loaded.
CLUSTER_LABEL_PROMPT = (
    "You are a patent analyst. Based on the following patent titles and summaries "
    "from the same technology cluster, provide:\n"
    "1. A concise cluster label (3-6 words)\n"
    "2. Top 5 technical keywords\n\n"
    "Format your response EXACTLY as:\n"
    "LABEL: <cluster label>\n"
    "KEYWORDS: <kw1>, <kw2>, <kw3>, <kw4>, <kw5>\n\n"
    "Patents:\n{patents}\n"
)

llm_labels, llm_keywords = {}, {}
_reuse_labels = (REUSED_CLUSTERS is not None and
                 (REUSED_CLUSTERS["cluster_id"].astype(int).values == cluster_labels).all())
if _reuse_labels:
    _lab = (pd.read_csv(_reused_clu, usecols=["cluster_id", "llm_label", "llm_keywords"])
              .fillna("").drop_duplicates("cluster_id").set_index("cluster_id"))
    for cid in cluster_ids:
        llm_labels[cid]   = _lab.loc[cid, "llm_label"]
        llm_keywords[cid] = str(_lab.loc[cid, "llm_keywords"]).split(", ")   # joined again with ", " below: identical text
        print(f"  C{cid:>2} ({int((cluster_df['cluster_id'] == cid).sum()):>4}): {llm_labels[cid]} | {', '.join(llm_keywords[cid])}")
    print(f"\n{len(llm_labels)} labels read from the reused run (no model loaded)")
else:
    if "tokenizer_s2" not in globals():
        tokenizer_s2 = AutoTokenizer.from_pretrained(S2_MODEL_NAME, trust_remote_code=True, padding_side="left")
        if tokenizer_s2.pad_token is None:
            tokenizer_s2.pad_token = tokenizer_s2.eos_token
    if "model_s2" not in globals():
        model_s2 = AutoModelForCausalLM.from_pretrained(
            S2_MODEL_NAME, torch_dtype=GEN_DTYPE, trust_remote_code=True,
            device_map="auto" if HAS_CUDA else None)
        if not HAS_CUDA:
            model_s2 = model_s2.to(DEVICE)
        model_s2 = model_s2.eval()
    with timed(f"6.8  Cluster labels ({S2_MODEL_NAME.split('/')[-1]})", n_items=len(cluster_ids)):
        for cid in tqdm(cluster_ids, desc="LLM labels"):
            rows   = cluster_df[cluster_df["cluster_id"] == cid]
            sample = rows.sample(min(10, len(rows)), random_state=RANDOM_STATE)
            patents_text = "\n".join([f"- {r['title']}: {r['final_summary'][:200]}..."
                                       for _, r in sample.iterrows()])
            output = generate_batch_s2([CLUSTER_LABEL_PROMPT.format(patents=patents_text)],
                                       max_new_tokens=100)[0]
            label, keywords = f"Cluster {cid}", []
            for line in output.split("\n"):
                if line.startswith("LABEL:"):
                    label = line.replace("LABEL:", "").strip()
                elif line.startswith("KEYWORDS:"):
                    keywords = [k.strip() for k in line.replace("KEYWORDS:", "").split(",")]
            llm_labels[cid], llm_keywords[cid] = label, keywords
            print(f"  C{cid:>2} ({len(rows):>4}): {label} | {', '.join(keywords)}")

# ── The only definition of the non-topic labels ───
NON_TOPIC_SENTINEL = "__noise__"
_JUNK_LABEL_PAT = re.compile(
    r"^(noise|unknown|unclassified|unlabel(l?ed)?|misc(ellaneous)?|other(s)?|"
    r"various|general|n/?a|none|null|cluster\s*-?\d+)(\s*\(\d+\))?$", re.I)
def is_junk_label(s):
    """The language model sometimes outputs uninformative labels such as Noise/Unknown. This function is only for the decision on "naming candidates";
    whether a record belongs to a topic is always decided by cluster_id == -1, not by a string comparison."""
    s = str(s).strip()
    return (not s) or s == NON_TOPIC_SENTINEL or bool(_JUNK_LABEL_PAT.match(s))

# Label clean-up: an uninformative label falls back to a placeholder name, for the naming logic of 6.9
_n_junk = sum(is_junk_label(v) for v in llm_labels.values())
llm_labels = {k: (f"Cluster {k}" if is_junk_label(v) else v) for k, v in llm_labels.items()}
print(f"\nLanguage-model label clean-up: {_n_junk} uninformative labels fell back to Cluster <id> ({len(llm_labels)} clusters in total)")

# cluster_id == -1 marks the HDBSCAN noise points, which have no label; a sentinel is used instead of "Noise",
# so that downstream steps do not treat it as a valid topic name
result_df["llm_label"]    = result_df["cluster_id"].map(llm_labels).fillna(NON_TOPIC_SENTINEL)
result_df["llm_keywords"] = result_df["cluster_id"].map(
    lambda x: ", ".join(llm_keywords.get(x, [])))
result_df.to_csv(result_path, index=False)
print(f"\nsaved LLM labels -> {result_path}")

free_gpu("model_s2", "tokenizer_s2")

### 6.9–6.10  合併主題

合併後得到 25 個主題，每個至少 30 筆；比這小的群會併入群中心最相近的主題。HDBSCAN 雜訊點的紀錄標為 `__noise__`，不屬於任何主題。6.10 可以在不改動任何數字的情況下替主題改名；釋出版本沒有改名。

In [ ]:
# ── 6.9  Merge the clusters into N_TARGET topics (hierarchical, by centroid) ──
# The merge works on the HDBSCAN cluster_id of 6.4, and the names come from the labels that 6.8 made for the
# same cluster_id, so both use the same identifier space.
# Note: merging and naming always happen in the cluster_id space; if you add another clustering result,
#     make sure that the id spaces agree. Otherwise the label look-up takes the label of an arbitrary member.
import numpy as np, pandas as pd, os
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

# N_TARGET (number of merged topics) and MIN_CLASS_N (minimum records per topic; a smaller topic joins the
# topic with the most similar centroid) are set in 1.3.

if "llm_label" not in result_df.columns:
    raise RuntimeError("Run 6.8 first (topic labels).")

# Check of the id space: llm_label must be a function of cluster_id
_chk = result_df.groupby("cluster_id")["llm_label"].nunique()
if int(_chk.max()) > 1:
    raise RuntimeError("llm_label is not unique within one cluster_id; the id spaces do not agree: "
                       f"{_chk[_chk > 1].index.tolist()[:5]}")

cid      = result_df["cluster_id"].values
label_of = (result_df.drop_duplicates("cluster_id")
                     .set_index("cluster_id")["llm_label"].to_dict())
kw_of    = (result_df.drop_duplicates("cluster_id")
                     .set_index("cluster_id")["llm_keywords"].to_dict())

valid_ids = sorted(c for c in set(cid) if c != -1)
cent  = np.vstack([embeddings[cid == c].mean(axis=0) for c in valid_ids])
sizes = np.array([(cid == c).sum() for c in valid_ids])
print(f"clusters to merge: {len(valid_ids)} | noise(-1): {(cid == -1).sum()}")

try:                                   # sklearn >= 1.2
    agg = AgglomerativeClustering(n_clusters=min(N_TARGET, len(valid_ids)),
                                  metric="cosine", linkage="average")
except TypeError:                      # sklearn < 1.2
    agg = AgglomerativeClustering(n_clusters=min(N_TARGET, len(valid_ids)),
                                  affinity="cosine", linkage="average")
super_id = agg.fit_predict(cent)

# ── Naming: from the largest to the smallest, take the label of a member cluster that is "not yet used" ───
# When the largest members of two groups have the same name, use the next largest member, so that no meaningless "name (2)" is made.
name_of, used = {}, set()
for s in sorted(set(super_id), key=lambda x: -int(sizes[super_id == x].sum())):
    mem = sorted(np.where(super_id == s)[0], key=lambda i: -sizes[i])
    nm  = None
    for i in mem:
        cand = str(label_of.get(valid_ids[i], "")).strip()
        if cand and (cand not in used) and (not is_junk_label(cand)):
            nm = cand
            break
    if nm is None:
        # No member has a usable label: give a structured placeholder name, for manual naming in merge_map_worksheet.csv
        nm = f"Topic {int(s):02d}"
    used.add(nm)
    name_of[s] = nm

# ── Merge topics that are too small: move the whole group to the nearest other super-topic by cosine ───
for _ in range(len(set(super_id))):
    cnt   = {s: int(sizes[super_id == s].sum()) for s in set(super_id)}
    small = [s for s, n in cnt.items() if n < MIN_CLASS_N]
    if not small or len(cnt) <= 2:
        break
    s      = min(small, key=lambda x: cnt[x])
    others = [o for o in cnt if o != s]
    sc_s   = cent[super_id == s].mean(axis=0, keepdims=True)
    sc_o   = np.vstack([cent[super_id == o].mean(axis=0) for o in others])
    tgt    = others[int(np.argmax(cosine_similarity(sc_s, sc_o)[0]))]
    print(f"  merged: {name_of[s]} ({cnt[s]}) -> {name_of[tgt]}")
    super_id[super_id == s] = tgt

MERGE_MAP = {c: name_of[s] for c, s in zip(valid_ids, super_id)}
MERGE_MAP[-1] = NON_TOPIC_SENTINEL

# With a reused run and the same clusters, the topic of each cluster is taken from that run: the merge works on
# floating-point centroids, and a borderline merge can turn out differently on other hardware. The merge of
# this run is compared with it.
if REUSED_CLUSTERS is not None and _reuse_labels:
    _rmap = (pd.read_csv(_reused_clu, usecols=["cluster_id", "merged_topic"]).fillna("")
               .drop_duplicates("cluster_id").set_index("cluster_id")["merged_topic"].to_dict())
    _n_diff = sum(MERGE_MAP.get(c) != _rmap.get(c) for c in valid_ids)
    print(f"merge of this run vs the reused run: {len(valid_ids) - _n_diff} of {len(valid_ids)} clusters in the same topic"
          + ("" if _n_diff == 0 else " -> the topics of the reused run are kept"))
    MERGE_MAP = {c: _rmap.get(c, MERGE_MAP[c]) for c in valid_ids}
    MERGE_MAP[-1] = NON_TOPIC_SENTINEL
result_df["merged_topic"] = result_df["cluster_id"].map(MERGE_MAP).fillna(NON_TOPIC_SENTINEL)

# ── Hard assertion: change "false-topic contamination" from a silent defect into a failure at run time ───
_bad = sorted({t for t in result_df["merged_topic"].unique()
               if t != NON_TOPIC_SENTINEL and is_junk_label(t)})
assert not _bad, f"merged_topic has uninformative topic names; the clean-up of 6.8/6.9 did not work: {_bad}"

# ── Result table ───────────────────────────────────────────────────────────
print(f"\n{'merged topic':42} {'n':>6}  top IPC / top company")
rows = []
for merged, n in result_df["merged_topic"].value_counts().items():
    mr  = result_df[result_df["merged_topic"] == merged]
    ipc = " | ".join(f"{i}({k})" for i, k in mr["main_ipc_4"].value_counts().head(2).items())
    co  = " | ".join(f"{c[:20]}({k})" for c, k in mr["current_assignee"].value_counts().head(2).items())
    print(f"{merged[:42]:42} {n:>6}  {ipc}   {co}")
    rows.append({"merged_topic": merged, "patent_count": n, "top_ipc": ipc, "top_company": co})

NON_TOPIC_VALUES = [NON_TOPIC_SENTINEL, "Off-topic / Non-HPC", "Unclassified", ""]
_topic_only = result_df.loc[~result_df["merged_topic"].isin(NON_TOPIC_VALUES), "merged_topic"]
hpc = len(_topic_only)
print(f"\nTopics: {_topic_only.nunique()} (non-topic not included)"
      f" | with topic {hpc:,} ({hpc/len(result_df)*100:.1f}%)"
      f" | smallest topic {_topic_only.value_counts().min()} records"
      f" | non-topic {len(result_df)-hpc:,} ({(len(result_df)-hpc)/len(result_df)*100:.1f}%)")

result_df.to_csv(result_path, index=False)
pd.DataFrame(rows).to_csv(os.path.join(SAVE_DIR, "topic_summary.csv"), index=False)

# ── Worksheet for manual edits: use this table to rename topics or to mark Off-topic ───
sheet = pd.DataFrame([{
    "cluster_id": c,
    "n": int(sizes[i]),
    "auto_merged_topic": MERGE_MAP[c],
    "llm_label": label_of.get(c, ""),
    "llm_keywords": kw_of.get(c, ""),
    "top_ipc": result_df[cid == c]["main_ipc_4"].value_counts().index[:1].tolist(),
} for i, c in enumerate(valid_ids)]).sort_values(["auto_merged_topic", "n"],
                                                ascending=[True, False])
sheet.to_csv(os.path.join(SAVE_DIR, "merge_map_worksheet.csv"), index=False)
print("saved -> cluster_results.csv, topic_summary.csv, merge_map_worksheet.csv")

# The mapping is printed for review; merge_map_worksheet.csv holds the same information
print("\nMERGE_MAP = {")
for c in valid_ids:
    print(f'    {c:>4}: "{MERGE_MAP[c]}",')
print(f'    -1: "{NON_TOPIC_SENTINEL}",\n}}')

In [ ]:
# ── 6.10  Optional renaming of topic labels (no change while RENAME is empty) ──
# The topic names come from the language model. Four pairs are nearly identical:
#   "Cybersecurity & Data Protection"   and "Cyber Security & Anomaly Detection"
#   "Power Management & Efficiency"     and "Power Management & Storage Optimization"
#   "Neural Network Optimization"       and "Neural Network Image Processing"
#   "Network Optimization & Management" and "High Performance Network Management"
# The released version 1.0 keeps the generated names. To rename, fill in RENAME and run this cell, then the
# cells that read the topic names (7.6, 7.7, 7.11 to 7.17, 6.11, 8.13). Only strings change; every number stays.
# 9.3 applies the same RENAME to topic_label.
RENAME = {
    # "old name": "new name",
}
_SENT = globals().get("NON_TOPIC_SENTINEL", "__noise__")
if RENAME:
    _cr = os.path.join(SAVE_DIR, "cluster_results.csv")
    result_df = pd.read_csv(_cr).fillna("")
    _before = set(result_df["merged_topic"])
    assert set(RENAME) <= _before, f"RENAME has old names that do not exist: {set(RENAME) - _before}"
    result_df["merged_topic"] = result_df["merged_topic"].replace(RENAME)
    _labels = [t for t in result_df["merged_topic"].unique() if t != _SENT]
    assert len(_labels) == len(set(_labels)), "duplicate topic names after the renaming"
    if "is_junk_label" in globals():
        assert not [t for t in _labels if is_junk_label(t)], "uninformative names after the renaming"
    result_df.to_csv(_cr, index=False)
    for _f, _col in (("topic_summary.csv", "merged_topic"), ("merge_map_worksheet.csv", "auto_merged_topic")):
        _p = os.path.join(SAVE_DIR, _f)
        if os.path.exists(_p):
            _t = pd.read_csv(_p).fillna(""); _t[_col] = _t[_col].replace(RENAME); _t.to_csv(_p, index=False)
    if "MERGE_MAP" in globals():
        MERGE_MAP = {k: RENAME.get(v, v) for k, v in MERGE_MAP.items()}
    print(f"Renamed {len(RENAME)} labels; {len(_labels)} topics. Run 7.6, 7.7, 7.11 to 7.17, 6.11 and 8.13 again.")
else:
    print("RENAME is empty: the generated topic names are kept.")

### 6.11–6.13  釋出的五種表徵

6.11 建立評估表 `valid_df`：所有釋出紀錄及其三個摘要層。6.12 以 SPECTER2 對每筆紀錄的五種表徵編碼，6.13 寫成 parquet 檔：

| 表徵 | 編碼的文字 |
|---|---|
| abstract | title [SEP] abstract（SPECTER2 原生的輸入格式，不截斷） |
| fulltext-truncated | title、abstract、background、summary、detailed description 串接，編碼器只保留前 512 個 token |
| direct-brief | title + direct-brief 摘要 |
| direct-full | title + direct-full 摘要 |
| synthesized | title + synthesized 摘要（即 6.2 的向量） |

In [ ]:
# ── 6.11  Evaluation table valid_df: all released records with their three summary layers ──
# The summary-layer measurements of section 8 do not depend on the topics and use every released record,
# so that the clustering does not select the evaluation sample. Only the within-topic similarity (8.13)
# uses topics; topic_mask selects the records with a topic of at least five records.
if "extract_tech_terms" not in globals():
    raise RuntimeError("Run 5.3 first (definition of the term extraction; section 8 uses it).")

pipeline_df = pd.read_csv(os.path.join(SAVE_DIR, "patent_full_pipeline.csv")).fillna("")
result_df   = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
stage1_df   = pd.read_csv(os.path.join(SAVE_DIR, "stage1_summaries.csv")).fillna("")

valid_df = (pipeline_df.merge(result_df[["patent_id", "merged_topic"]], on="patent_id", how="left")
                       .fillna("").reset_index(drop=True))
for _path, _col in ((DIRECT_PATH, "baseline_summary"), (MATCHED_PATH, "baseline_matched_summary")):
    _layer = pd.read_csv(_path).fillna("")[["patent_id", _col]]
    valid_df = valid_df.merge(_layer, on="patent_id", how="left").fillna("")
for _col in ("final_summary", "baseline_summary", "baseline_matched_summary"):
    _n_empty = int((valid_df[_col].astype(str).str.strip() == "").sum())
    print(f"  {_col:26} empty in {_n_empty:,} of {len(valid_df):,} records")

NON_TOPIC  = ["__noise__", "Off-topic / Non-HPC", "Unclassified", ""]
_tc        = valid_df.loc[~valid_df["merged_topic"].isin(NON_TOPIC), "merged_topic"].value_counts()
topic_mask = (~valid_df["merged_topic"].isin(NON_TOPIC)
              & valid_df["merged_topic"].isin(_tc[_tc >= 5].index)).values

groups_valid = build_family_groups(valid_df["patent_id"].values)
print(f"valid_df (summary-layer measurements): {len(valid_df):,} records | "
      f"families {pd.Series(groups_valid).nunique():,}")
print(f"records with a topic (for 8.13)       : {topic_mask.sum():,} records | "
      f"topics {valid_df.loc[topic_mask, 'merged_topic'].nunique()}")

In [ ]:
# ── 6.12  SPECTER2 embeddings of the five representations ────────────────────
# Five representations are released side by side; their content is described, they are not ranked:
#   abstract             title [SEP] abstract      the existing abstract, about 115 words, no cut
#   fulltext-truncated   five fields joined         what is left after the 512-token cut of the encoder
#   direct-brief         title + baseline_summary   single prompt, short reading (about 2,000 tokens)
#   direct-full          title + baseline_matched   single prompt, full reading (about 5,000 tokens)
#   synthesized          title + final_summary      facets and synthesis, full reading (about 5,000 tokens)
# With a reused run, its released embeddings are read (same record order required); otherwise they are computed.
from transformers import AutoTokenizer as AT
from adapters import AutoAdapterModel
from sklearn.preprocessing import normalize as _l2norm

if "specter_model" not in globals():
    specter_tokenizer = AT.from_pretrained("allenai/specter2_base")
    specter_model     = AutoAdapterModel.from_pretrained("allenai/specter2_base")
    _ = specter_model.load_adapter("allenai/specter2", source="hf",
                                   load_as="proximity", set_active=True)
    specter_model.set_active_adapters("proximity")
    specter_model = specter_model.eval().to(DEVICE)

EMB_BATCH = 64
def embed_batch(texts, desc="embed", batch_size=EMB_BATCH):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        inputs = specter_tokenizer(texts[i:i+batch_size], return_tensors="pt",
                                   truncation=True, padding=True, max_length=512
                                   ).to(specter_model.device)
        with torch.no_grad():
            out = specter_model(**inputs)
        embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return _l2norm(np.vstack(embs), norm="l2")

def reused_or_embed(rep, texts, label):
    """Read the reused embeddings of one representation, or compute them."""
    _p = os.path.join(SAVE_DIR, "reused_embeddings", f"embeddings_{rep}.parquet")
    if os.path.exists(_p):
        _e = pd.read_parquet(_p)
        if list(_e["patent_id"].astype(str)) == list(valid_df["patent_id"].astype(str)):
            print(f"  {rep:20} read from the reused run")
            return _e.drop(columns=["patent_id"]).values.astype(np.float32)
        print(f"  {rep:20} reused file has another record order; computed again")
    with timed(f"6.12  SPECTER2 embedding ({label})", n_items=len(texts)):
        return embed_batch(texts, desc=f"embed [{rep}]")

_SUM_COL = "summary_filled" if "summary_filled" in valid_df.columns else "summary"
orig_texts = (valid_df["title"].astype(str) + " " + valid_df["abstract"].astype(str) +
              " " + valid_df["background"].astype(str) +
              " " + valid_df[_SUM_COL].astype(str) +
              " " + valid_df["detailed_description"].astype(str)).tolist()
X_orig = reused_or_embed("fulltext-truncated", orig_texts,
                         "fulltext-truncated: five fields joined, encoder cut at 512")
X_direct = reused_or_embed("direct-brief", (valid_df["title"].astype(str) + " " +
                                            valid_df["baseline_summary"].astype(str)).tolist(), "direct-brief")
# abstract: the native input of SPECTER2 is title + separator token + abstract; the abstracts fit into 512 tokens
_SEP = specter_tokenizer.sep_token or "[SEP]"
X_abs = reused_or_embed("abstract", (valid_df["title"].astype(str) + _SEP +
                                     valid_df["abstract"].astype(str)).tolist(), "abstract, title[SEP]abstract")
X_match = None
if valid_df["baseline_matched_summary"].astype(str).str.len().gt(0).any():
    X_match = reused_or_embed("direct-full", (valid_df["title"].astype(str) + " " +
                                              valid_df["baseline_matched_summary"].astype(str)).tolist(), "direct-full")

# synthesized: the embeddings of 6.2, normalised to unit length
embed_df = pd.read_csv(os.path.join(SAVE_DIR, "specter2_embeddings.csv"))
emb_cols = [c for c in embed_df.columns if c != "patent_id"]
_m = valid_df[["patent_id"]].merge(embed_df, on="patent_id", how="left")
assert _m[emb_cols].notna().all().all(), "the embeddings of 6.2 miss records; run 6.2 to the end first"
X_twostage = _l2norm(_m[emb_cols].values.astype(np.float32), norm="l2")
labels_valid = valid_df["merged_topic"].values

# The five representations for all later steps (a None entry is skipped)
REPRESENTATIONS = {
    "abstract":           X_abs,
    "fulltext-truncated": X_orig,
    "direct-brief":       X_direct,
    "direct-full":        X_match,
    "synthesized":        X_twostage,
}
for _k, _v in REPRESENTATIONS.items():
    print(f"  {_k:20} {'-' if _v is None else str(_v.shape)}")

In [ ]:
# ── 6.13  Write the embeddings of the five representations (parquet, float32) ──
# Full text, three condensed layers and the embedding of each are released for the same records; this cell
# writes the third part. A CSV of 768-dimensional floats would be more than three times larger.
import os
import numpy as np, pandas as pd

EMB_DIR = os.path.join(SAVE_DIR, "embeddings")
os.makedirs(EMB_DIR, exist_ok=True)
_ids   = valid_df["patent_id"].astype(str).values
_saved = []

for _rep, _X in REPRESENTATIONS.items():
    if _X is None:
        print(f"  (skipped {_rep}: not built)")
        continue
    if len(_X) != len(_ids):
        raise RuntimeError(f"{_rep}: {len(_X):,} rows, but valid_df has {len(_ids):,}")
    _cols = [f"dim_{i}" for i in range(_X.shape[1])]
    _out  = pd.DataFrame(np.asarray(_X, dtype=np.float32), columns=_cols)
    _out.insert(0, "patent_id", _ids)
    _base = f"embeddings_{_rep.replace(' ', '_').replace('/', '_')}"
    try:                                    # needs pyarrow or fastparquet
        _p = os.path.join(EMB_DIR, _base + ".parquet")
        _out.to_parquet(_p, index=False, compression="gzip")
    except Exception as _e:                 # falls back to a compressed CSV when no parquet engine is available
        _p = os.path.join(EMB_DIR, _base + ".csv.gz")
        _out.to_csv(_p, index=False, compression="gzip")
        print(f"  (parquet not available; saved as csv.gz: {type(_e).__name__})")
    _mb = os.path.getsize(_p) / 1e6
    _saved.append({"representation": _rep, "file": os.path.basename(_p),
                   "rows": int(len(_X)), "dim": int(_X.shape[1]), "MB": round(_mb, 2)})
    print(f"  {os.path.basename(_p):46} {len(_X):>6,} x {_X.shape[1]:>4}  {_mb:>7.2f} MB")

if not _saved:
    raise RuntimeError("No representation was written: run 6.12 first.")
pd.DataFrame(_saved).to_csv(os.path.join(EMB_DIR, "MANIFEST.csv"), index=False)
print(f"\n{len(_saved)} representations released -> {EMB_DIR}")
print("(specter2_embeddings.csv of 6.2 holds the synthesized representation before normalisation; the later steps read it)")

### 第 6 節結果（釋出版本）

HDBSCAN 分出 51 個群，1,502 筆（23.0%）屬於雜訊；輪廓係數 0.588，DBCV 0.3105。雜訊比例高於當初訂的次要標準 20%；把 `min_cluster_size` 調小會得到更多群、DBCV 更低，雜訊卻沒有明顯減少（見 6.5 的表）。合併後的 25 個主題涵蓋 5,033 筆（77.0%）。最大的主題是 Advanced Packaging Solutions（731 筆）、Distributed Computing Optimization（609 筆）與 Power Management & Efficiency（523 筆）；最小的主題有 31 筆。有四組主題名稱幾乎相同，因為模型替相關的群取了相近的名字；每一組都是兩群不同的紀錄，可以用 6.10 改成不同的名稱。

## 7  權屬證據與使用示範

### 7.1–7.4  每一筆權屬異動的證據等級

`current_assignee` 是 Google Patents 頁面列出的第一位現任權利人，不一定有所有權移轉的紀錄支持：最後一筆讓與紀錄可能是擔保設定或擔保解除，可能列了好幾位當事人，也可能根本沒有。7.1 解析 `legal_events` 中所有讓與紀錄，以九條依序判斷的規則分類最新的一筆，寫出 `ownership_events.csv`。這個檔保留原始欄位，另外給一個建議解析 `effective_assignee`：沒有任何移轉所有權的紀錄支持異動時，退回原始申請人。使用者也可以改用自己的門檻。7.2 到 7.4 列出權利人名稱的不同寫法，並為圖表加上正規化名稱。

In [ ]:
# ── 7.1  Evidence grade of each change of ownership (read-only) ──────────────
# current_assignee is the first current assignee listed on the Google Patents page (corpus/json_to_csv.py) and does not necessarily come from the last assignment record; that record can
#   (a) be no transfer of ownership at all (security interest, release of a security interest, corrective record),
#   (b) name more than one party (up to 16 in this corpus); then the record cannot show which one the page lists,
#   (c) be absent (current_assignee differs from assignee, but there is no assignment record).
# This section parses all assignment segments with code="AS" in legal_events, takes the record with the latest "event date + effective date",
# decides its assignment type, records the number of parties that the record names and writes the release file
# ownership_events.csv, so that users can select their own entity resolution rule.
import os, re, ast
import pandas as pd

# The description of one AS event joins several assignment records, always in this format:
#   Owner name:<assignee>Free format text:<assignment type>;ASSIGNOR:<assignor>
#   REEL/FRAME:<USPTO assignment record number>Effective date:<YYYYMMDD>
SEGMENT_RE = re.compile(
    r"Owner name:(?P<owner>.*?)"
    r"Free format text:(?P<conveyance>.*?)"
    r"(?:REEL/FRAME:(?P<reel>[\d/]+))?"
    r"(?:Effective date:(?P<effective>\d{8}))?"
    r"(?=Owner name:|$)", re.S)

# Decision of the assignment type: compare in order; the first hit decides. RELEASE must come before SECURITY:
# "RELEASE OF SECURITY INTEREST" contains both keywords, and its meaning is a release, not a grant of security.
CONVEYANCE_RULES = [
    ("release_of_lien",   ("RELEASE",)),
    ("security_interest", ("SECURITY",)),
    ("change_of_name",    ("CHANGE OF NAME", "CERTIFICATE OF AMENDMENT")),
    ("merger",            ("MERGER",)),
    ("partial_transfer",  ("PARTIAL",)),
    ("corrective",        ("CORRECTIVE",)),
    ("true_assignment",   ("ASSIGNMENT OF ASSIGNORS INTEREST", "NUNC PRO TUNC",
                           "CONFIRMATORY", "PATENT ASSIGNMENT")),
]

# Only these three assignment types actually transfer ownership; the others cannot support a change of assignee.
TITLE_TRANSFERRING = {"true_assignment", "merger", "change_of_name"}

# Verified data-source error: this name never occurs in any legal event, and the original assignee is the same in all records.
ASSIGNEE_OVERRIDES = {"Everpure Inc": "Pure Storage Inc"}


def _latest_record(legal_events):
    """Take the assignment record with the latest event date + effective date.

    Return (assignment type text, number of parties that the record names); return None when there is no assignment record.
    """
    try:
        events = (ast.literal_eval(legal_events)
                  if str(legal_events).strip().startswith("[") else [])
    except (ValueError, SyntaxError):
        return None
    segments = []
    for ev in events:
        if not isinstance(ev, dict) or ev.get("code") != "AS":
            continue
        for m in SEGMENT_RE.finditer(ev.get("description", "")):
            segments.append((ev.get("date", ""), m.group("effective") or "",
                             (m.group("conveyance") or "").strip()))
    if not segments:
        return None
    newest = max((s[0], s[1]) for s in segments)
    same = [s for s in segments if (s[0], s[1]) == newest]
    return same[0][2], len(same)


def _conveyance(text: str) -> str:
    upper = text.upper()
    for label, keys in CONVEYANCE_RULES:
        if any(k in upper for k in keys):
            return label
    return "other"


def _norm(s: pd.Series) -> pd.Series:
    """Only to decide whether two names are the same: removes case and non-letter characters; no fuzzy match."""
    return s.astype(str).str.upper().str.replace(r"[^A-Z]", "", regex=True)


# ── Read the released corpus ─────────────────────────────────────────────────
_COLS = ["patent_id", "assignee", "current_assignee",
         "reassignment_assignee", "legal_events"]
_src = pd.read_csv(CLEAN_PATH, usecols=_COLS, low_memory=False).fillna("")

_ids_fp = os.path.join(SAVE_DIR, "released_corpus_ids.csv")
if "result_df" in globals():
    _keep = set(result_df["patent_id"])
elif os.path.exists(_ids_fp):
    _keep = set(pd.read_csv(_ids_fp, dtype=str)["patent_id"])
else:
    raise RuntimeError("Released corpus identifiers not found: run 3.5 first.")

own = _src[_src["patent_id"].isin(_keep)].reset_index(drop=True)
if len(own) != len(_keep):
    raise RuntimeError(f"Released corpus mismatch: source hits {len(own):,}, fingerprint {len(_keep):,}")

with timed("7.1  Grading of the assignment evidence", n_items=len(own)):
    _rec = own["legal_events"].map(_latest_record)
    own["conveyance_type"] = _rec.map(
        lambda r: "no_assignment_record" if r is None else _conveyance(r[0]))
    own["n_parties_in_record"] = _rec.map(lambda r: 0 if r is None else r[1])
    own["assignee_changed"] = (
        (_norm(own["current_assignee"]) != _norm(own["assignee"])) &
        (own["current_assignee"].str.strip() != "") &
        (own["assignee"].str.strip() != ""))
    # Whether a record that "transfers ownership" supports the change of assignee
    own["change_supported"] = (own["assignee_changed"] &
                               own["conveyance_type"].isin(TITLE_TRANSFERRING))
    # When the record names more than one party, the record cannot show which party current_assignee is
    own["party_ambiguous"] = own["assignee_changed"] & (own["n_parties_in_record"] > 1)

    # effective_assignee: falls back to the original assignee when no transfer record supports the change; a verified error is renamed directly.
    own["effective_assignee"] = own["current_assignee"]
    _revert = own["assignee_changed"] & ~own["change_supported"]
    own.loc[_revert, "effective_assignee"] = own.loc[_revert, "assignee"]
    own["effective_assignee"] = own["effective_assignee"].replace(ASSIGNEE_OVERRIDES)

# ── Report ───────────────────────────────────────────────────────────────────
_mv = own[own["assignee_changed"]]
print(f"released corpus              : {len(own):>6,}")
print(f"current_assignee != assignee : {len(_mv):>6,}  ({len(_mv)/len(own)*100:.1f}%)\n")

_TYPE_LABEL = {"true_assignment": "true transfer", "no_assignment_record": "no assignment record",
       "release_of_lien": "release of lien", "security_interest": "unreleased security",
       "corrective": "corrective record", "change_of_name": "change of name", "merger": "merger",
       "partial_transfer": "partial transfer", "other": "other"}
print(f"{'last record type':<22}{'records':>6}{'share':>9}{'multiple':>10}{'transfers':>11}")
print("-" * 60)
for k, v in _mv["conveyance_type"].value_counts().items():
    multi = int((_mv[_mv["conveyance_type"] == k]["n_parties_in_record"] > 1).sum())
    mark = "yes" if k in TITLE_TRANSFERRING else "no"
    print(f"{_TYPE_LABEL.get(k, k):<20}{v:>7,}{v/len(_mv)*100:>8.1f}%{multi:>10}{mark:>11}")

print(f"\nassignee changes supported by a transfer : {int(own['change_supported'].sum()):>5,}"
      f"  ({own['change_supported'].sum()/max(len(_mv), 1)*100:.1f}% of the changes)")
print(f"last record names more than one party    : {int(own['party_ambiguous'].sum()):>5,}"
      f"  ({own['party_ambiguous'].sum()/max(len(_mv), 1)*100:.1f}% of the changes; the record does not show which party is listed)")
print(f"assignee changed after the correction    : "
      f"{int((own['effective_assignee'] != own['current_assignee']).sum()):>5,}"
      f"  ({(own['effective_assignee'] != own['current_assignee']).mean()*100:.1f}% of the whole corpus)")

print("\nTop ten assignees: before vs after the correction")
_a = own["current_assignee"].value_counts().head(10)
_b = own["effective_assignee"].value_counts().head(10)
print(f"  {'current_assignee':<40}{'n':>5}   {'effective_assignee':<40}{'n':>5}")
for _i in range(min(10, len(_a), len(_b))):
    print(f"  {_a.index[_i][:38]:<40}{_a.iloc[_i]:>5}   {_b.index[_i][:38]:<40}{_b.iloc[_i]:>5}")

print("\nCheck of data-source errors:")
for _bad, _real in ASSIGNEE_OVERRIDES.items():
    _r = own[own["current_assignee"] == _bad]
    if not len(_r):
        continue
    print(f"  {_bad} ({len(_r)} records)")
    print(f"    original assignee is always {_real}  : {(_r['assignee'] == _real).all()}")
    print(f"    legal events that mention '{_bad.split()[0]}' : "
          f"{int(_r['legal_events'].str.contains(_bad.split()[0], case=False).sum())}")
    print(f"    times '{_real}' occurs as current_assignee : "
          f"{int((own['current_assignee'] == _real).sum())}")

_out = own[["patent_id", "assignee", "current_assignee", "reassignment_assignee",
            "assignee_changed", "conveyance_type", "n_parties_in_record",
            "change_supported", "party_ambiguous", "effective_assignee"]]
_out.to_csv(os.path.join(SAVE_DIR, "ownership_events.csv"), index=False)
print(f"\nsaved -> ownership_events.csv ({len(_out):,} records)")

In [ ]:
# ── 7.2  Spelling variants of the assignee names (read-only) ────────────────
import os, re
import pandas as pd

_rd = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
print(f"distinct current_assignee in the original: {_rd['current_assignee'].nunique():,}")
print(f"empty values: {(_rd['current_assignee'].str.strip() == '').sum():,}\n")
print("The 30 most frequent names (scan by eye for different spellings of the same company):")
for name, cnt in _rd["current_assignee"].value_counts().head(30).items():
    print(f"  {str(name)[:55]:57} {cnt:>5}")

In [ ]:
# ── 7.3  Current versus original assignee among the candidates (read-only) ──
import pandas as pd, warnings
warnings.filterwarnings("ignore")

_cols = ["patent_id", "current_assignee", "assignee", "reassignment_assignee"]
_src  = pd.read_csv(CLEAN_PATH, usecols=_cols, low_memory=False).fillna("")

# A current assignee that differs from the original assignee means that a transfer happened
_moved = _src[(_src["current_assignee"].str.strip() != "") &
              (_src["assignee"].str.strip() != "") &
              (_src["current_assignee"] != _src["assignee"])]

print(f"patents with a transfer of rights: {len(_moved):,} / {len(_src):,}\n")
print(f"{'current_assignee':42} {'original assignee':38} {'records':>5}")
print("-" * 92)
_pairs = (_moved.groupby(["current_assignee", "assignee"]).size()
          .reset_index(name="n").sort_values("n", ascending=False))
for _, r in _pairs.head(30).iterrows():
    print(f"{str(r['current_assignee'])[:40]:42} {str(r['assignee'])[:36]:38} {r['n']:>5}")

In [ ]:
# ── 7.4  Normalised assignee names (adds assignee_canon and assignee_display) ──
import os, re
import pandas as pd

LEGAL_SUFFIXES = [
    "incorporated", "inc", "corporation", "corp", "company", "co",
    "limited", "ltd", "llc", "llp", "lp", "plc", "gmbh", "ag", "sa", "nv",
    "bv", "kk", "kabushiki kaisha", "oy", "ab", "as", "spa", "srl",
    "pte", "pty", "holdings", "holding",
]

def canon_assignee(name: str) -> str:
    """Normalize to the standard form for comparison: lower case, no punctuation, no legal-entity suffix."""
    s = str(name).strip().lower()
    if not s:
        return ""
    s = s.replace("&", " and ")
    s = re.sub(r"[.,;:()\"']", " ", s)
    s = s.replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()
    changed = True
    while changed:                      # repeated suffixes such as "co ltd" must be removed again and again
        changed = False
        for suf in LEGAL_SUFFIXES:
            if s.endswith(" " + suf):
                s = s[: -(len(suf) + 1)].strip()
                changed = True
    return re.sub(r"\s+", " ", s).strip()

# Manual alias table: write here the names that the rules cannot merge, or the names that need a formal display name
ALIAS = {
    "taiwan semiconductor manufacturing tsmc": "Taiwan Semiconductor Manufacturing (TSMC)",
    "taiwan semiconductor manufacturing":      "Taiwan Semiconductor Manufacturing (TSMC)",
    "international business machines":         "IBM",
    "advanced micro devices":                  "AMD",
    "hewlett packard enterprise development":  "Hewlett Packard Enterprise",
    "sk hynix":                                "SK Hynix",
    "nvidia":                                  "NVIDIA",
    "emc":                                     "EMC",
}

def display_assignee(name: str) -> str:
    c = canon_assignee(name)
    if c in ALIAS:
        return ALIAS[c]
    return " ".join(w.capitalize() for w in c.split()) if c else "(unknown)"

result_path = os.path.join(SAVE_DIR, "cluster_results.csv")
result_df   = pd.read_csv(result_path).fillna("")

result_df["assignee_canon"]   = result_df["current_assignee"].map(canon_assignee)
result_df["assignee_display"] = result_df["current_assignee"].map(display_assignee)

print(f"distinct original names: {result_df['current_assignee'].nunique():,}")
print(f"after normalization    : {result_df['assignee_display'].nunique():,}")
print(f"reduction              : {result_df['current_assignee'].nunique() - result_df['assignee_display'].nunique():,}\n")

print("Merged variants (only the first 25 groups):")
_g = result_df.groupby("assignee_display")["current_assignee"].unique()
_shown = 0
for disp, variants in _g.items():
    if len(variants) > 1:
        print(f"  {disp}")
        for v in variants:
            print(f"      <- {v!r}")
        _shown += 1
        if _shown >= 25:
            break

result_df.to_csv(result_path, index=False)
print("\nsaved -> cluster_results.csv (two columns added; the original columns are not changed)")

### 7.5–7.6  語意地圖上的技術組合

7.5 列出前十五大權利人的主要分類號，以及它們的紀錄落在哪些群。7.6 以互動式 HTML 畫三維地圖：前四大權利人的紀錄，以及依主題著色的全部紀錄。

In [ ]:
# ── 7.5  Portfolio of the largest assignees: class codes and clusters ───────
result_df  = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
cluster_df = result_df[result_df["cluster_id"] != -1]

TOP_N_COMPANIES = 15
top_companies = cluster_df["current_assignee"].value_counts().head(TOP_N_COMPANIES).index.tolist()

for co in top_companies:
    co_rows  = cluster_df[cluster_df["current_assignee"] == co]
    co_total = len(co_rows)
    print(f"\n{co[:60]}  (n={co_total})")
    print(f"  {'IPC':8} {'n':>5} {'share':>7}  clusters")
    for ipc, cnt in co_rows["main_ipc_4"].value_counts().head(5).items():
        cdist = co_rows[co_rows["main_ipc_4"] == ipc]["cluster_id"].value_counts().head(4)
        cstr  = " ".join(f"C{c}({n})" for c, n in cdist.items())
        print(f"  {ipc:8} {cnt:>5} {cnt/co_total*100:>6.1f}%  {cstr}")
    print(f"  spans {co_rows['cluster_id'].nunique()} clusters")

In [ ]:
# ── 7.6  Interactive three-dimensional maps (plotly; also saved as HTML) ──────
import plotly.express as px

_TOPIC_EXCLUDE = ["__noise__", "Off-topic / Non-HPC", "Unclassified"]
top4 = result_df["current_assignee"].value_counts().head(4).index.tolist()
plot_df = result_df[result_df["current_assignee"].isin(top4)]

fig1 = px.scatter_3d(
    plot_df, x="umap_3d_x", y="umap_3d_y", z="umap_3d_z",
    color="current_assignee", color_discrete_sequence=PALETTE,
    hover_data=["patent_id", "title", "main_ipc_4", "merged_topic"],
    opacity=0.85, title="Figure 7.1  Records of the four largest assignees on the three-dimensional UMAP map")
_ = fig1.update_traces(marker=dict(size=2))
_ = fig1.update_layout(height=650, legend_title="assignee", template="plotly_white")
fig1.show()

plot_all = result_df[~result_df["merged_topic"].isin(_TOPIC_EXCLUDE)]
fig2 = px.scatter_3d(
    plot_all, x="umap_3d_x", y="umap_3d_y", z="umap_3d_z",
    color="merged_topic",
    hover_data=["patent_id", "title", "current_assignee", "main_ipc_4"],
    opacity=0.75, title="Figure 7.2  Merged topics on the three-dimensional UMAP map")
_ = fig2.update_traces(marker=dict(size=2))
_ = fig2.update_layout(height=650, legend_title="topic", template="plotly_white")
fig2.show()

fig1.write_html(os.path.join(FIG_DIR, "fig_7_1_umap3d_top4_assignees.html"))
fig2.write_html(os.path.join(FIG_DIR, "fig_7_2_umap3d_topics.html"))
print("saved -> figures/fig_7_1_umap3d_top4_assignees.html, figures/fig_7_2_umap3d_topics.html")

### 7.7–7.10  同一張地圖依分類與依主題著色，以及權利人的技術組合

圖 7.3 把同一個二維 UMAP 投影畫兩次。a 圖依第一個 CPC 代碼的三碼類別著色，b 圖依合併主題著色。兩張圖只有著色依據不同：一個類別拆成好幾個主題，或一個主題橫跨好幾個類別的地方，就是主題層帶有分類沒有的資訊之處。7.16 以數字做同樣的比較。

圖 7.5 到 7.8 比較大型權利人的技術組合：六項組合指標的雷達圖、依群分布比例畫的樹狀圖與熱力圖，以及自組織映射圖。圖例中的權利人類型只替最大的幾家公司人工指定，其餘都歸為「Other」。

In [ ]:
# ── 7.7  Figure 7.3: the same two-dimensional map coloured by class and by topic (CPU only) ──
# One projection, drawn twice: panel a coloured by the three-character class (first three characters of the
# first CPC path, column main_ipc_3), panel b by the merged topic. The coordinates are identical, so the panels
# differ only in the colouring. The UMAP settings are those of 6.4 (n_neighbors=15, min_dist=0.0, cosine, the
# same seed) with two output dimensions. Files: umap2d_coords.csv (cached coordinates; delete it after new
# embeddings), figures/fig_7_3_umap2d_class_vs_topic.png and an interactive three-dimensional map by class.
# Colours: tab20 for the ten classes and twelve topics of the figure in the manuscript; every other record is grey.
from matplotlib.lines import Line2D

TOP_N_IPC3   = 10          # panel a: the ten largest three-character classes; the others are grey
TOP_N_TOPIC  = 12          # panel b: the twelve largest topics; the other topics and the noise are grey
NOISE_LABELS = ["__noise__", "Off-topic / Non-HPC", "Unclassified", ""]
_CMAP = [plt.get_cmap("tab20")(i) for i in range(20)]

_clu = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
_clu["patent_id"] = _clu["patent_id"].astype(str)
if "main_ipc_3" not in _clu.columns:
    _clu = add_ipc3(_clu)

_coord_path = os.path.join(SAVE_DIR, "umap2d_coords.csv")

if os.path.exists(_coord_path):
    _xy = pd.read_csv(_coord_path)
    _xy["patent_id"] = _xy["patent_id"].astype(str)
    print(f"read the existing coordinates -> {_coord_path} ({len(_xy):,} records)")
else:
    # Embedding source: specter2_embeddings.csv of 6.2, otherwise the parquet of 6.13 (both are the synthesized representation)
    _emb, _src = None, None
    for _cand in (os.path.join(SAVE_DIR, "specter2_embeddings.csv"),
                  os.path.join(SAVE_DIR, "embeddings", "embeddings_synthesized.parquet")):
        if os.path.exists(_cand):
            _emb = pd.read_parquet(_cand) if _cand.endswith(".parquet") else pd.read_csv(_cand)
            _src = _cand
            break
    if _emb is None:
        raise RuntimeError("Embedding file not found (specter2_embeddings.csv or "
                           "embeddings/embeddings_synthesized.parquet). Run 6.2 or 6.13 first.")
    _emb["patent_id"] = _emb["patent_id"].astype(str)
    _m = _clu[["patent_id"]].merge(_emb, on="patent_id", how="left")
    _miss = _m.drop(columns=["patent_id"]).isna().all(axis=1)
    if _miss.any():
        raise RuntimeError(f"{int(_miss.sum()):,} records have no embedding; the projection is not possible")
    _X = _m.drop(columns=["patent_id"]).values.astype(np.float32)
    print(f"embedding source -> {_src} | {_X.shape}")

    import umap
    with timed("7.7  UMAP 768d -> 2d (for the figure)", n_items=len(_X)):
        _E2 = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.0,
                        metric="cosine", random_state=RANDOM_STATE,
                        low_memory=False).fit_transform(_X)
    _xy = pd.DataFrame({"patent_id": _m["patent_id"].values,
                        "umap_2d_x": _E2[:, 0], "umap_2d_y": _E2[:, 1]})
    _xy.to_csv(_coord_path, index=False)
    print(f"saved -> {_coord_path}")

_plot = (_clu.merge(_xy, on="patent_id", how="left")
             .dropna(subset=["umap_2d_x", "umap_2d_y"])
             .reset_index(drop=True))
print(f"records in the figure: {len(_plot):,} / {len(_clu):,}")


_ipc3   = _plot["main_ipc_3"].astype(str)
_topic  = _plot["merged_topic"].astype(str)
_n_ipc3 = _ipc3.nunique()
_n_top  = _topic[~_topic.isin(NOISE_LABELS)].nunique()

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6))
for ax, lab, top_n, title, other in (
        (axes[0], _ipc3,  TOP_N_IPC3,  f"a  Three-character CPC class (largest {TOP_N_IPC3} of {_n_ipc3})", "other classes"),
        (axes[1], _topic, TOP_N_TOPIC, f"b  Merged topic (largest {TOP_N_TOPIC} of {_n_top})",            "other or no topic")):
    lab  = lab.where(~lab.isin(NOISE_LABELS), "")
    top  = lab[lab != ""].value_counts().head(top_n).index.tolist()
    grey = ~lab.isin(top)
    _ = ax.scatter(_plot.loc[grey, "umap_2d_x"], _plot.loc[grey, "umap_2d_y"], s=0.6,
                   c=[OTHER_GREY], linewidths=0, rasterized=True)
    handles = []
    for k, t in enumerate(top):
        sel = (lab == t).values
        _ = ax.scatter(_plot.loc[sel, "umap_2d_x"], _plot.loc[sel, "umap_2d_y"], s=0.8,
                       c=[_CMAP[k % 20]], linewidths=0, rasterized=True)
        handles.append(Line2D([0], [0], marker="o", ls="", ms=3, color=_CMAP[k % 20],
                              label=(t if len(t) <= 44 else t[:43] + "…")))
    handles.append(Line2D([0], [0], marker="o", ls="", ms=3, color=OTHER_GREY, label=other))
    _ = ax.set_title(title, fontsize=8)
    _ = ax.set_xticks([]); _ = ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_linewidth(0.4)
    _ = ax.legend(handles=handles, fontsize=4.6, loc="upper center", bbox_to_anchor=(0.5, -0.02),
                  ncol=2, frameon=False, handletextpad=0.2, columnspacing=0.8)
_ = fig.suptitle(f"Figure 7.3  UMAP of the synthesized embeddings ({len(_plot):,} records)", fontsize=9, x=0.01, ha="left")
fig.tight_layout(rect=(0, 0, 1, 0.96))
_ = save_fig(fig, "fig_7_3_umap2d_class_vs_topic")
plt.show()

# ── Record counts of the three-character classes (for the figure caption) ───
_tab3 = (_ipc3.value_counts().rename_axis("main_ipc_3").reset_index(name="n"))
_tab3["share_%"] = (_tab3["n"] / len(_plot) * 100).round(1)
os.makedirs(os.path.join(SAVE_DIR, "tables"), exist_ok=True)
_tab3.to_csv(os.path.join(SAVE_DIR, "tables", "ipc3_counts.csv"), index=False)
print(f"\n{_n_ipc3} three-character classes in total; the first {TOP_N_IPC3} are listed below:")
print(_tab3.head(TOP_N_IPC3).to_string(index=False))

# ── Interactive three-dimensional map by class (coordinates of 6.4, no new projection) ───
import plotly.express as px
_top3 = _tab3.head(TOP_N_IPC3)["main_ipc_3"].tolist()
_p3 = _plot.copy()
_p3["ipc3_group"] = np.where(_p3["main_ipc_3"].isin(_top3), _p3["main_ipc_3"], "Other")
_fig3 = px.scatter_3d(_p3, x="umap_3d_x", y="umap_3d_y", z="umap_3d_z",
                      color="ipc3_group",
                      hover_data=["patent_id", "title", "main_ipc_4", "merged_topic"],
                      opacity=0.75, title="Figure 7.4  Three-character CPC classes on the three-dimensional UMAP map")
_ = _fig3.update_traces(marker=dict(size=2))
_ = _fig3.update_layout(height=650, legend_title="class", template="plotly_white")
_fig3.show()
_fig3.write_html(os.path.join(FIG_DIR, "fig_7_4_umap3d_classes.html"))
print("saved -> figures/fig_7_4_umap3d_classes.html")

In [ ]:
# ── 7.8  Radar chart of the portfolios of the eight largest assignees (plotly; also saved as HTML) ──
# Each axis is scaled to 0-1 across the eight assignees, so the chart compares profiles, not absolute sizes.
import plotly.graph_objects as go

TOP_N = 8
top_cos = result_df["current_assignee"].value_counts().head(TOP_N).index.tolist()

company_stats = []
for co in top_cos:
    co_rows  = result_df[result_df["current_assignee"] == co]
    hpc_rows = co_rows[~co_rows["merged_topic"].isin(["__noise__","Off-topic / Non-HPC","Unclassified",""])]
    company_stats.append({
        "company"       : co[:25],
        "patent_cnt"    : len(co_rows),
        "topic_breadth" : hpc_rows["merged_topic"].nunique(),
        "topic_depth"   : (hpc_rows["merged_topic"].value_counts().iloc[0] / len(hpc_rows) * 100
                           if len(hpc_rows) else 0),
        "hpc_share"    : len(hpc_rows) / len(co_rows) * 100 if len(co_rows) else 0,
        "ipc_diversity" : co_rows["main_ipc_4"].nunique(),
        "cluster_spread": co_rows[co_rows["cluster_id"] != -1]["cluster_id"].nunique(),
    })
stats_df = pd.DataFrame(company_stats)

# Normalize each metric to 0-1
def norm_col(col):
    lo, hi = col.min(), col.max()
    return pd.Series([0.5]*len(col)) if hi == lo else (col - lo) / (hi - lo)

METRICS = ["patent_cnt","topic_breadth","topic_depth","hpc_share",
           "ipc_diversity","cluster_spread"]
for mcol in METRICS:
    stats_df[f"n_{mcol}"] = norm_col(stats_df[mcol])

categories = ["patents", "topic breadth<br>(number of topics)", "topic depth<br>(share of the largest topic)",
              "share with a topic", "class diversity<br>(four-character codes)", "cluster spread"]
colors = PALETTE

fig = go.Figure()
for i, row in stats_df.iterrows():
    vals = [row[f"n_{m}"] for m in METRICS]; vals += [vals[0]]
    cats = categories + [categories[0]]
    _ = fig.add_trace(go.Scatterpolar(
        r=vals, theta=cats, fill="toself", name=row["company"],
        line=dict(color=colors[i % len(colors)], width=2), opacity=0.6))
_ = fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1],
                               tickvals=[0.25, 0.5, 0.75, 1.0],
                               ticktext=["25%","50%","75%","100%"])),
    title="Figure 7.5  Portfolio profiles of the eight largest assignees (each axis scaled to 0-1)",
    template="plotly_white",
    height=600, showlegend=True)
fig.show()

print(f"\n  {'company':28} {'patents':>8} {'breadth':>8} {'HPC %':>8} {'IPC div':>8} {'clusters':>9}")
for _, row in stats_df.iterrows():
    print(f"  {row['company']:28} {row['patent_cnt']:>8} {row['topic_breadth']:>8} "
          f"{row['hpc_share']:>7.1f}% {row['ipc_diversity']:>8} {row['cluster_spread']:>9}")

fig.write_html(os.path.join(FIG_DIR, "fig_7_5_assignee_radar.html"))
print("saved -> figures/fig_7_5_assignee_radar.html")
stats_df[["company"]+METRICS].to_csv(
    os.path.join(SAVE_DIR, "company_radar_stats.csv"), index=False)

In [ ]:
# ── 7.9  Figures 7.6 and 7.7: assignees grouped by their cluster portfolios ──
# Each of the fifteen largest assignees is described by the share of its records in each cluster.
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.patches import Patch

plot_df = cluster_df[cluster_df["current_assignee"].isin(top_companies)].copy()

pivot_df = (plot_df.groupby(["current_assignee", "cluster_id"]).size()
            .reset_index(name="count")
            .pivot(index="current_assignee", columns="cluster_id", values="count")
            .fillna(0))
pivot_ratio = pivot_df.div(pivot_df.sum(axis=1), axis=0)

SHORT_MAP = {
    "Taiwan Semiconductor Manufacturing Co TSMC Ltd": "TSMC",
    "International Business Machines Corp": "IBM",
    "Advanced Micro Devices Inc": "AMD",
    "Micron Technology Inc": "Micron",
    "Nvidia Corp": "Nvidia",
    "Intel Corp": "Intel",
    "Amazon Technologies Inc": "Amazon",
    "Samsung Electronics Co Ltd": "Samsung",
    "SK Hynix Inc": "SK Hynix",
    "Google LLC": "Google",
    "Oracle International Corp": "Oracle",
    "Pure Storage Inc": "Pure Storage",
    "Cerebras Systems Inc": "Cerebras",
    "Psiquantum Corp": "PsiQuantum",
    "EMC Corp": "EMC",
    "Nutanix Inc": "Nutanix",
    "Lodestar Licensing Group LLC": "Lodestar",
    "Everpure Inc": "Everpure",
    "Microsoft Technology Licensing LLC": "Microsoft",
    "Hewlett Packard Enterprise Development LP": "HPE",
    "Saudi Arabian Oil Co": "Saudi Aramco",
}
shorten_company = lambda name: SHORT_MAP.get(name, name[:20])
pivot_ratio.index = [shorten_company(x) for x in pivot_ratio.index]

# Assignee types, set by hand for the largest companies (Everpure is the current name of Pure Storage); others are "Other"
COMPANY_GROUP = {
    "Micron": "Memory / Storage", "SK Hynix": "Memory / Storage",
    "Pure Storage": "Memory / Storage", "Everpure": "Memory / Storage", "EMC": "Memory / Storage",
    "TSMC": "Semiconductor / Manufacturing", "Samsung": "Semiconductor / Manufacturing",
    "Intel": "Compute / Platform", "AMD": "Compute / Platform",
    "IBM": "Compute / Platform", "Nvidia": "Compute / Platform",
    "Amazon": "Cloud / AI Application", "Google": "Cloud / AI Application",
    "Oracle": "Cloud / AI Application", "Nutanix": "Cloud / AI Application",
    "Cerebras": "Specialized Startup", "PsiQuantum": "Specialized Startup",
    "Lodestar": "Specialized Startup",
}
GROUP_COLOR = {
    "Memory / Storage": PALETTE[0], "Semiconductor / Manufacturing": PALETTE[1],
    "Compute / Platform": PALETTE[2], "Cloud / AI Application": PALETTE[3],
    "Specialized Startup": PALETTE[6], "Other": OTHER_GREY,
}

Z = linkage(pivot_ratio.values, method="ward")
fig, ax = plt.subplots(figsize=(9.6, 4.4))
ddata = dendrogram(Z, labels=pivot_ratio.index.tolist(), leaf_rotation=90, leaf_font_size=8,
                   color_threshold=0, above_threshold_color=INK_2, ax=ax)
# a coloured square under each leaf shows the assignee type; the label itself stays in the text colour
_xt = ax.get_xticks()
for _x, _name in zip(_xt, ddata["ivl"]):
    _ = ax.scatter(_x, 0, marker="s", s=36, color=GROUP_COLOR[COMPANY_GROUP.get(_name, "Other")],
                   clip_on=False, zorder=3, edgecolors="white", linewidths=0.6)
_ = ax.legend(handles=[Patch(facecolor=c, label=g) for g, c in GROUP_COLOR.items()],
              title="assignee type", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7, title_fontsize=7)
_ = ax.set_ylabel("Ward distance")
_ = ax.set_title(f"Figure 7.6  The {len(pivot_ratio)} largest assignees grouped by their cluster portfolios")
fig.tight_layout()
_ = save_fig(fig, "fig_7_6_assignee_dendrogram")
plt.show()

ordered = ddata["ivl"]
g = sns.clustermap(
    pivot_ratio.loc[ordered], row_cluster=False, col_cluster=False,
    row_colors=[GROUP_COLOR[COMPANY_GROUP.get(n, "Other")] for n in ordered],
    cmap=SEQ_CMAP, figsize=(10, 5.4), linewidths=0.3, linecolor="white",
    dendrogram_ratio=(0.1, 0.02),   # no dendrograms are drawn here; keep only room for the colour bar
    cbar_pos=(0.02, 0.3, 0.015, 0.35), cbar_kws={"label": "share of records"})
_ = g.ax_heatmap.set_title("Figure 7.7  Share of each assignee's records in each cluster (rows in dendrogram order)",
                           pad=14, loc="left", fontsize=10)
_ = g.ax_heatmap.set_xlabel("cluster_id"); _ = g.ax_heatmap.set_ylabel("")
g.savefig(os.path.join(FIG_DIR, "fig_7_7_assignee_cluster_heatmap.png"))
print("saved -> figures/fig_7_7_assignee_cluster_heatmap.png")
plt.show()

In [ ]:
# ── 7.10  Figure 7.8: self-organising map of the assignee portfolios ─────────
from minisom import MiniSom

X_som = pivot_ratio.values
names = pivot_ratio.index.tolist()

som_x, som_y = 5, 5
som = MiniSom(x=som_x, y=som_y, input_len=X_som.shape[1],
              sigma=1.0, learning_rate=0.5, random_seed=RANDOM_STATE)
som.random_weights_init(X_som)
som.train_random(X_som, 1000)

fig, ax = plt.subplots(figsize=(6.0, 5.4))
_pc = ax.pcolor(som.distance_map().T, cmap=SEQ_CMAP, edgecolors="white", linewidth=0.5)
_ = fig.colorbar(_pc, ax=ax, label="distance to the neighbouring units (U-matrix)")
_cell_n = {}
for i, name in enumerate(names):
    w = som.winner(X_som[i])
    k = _cell_n.get(w, 0); _cell_n[w] = k + 1          # stack the names that fall into the same unit
    _ = ax.scatter(w[0] + 0.12, w[1] + 0.82 - 0.22 * k, marker="s", s=22,
                   color=GROUP_COLOR[COMPANY_GROUP.get(name, "Other")], edgecolors="white", linewidths=0.4)
    _ = ax.text(w[0] + 0.24, w[1] + 0.82 - 0.22 * k, name[:13], va="center", fontsize=6.5, color=INK)
_ = ax.set_xlim(0, som_x); _ = ax.set_ylim(0, som_y)
_ = ax.set_xticks([]); _ = ax.set_yticks([])
_ = ax.legend(handles=[Patch(facecolor=c, label=g) for g, c in GROUP_COLOR.items()],
              loc="upper left", bbox_to_anchor=(1.25, 1), fontsize=7, title="assignee type", title_fontsize=7)
_ = ax.set_title("Figure 7.8  Self-organising map (5 x 5) of the assignee portfolios")
fig.tight_layout()
_ = save_fig(fig, "fig_7_8_assignee_som")
plt.show()

### 7.11–7.12  主題分類器

四種分類器以家族分組切分，從 synthesized 向量預測合併主題；表現最好的模型會存檔，方便使用者把新紀錄歸到既有主題。主題本身就是同一組向量的分群結果，分數高是預期中的事，不能拿來檢核主題；主題的檢核在 7.16。這一格也量測隨機切分會讓分數虛高多少：同一家族的成員若同時落在訓練集與測試集，分數就會偏高。

In [ ]:
# ── 7.11  Topic classifier: predict merged_topic from the synthesized embedding ──
# A convenience tool for users who want to assign new records to the topics. The topics come from clusters
# of the same embeddings, so this is circular and is not evidence for the topic layer (that is 7.16).
# The classifiers were chosen with their run time in mind: HistGradientBoostingClassifier bins the features
# and trained in about 35 s on the full corpus (the exact-split GradientBoostingClassifier took about 85 min);
# the SVM does not set probability=True, and 7.12 ranks by decision_function instead.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import normalize, LabelEncoder
import joblib

embed_df  = pd.read_csv(os.path.join(SAVE_DIR, "specter2_embeddings.csv"))
result_df = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")

hpc_df = result_df[~result_df["merged_topic"].isin(
    ["__noise__", "Off-topic / Non-HPC", "Unclassified"])].reset_index(drop=True)

embed_hpc = hpc_df[["patent_id"]].merge(embed_df, on="patent_id", how="left")
X = normalize(embed_hpc.drop(columns=["patent_id"]).values.astype(np.float32), norm="l2")
y = np.array(hpc_df["merged_topic"].values)

# Remove the topics with < 5 samples (a requirement of the stratified split)
topic_counts = pd.Series(y).value_counts()
mask = pd.Series(y).isin(topic_counts[topic_counts >= 5].index).values
X, y = X[mask], y[mask]
le = LabelEncoder(); y_enc = le.fit_transform(y)

# Family-grouped split (1.6): continuations have nearly identical text, so a random split leaks
hpc_mask_df = hpc_df[mask].reset_index(drop=True)
groups = build_family_groups(hpc_mask_df["patent_id"].values)
idx = np.arange(len(X))
idx_train, idx_test, y_train, y_test = family_train_test_split(idx, y_enc, groups)
X_train, X_test = X[idx_train], X[idx_test]
print(f"n={len(X)} | topics={len(set(y))} | families={pd.Series(groups).nunique()} | "
      f"train={len(X_train)} | test={len(X_test)}")
_miss_tr = len(set(y_enc)) - len(set(y_train)); _miss_te = len(set(y_enc)) - len(set(y_test))
if _miss_tr or _miss_te:
    print(f"  Note: a grouped split cannot stratify; train lacks {_miss_tr} classes, test lacks {_miss_te} classes"
          f" (macro-F1 with zero_division=0)")

# ── Safety checks (read-only; writes no file) ─────────────────────────────
assert hpc_df["patent_id"].is_unique, "patent_id has duplicates; check the upstream files first"
_n_topics = len(set(y))
if _n_topics > 50:
    raise RuntimeError(
        f"merged_topic has {_n_topics} values, so the topic merge of 6.9 has not run.\n"
        "Merge the topics to about 20-30 before you run this section: with too many classes, the training time of the gradient-boosted trees (one set of trees for each class) "
        "and of the SVM (support vector machine; one-vs-one needs N×(N-1)/2 binary classifiers) "
        "grows very quickly with the number of classes."
    )
# ──────────────────────────────────────────────────────────────────────────

models = {
    "Logistic Regression"   : LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"),
    "Random Forest"         : RandomForestClassifier(n_estimators=200, max_depth=20,
                                                     random_state=RANDOM_STATE, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=100, max_depth=5,
                                                             learning_rate=0.1,
                                                             random_state=RANDOM_STATE),
    "SVM (RBF)"             : SVC(kernel="rbf", C=10, gamma="scale", cache_size=1000,
                                  random_state=RANDOM_STATE),
}

results = {}
print(f"\n  {'model':25} {'Acc':>9} {'F1 macro':>10} {'F1 weighted':>12} {'fit+pred':>10}")
for name, clf in models.items():
    with timed(f"7.11  {name}", n_items=len(X_train), quiet=True) as t:
        _ = clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
    results[name] = {
        "clf": clf,
        "acc": accuracy_score(y_test, y_pred),
        "f1_mac": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_wei": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "y_pred": y_pred,
        "sec": t.seconds,
    }
    r = results[name]
    print(f"  {name:25} {r['acc']*100:>8.1f}% {r['f1_mac']:>10.4f} "
          f"{r['f1_wei']:>12.4f} {r['sec']:>9.1f}s")

best_name = max(results, key=lambda k: results[k]["f1_mac"])
best = results[best_name]
print(f"\nbest model: {best_name} (F1 macro = {best['f1_mac']:.4f})")
print(classification_report(y_test, best["y_pred"], target_names=le.classes_,
                            zero_division=0))

from sklearn.model_selection import GroupKFold
with timed(f"7.11  5-fold GroupKFold CV ({best_name})", n_items=len(X)):
    cv_scores = cross_val_score(best["clf"], X, y_enc,
                                cv=GroupKFold(n_splits=5).split(X, y_enc, groups),
                                scoring="f1_macro", n_jobs=-1)
print(f"5-fold GroupKFold CV F1 macro: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}"
      f" (families as the fold unit)")

model_path = os.path.join(SAVE_DIR, f"classifier_{best_name.replace(' ', '_').lower()}.pkl")
_ = joblib.dump({"model": best["clf"], "label_encoder": le}, model_path)
print(f"saved -> {model_path}")

# The training cost of each classifier is saved with its scores
pd.DataFrame([{"model": k, "acc": round(v["acc"], 4),
               "f1_macro": round(v["f1_mac"], 4), "f1_weighted": round(v["f1_wei"], 4),
               "fit_predict_sec": round(v["sec"], 2), "hardware": HARDWARE}
              for k, v in results.items()]).to_csv(
    os.path.join(SAVE_DIR, "topic_classifier_benchmark.csv"), index=False)
print("saved -> topic_classifier_benchmark.csv")

# ── Leakage check: random split vs family-grouped split (the same SVM) ─────
# The difference between the two scores is the optimistic bias that the near-duplicate texts of family
# continuations cause under a random split.
with timed("7.11  Leakage check (random vs family split, SVM)", n_items=len(X)):
    _tr_r, _te_r, _ytr_r, _yte_r = train_test_split(
        np.arange(len(X)), y_enc, test_size=0.2,
        random_state=RANDOM_STATE, stratify=y_enc)
    _clf_r  = SVC(kernel="rbf", C=10, gamma="scale", cache_size=1000,
                  random_state=RANDOM_STATE).fit(X[_tr_r], _ytr_r)
    _f1_rand = f1_score(_yte_r, _clf_r.predict(X[_te_r]),
                        average="macro", zero_division=0)
print(f"SVM macro-F1 with random split {_f1_rand:.4f}  vs  with family-grouped split "
      f"{results['SVM (RBF)']['f1_mac']:.4f} (difference = optimistic bias from family leakage)")

In [ ]:
# ── 7.12  Batch prediction example and accuracy by assignee and class ───────
def predict_batch(df, top_k=3):
    pred_embed = df[["patent_id"]].merge(embed_df, on="patent_id", how="left")
    missing = pred_embed.drop(columns=["patent_id"]).isna().all(axis=1)
    if missing.any():
        raise ValueError(f"missing embeddings for {missing.sum()} patent_ids")
    X_pred = normalize(pred_embed.drop(columns=["patent_id"]).values.astype(np.float32),
                       norm="l2")
    clf = best["clf"]
    y_pred = le.inverse_transform(clf.predict(X_pred))
    # top-k: use probabilities when they are available; to save about 5.7 times the training time, the SVM does not set
    # probability=True and sorts by the one-vs-rest scores of decision_function instead;
    # the ranking is the same as with probabilities (only no calibrated probability values are output).
    scores = None
    if hasattr(clf, "predict_proba"):
        scores = clf.predict_proba(X_pred)
    elif hasattr(clf, "decision_function"):
        d = clf.decision_function(X_pred)
        scores = d if d.ndim == 2 else np.column_stack([-d, d])
    if scores is None:
        return y_pred, [[p] for p in y_pred]
    top_idx = np.argsort(scores, axis=1)[:, ::-1][:, :top_k]
    top_k_labels = [list(le.inverse_transform(r)) for r in top_idx]
    return y_pred, top_k_labels

INFER_N   = min(700, len(hpc_mask_df))
sample_df = hpc_mask_df.sample(INFER_N, random_state=RANDOM_STATE).reset_index(drop=True)
with timed(f"7.12  Batch inference example ({best_name})", n_items=INFER_N):
    preds, top3s = predict_batch(sample_df)
sample_df["predicted_topic"] = preds
sample_df["top3"]    = top3s
sample_df["correct"] = sample_df["predicted_topic"] == sample_df["merged_topic"]

acc = sample_df["correct"].mean()
print(f"sample accuracy: {acc*100:.1f}%  ({sample_df['correct'].sum()}/{INFER_N})")

print("\nper-company accuracy (top 10 by volume):")
co_acc = sample_df.groupby("current_assignee")["correct"].agg(total="count", correct="sum")
co_acc["accuracy"] = (co_acc["correct"] / co_acc["total"] * 100).round(1)
print(co_acc.sort_values("total", ascending=False).head(10).to_string())

print("\nper-IPC accuracy (top 10 by volume):")
ipc_acc = sample_df.groupby("main_ipc_4")["correct"].agg(total="count", correct="sum")
ipc_acc["accuracy"] = (ipc_acc["correct"] / ipc_acc["total"] * 100).round(1)
print(ipc_acc.sort_values("total", ascending=False).head(10).to_string())

sample_df[["patent_id","current_assignee","main_ipc_4","merged_topic",
           "predicted_topic","top3","correct"]].to_csv(
    os.path.join(SAVE_DIR, "inference_results.csv"), index=False)

cross = (sample_df.groupby(["current_assignee","predicted_topic","main_ipc_4"])
         .size().reset_index(name="count"))
cross.to_csv(os.path.join(SAVE_DIR, "cross_analysis.csv"), index=False)
print("\nsaved -> inference_results.csv, cross_analysis.csv")

### 7.13–7.15  使用示範

三個簡短的例子示範新增欄位的用法：涉訟家族落在哪些主題、各主題在不同申請年的比例，以及語料內部主題之間的引用。都是描述性的，不做檢定，也不做因果解讀。

In [ ]:
# ── 7.13  Usage example: topics of the litigated families ───────────────────
# Reads litigation_family.csv of 3.8. Descriptive only: the topics of the litigated families and their
# citation counts next to the whole corpus. The subset is small, so there is no test and no causal reading.
# citation_count is not in m1 (3.7), so it is read from the candidate file.
_lit_path = os.path.join(SAVE_DIR, "litigation_family.csv")
if not os.path.exists(_lit_path):
    raise RuntimeError("litigation_family.csv not found; run 3.8 first.")
lit_df = pd.read_csv(_lit_path).fillna("")
_clu   = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
_meta  = pd.read_csv(CLEAN_PATH, usecols=["patent_id", "citation_count"]).fillna(0)

with timed("7.13  Litigated-family landscape", n_items=len(lit_df)):
    lit_ids  = set(lit_df["patent_id"])
    _n_seed  = int((lit_df["is_seed"].astype(str).str.lower() == "true").sum())
    _dfm = _clu[["patent_id", "merged_topic"]].merge(_meta, on="patent_id", how="left")
    _dfm = _dfm[~_dfm["merged_topic"].isin(["__noise__", "Off-topic / Non-HPC", "Unclassified", ""])]
    _dfm["litigated"] = _dfm["patent_id"].isin(lit_ids)
    n_lit = int(_dfm["litigated"].sum())

    print(f"Litigated-family subset: {len(lit_df)} records (seed {_n_seed} records, "
          f"{lit_df['family_root'].nunique()} families, "
          f"{lit_df['case_group'].nunique()} case groups); {n_lit} of them have a topic label")

    topic_tab = (_dfm.groupby("merged_topic")["litigated"]
                 .agg(n_patents="size", n_litigated="sum"))
    topic_tab["litigated_pct"] = (topic_tab["n_litigated"] / topic_tab["n_patents"] * 100).round(2)
    topic_tab = topic_tab.sort_values("n_litigated", ascending=False)
    _hit = topic_tab[topic_tab["n_litigated"] > 0]
    print(f"\nThe litigated families fall in {len(_hit)} topics (only topics with litigation are listed; "
          f"at this sample size the percentages only show the position and must not be read as risk rates):")
    print(_hit.to_string())

    cc = pd.to_numeric(_dfm["citation_count"], errors="coerce").fillna(0)
    print(f"\ncitation_count median: litigated families {cc[_dfm['litigated']].median():.0f}"
          f" (n={n_lit}) vs all {cc.median():.0f} (n={len(_dfm)})"
          f"  — descriptive comparison; the sample is too small for a statistical test; no causal claim")

topic_tab.reset_index().to_csv(os.path.join(SAVE_DIR, "usage_litigation_topics.csv"),
                               index=False)
print("saved -> usage_litigation_topics.csv")

In [ ]:
# ── 7.14  Usage example: topic shares by application year (Figure 7.9) ───────
# Descriptive only. The corpus holds granted patents, and many recent applications are still under
# examination, so the drop after 2022 is the grant delay, not a change in technology. Any yearly chart of
# this corpus needs this right-truncation note.
_pipe = pd.read_csv(os.path.join(SAVE_DIR, "patent_full_pipeline.csv"),
                    usecols=["patent_id", "application_year"]).fillna("")
_clu  = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
_dfm  = _pipe.merge(_clu[["patent_id", "merged_topic"]], on="patent_id", how="inner")
_dfm  = _dfm[~_dfm["merged_topic"].isin(["__noise__", "Off-topic / Non-HPC", "Unclassified", ""])]
_dfm["application_year"] = pd.to_numeric(_dfm["application_year"], errors="coerce")

with timed("7.14  Topic trends by filing year", n_items=len(_dfm)):
    top8 = _dfm["merged_topic"].value_counts().head(8).index
    tab = (_dfm[_dfm["merged_topic"].isin(top8)]
           .pivot_table(index="application_year", columns="merged_topic",
                        values="patent_id", aggfunc="count").fillna(0))
    share = tab.div(tab.sum(axis=1), axis=0) * 100              # columns in alphabetical order (as written to the CSV)
_plot_share = share[[t for t in top8 if t in share.columns]]    # figure: fixed colour order, largest topic first

fig, ax = plt.subplots(figsize=(8.4, 3.6))
_ = ax.stackplot(_plot_share.index, [_plot_share[c].values for c in _plot_share.columns], labels=_plot_share.columns,
                 colors=PALETTE[:len(_plot_share.columns)], edgecolor="white", linewidth=0.6)
_ = ax.axvspan(2022.5, share.index.max() + 0.5, color="white", alpha=0.45, zorder=3, linewidth=0)
_ = ax.axvline(2022.5, color=INK_2, linestyle="--", linewidth=0.8, zorder=4)
_ = ax.text(2022.6, 101, "right-truncated: granted patents only", fontsize=7, color=INK_2, va="bottom")
_ = ax.set_xlim(share.index.min(), share.index.max())
_ = ax.set_ylim(0, 100)
_ = ax.set_ylabel("share of the eight largest topics (percent)")
_ = ax.set_xlabel("application year")
_ = ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
_ = ax.set_title("Figure 7.9  Shares of the eight largest topics by application year", pad=14)
fig.tight_layout()
_ = save_fig(fig, "fig_7_9_topic_trends")
plt.show()
share.round(2).to_csv(os.path.join(SAVE_DIR, "usage_topic_trends.csv"))
print("saved -> usage_topic_trends.csv")

In [ ]:
# ── 7.15  Usage example: citations between topics inside the corpus ─────────
# The citations column is parsed, and an edge is kept when the cited patent is also a released record; the
# edges are then counted by citing and cited topic. citations is read from the candidate file and limited to
# released_corpus_ids.csv (3.5), so "inside the corpus" means the released records.
import ast as _ast

_rel  = set(pd.read_csv(os.path.join(SAVE_DIR, "released_corpus_ids.csv"),
                        dtype=str)["patent_id"])
_pipe = pd.read_csv(CLEAN_PATH, usecols=["patent_id", "citations"], dtype=str).fillna("")
_pipe = _pipe[_pipe["patent_id"].isin(_rel)].reset_index(drop=True)
_clu  = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
_topic = dict(zip(_clu["patent_id"], _clu["merged_topic"]))

with timed("7.15  Citation flows between topics", n_items=len(_pipe)):
    in_corpus = set(_pipe["patent_id"])
    edges = []
    for pid, cit in tqdm(zip(_pipe["patent_id"], _pipe["citations"]),
                         total=len(_pipe), desc="parse citations"):
        t_src = _topic.get(pid, "")
        if not t_src or t_src in ("__noise__", "Off-topic / Non-HPC", "Unclassified"):
            continue
        try:
            items = _ast.literal_eval(cit) if cit.startswith("[") else []
        except (ValueError, SyntaxError):
            continue
        for it in items:
            tgt = str(it.get("publication_number", ""))
            if tgt in in_corpus:
                t_tgt = _topic.get(tgt, "")
                if t_tgt and t_tgt not in ("__noise__", "Off-topic / Non-HPC", "Unclassified"):
                    edges.append((t_src, t_tgt))
    print(f"citation edges within the corpus: {len(edges)} (corpus scope: released corpus, {len(in_corpus):,} records)")
    flow = pd.DataFrame(edges, columns=["citing_topic", "cited_topic"])
    mat = pd.crosstab(flow["citing_topic"], flow["cited_topic"])
    diag = sum(mat.loc[t, t] for t in mat.index if t in mat.columns)
    print(f"share of citations within the same topic: {diag / max(len(edges), 1) * 100:.1f}%")
    off = flow[flow["citing_topic"] != flow["cited_topic"]]
    print("\nTop 10 citation flows between topics:")
    print(off.value_counts().head(10).to_string())
mat.to_csv(os.path.join(SAVE_DIR, "usage_citation_flows.csv"))
print("saved -> usage_citation_flows.csv")

### 7.16  主題與專利分類的一致性

主題建構過程完全沒有使用分類號，所以主題與分類的一致程度可以當作主題層的外部檢核。兩個極端都有意義：完全不一致，表示主題可能反映的是模型而不是技術；完全一致，表示主題沒有提供分類以外的資訊。這一格報告標準化互資訊、調整後蘭德指數、同質性與完整性，打亂主題標籤後的同一組指標，以及每個主題最大類別的占比和全語料最大類別占比的比較。

In [ ]:
# ── 7.16  Agreement of the topics with the patent classification (CPC) ────────
# The topics of section 6 come from clusters of the summary embeddings, and their construction uses no
# classification code (the class columns come from the first CPC path on Google Patents; the column names keep
# "ipc"). The agreement of the two is therefore an external check of whether the topics follow a technical
# structure:
#   no relation at all → the topics can be only a product of the model;
#   complete agreement → the semantic layer gives no information beyond the administrative classification.
# Both directions have a meaning, so this section only reports the measured values and sets no pass threshold.
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                            homogeneity_score, completeness_score)

EXCLUDE_TOPICS = ["__noise__", "Off-topic / Non-HPC", "Unclassified", ""]
TBL_DIR = os.path.join(SAVE_DIR, "tables")
os.makedirs(TBL_DIR, exist_ok=True)

# ── Build the data: topic labels × four-character patent class (CPC) ────────────
_clu = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
if "main_ipc_4" in _clu.columns:
    _dfm = _clu[["patent_id", "cluster_id", "merged_topic", "main_ipc_4"]].copy()
else:
    _ipc = pd.read_csv(CLEAN_PATH, usecols=["patent_id", "main_ipc_4"],
                       low_memory=False).fillna("")
    _dfm = _clu[["patent_id", "cluster_id", "merged_topic"]].merge(
        _ipc, on="patent_id", how="inner")

_n_all = len(_dfm)
_dfm = _dfm[~_dfm["merged_topic"].isin(EXCLUDE_TOPICS)]
_dfm = _dfm[_dfm["main_ipc_4"].astype(str).str.strip() != ""].reset_index(drop=True)

print(f"clustering result           : {_n_all:,} records")
print(f"with topic and class        : {len(_dfm):,} records"
      f" ({_n_all - len(_dfm):,} records excluded: noise, no topic or no class)")
print(f"topic coverage              : {len(_dfm)/_n_all*100:.1f}%"
      f"  <- records without a topic label are not in any metric below")
print(f"distinct topics             : {_dfm['merged_topic'].nunique():,}")
_dfm = add_ipc3(_dfm)
print(f"distinct IPC four-character : {_dfm['main_ipc_4'].nunique():,}")
print(f"distinct IPC three-character: {_dfm['main_ipc_3'].nunique():,}"
      f"  <- closer to the granularity of the {_dfm['merged_topic'].nunique():,} topics")

# Agreement at the three-character scale: the numbers of topics and of three-character classes have a similar order of magnitude; this is the fairer external check
_t3 = _dfm["merged_topic"].astype("category").cat.codes.to_numpy()
_i3 = _dfm["main_ipc_3"].astype("category").cat.codes.to_numpy()
print(f"\nThree-character scale: NMI {normalized_mutual_info_score(_t3, _i3):.4f}"
      f" | ARI {adjusted_rand_score(_t3, _i3):.4f}"
      f" | homogeneity {homogeneity_score(_i3, _t3):.4f}"
      f" | completeness {completeness_score(_i3, _t3):.4f}")
print("(the four-character scale below is the fine-grained comparison)")

with timed("7.16  Agreement of topics and classification", n_items=len(_dfm)):
    _t = _dfm["merged_topic"].astype("category").cat.codes.to_numpy()
    _i = _dfm["main_ipc_4"].astype("category").cat.codes.to_numpy()

    # ── Table 1  Overall agreement metrics ───────────────────────────────────
    _metrics = [
        ("normalized mutual information (NMI)",  normalized_mutual_info_score(_t, _i),
         "0 = independent partitions; 1 = identical partitions"),
        ("adjusted Rand index (ARI)", adjusted_rand_score(_t, _i),
         "corrected for chance; 0 = no better than random"),
        ("homogeneity", homogeneity_score(_i, _t),
         "whether each topic holds a single class only"),
        ("completeness", completeness_score(_i, _t),
         "whether each class falls in a single topic only"),
    ]
    # Random baseline: compute again after the topic labels are shuffled, as a comparison
    _rng = np.random.default_rng(RANDOM_STATE)
    _shuf = _rng.permutation(_t)
    _base_nmi = normalized_mutual_info_score(_shuf, _i)
    _base_ari = adjusted_rand_score(_shuf, _i)

    t_metric = pd.DataFrame([(k, round(v, 4), d) for k, v, d in _metrics],
                            columns=["metric", "value", "note"])

    # ── Table 2  Class concentration of each topic ───────────────────────────
    _rows = []
    for topic, grp in _dfm.groupby("merged_topic"):
        vc = grp["main_ipc_4"].value_counts()
        _rows.append([topic, len(grp), vc.index[0], round(vc.iloc[0] / len(grp) * 100, 1),
                      grp["main_ipc_4"].nunique()])
    t_topic = (pd.DataFrame(_rows, columns=["topic", "records", "largest_cpc_subclass", "largest_cpc_subclass_share_pct",
                                           "distinct_cpc_subclasses"])
               .sort_values("records", ascending=False).reset_index(drop=True))

    # ── Table 3  Number of topics over which each main class spreads ─────────
    _rows = []
    for ipc, grp in _dfm.groupby("main_ipc_4"):
        if len(grp) < 30:                      # a class that is too small is not representative
            continue
        vc = grp["merged_topic"].value_counts()
        _rows.append([ipc, len(grp), vc.index[0], round(vc.iloc[0] / len(grp) * 100, 1),
                      grp["merged_topic"].nunique()])
    t_ipc = (pd.DataFrame(_rows, columns=["cpc_subclass", "records", "largest_topic", "largest_topic_share_pct",
                                          "distinct_topics"])
             .sort_values("records", ascending=False).reset_index(drop=True))

# ── Report ───────────────────────────────────────────────────────────────────
print("\nTable 1  Overall agreement metrics")
print(t_metric.to_string(index=False))
print(f"\n  random baseline (shuffled topic labels): NMI {_base_nmi:.4f} and ARI {_base_ari:.4f}")
print(f"  actual value above the random baseline : NMI +{_metrics[0][1] - _base_nmi:.4f} and "
      f"ARI +{_metrics[1][1] - _base_ari:.4f}")

_dom = t_topic["largest_cpc_subclass_share_pct"]
print(f"\nTable 2  Class concentration of each topic ({len(t_topic)} topics)")
print(f"  share of the largest class: median {_dom.median():.1f}%, "
      f"quartiles {_dom.quantile(.25):.1f}%–{_dom.quantile(.75):.1f}%")
print(f"  topics with a single-class share ≥ 50%: {int((_dom >= 50).sum())}"
      f" ({(_dom >= 50).mean()*100:.0f}%)")
print(f"  mean number of classes covered: {t_topic['distinct_cpc_subclasses'].mean():.1f}")
print("\n  The 15 topics with the most records:")
print(t_topic.head(15).to_string(index=False))

print(f"\nTable 3  Topic dispersion of the main classes (classes with ≥ 30 records: {len(t_ipc)})")
print(f"  median share of the largest topic: {t_ipc['largest_topic_share_pct'].median():.1f}%")
print(f"  spread over {t_ipc['distinct_topics'].mean():.1f} topics on average")
print(t_ipc.head(15).to_string(index=False))

t_metric.to_csv(os.path.join(TBL_DIR, "e6_topic_ipc_metrics.csv"), index=False)
t_topic.to_csv(os.path.join(TBL_DIR, "e6_topic_ipc_concentration.csv"), index=False)
t_ipc.to_csv(os.path.join(TBL_DIR, "e6_ipc_topic_dispersion.csv"), index=False)
print(f"\nsaved -> tables/e6_topic_ipc_metrics.csv, "
      f"e6_topic_ipc_concentration.csv, e6_ipc_topic_dispersion.csv")

# ── Table 4  Largest-class share of each topic and its distribution ────────
# Definition:
#   largest-class share of a topic = records of the most frequent class in the topic ÷ all records of the topic.
#   overall weighted share         = sum of the largest-class records of all topics ÷ all records.
# The reference point for the reading is the random baseline, and the baseline is not a number that someone selected: it is equal to "the share of the largest class in the whole
# corpus". This means: if topics are assigned at random without any look at the content, the most frequent class of each topic already
# has this share. Thus
#   share > baseline → the topic is more concentrated than a random assignment, and the administrative classification captures it;
#   share < baseline → the topic is more spread than a random assignment, and the administrative classification does not capture it.
# Both directions have a meaning (too low means that the topics are unrelated to the administrative classification; too high means that the semantic layer gives no information
# beyond the classification), so only the values and the distribution are reported, with no pass threshold.

def ipc_top_share(frame, ipc_col, topic_col="merged_topic"):
    _g   = frame.groupby(topic_col)[ipc_col]
    _per = pd.DataFrame({
        "n":     _g.size(),
        "top":   _g.agg(lambda s: s.value_counts().index[0]),
        "top_n": _g.agg(lambda s: int(s.value_counts().iloc[0])),
        "n_ipc": _g.nunique(),
    })
    _per["top_share"] = _per["top_n"] / _per["n"]
    return float(_per["top_n"].sum() / _per["n"].sum()), _per.sort_values("n", ascending=False)

_rng2  = np.random.default_rng(RANDOM_STATE)
_share_tables = {}
print("\n" + "=" * 70)
print("Table 4  Largest-class share of each topic")
print("=" * 70)
for _col, _name in (("main_ipc_3", "three-character classes"), ("main_ipc_4", "four-character subclasses")):
    if _col not in _dfm.columns:
        continue
    _ov, _per = ipc_top_share(_dfm, _col)
    # Random baseline: compute again after the topic labels are shuffled. Its value is equal to the share of the largest class in the whole corpus,
    # that is, the share that the most frequent class of each topic already has when topics are assigned at random "without any look at the content".
    _shufd = _dfm.assign(merged_topic=_rng2.permutation(_dfm["merged_topic"].values))
    _ov_r, _ = ipc_top_share(_shufd, _col)
    _p    = _per["top_share"]
    _big  = _dfm[_col].value_counts()
    print(f"\n{_name} ({_dfm[_col].nunique()} classes, {len(_per)} topics)")
    print(f"  largest class         : {_big.index[0]}  {_big.iloc[0]:,} records / {len(_dfm):,} records"
          f" = {_big.iloc[0]/len(_dfm):.4f}  ← this number is the random baseline")
    print(f"  overall weighted share: {_ov:.4f}   random baseline {_ov_r:.4f}   above by {_ov - _ov_r:+.4f}")
    print(f"  share distribution    : min {_p.min():.3f} | 25th percentile {_p.quantile(.25):.3f} | "
          f"median {_p.median():.3f} | 75th percentile {_p.quantile(.75):.3f} | max {_p.max():.3f}")
    _bins = [0, .2, .4, .6, .8, 1.0001]
    _lab  = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
    _hist = pd.cut(_p, bins=_bins, labels=_lab, right=False).value_counts().reindex(_lab).fillna(0)
    print("  histogram:")
    for _b, _n in _hist.items():
        print(f"    {_b}  {int(_n):>3}  {'█' * int(_n)}")
    _n_lo = int((_p < _ov_r).sum())
    print(f"  topics below baseline : {_n_lo} / {len(_per)}"
          f" (covering {int(_per.loc[_p < _ov_r, 'n'].sum()):,} records, "
          f"{_per.loc[_p < _ov_r, 'n'].sum() / len(_dfm) * 100:.1f}%)"
          f" ← the administrative classification does not capture these topics")
    print(f"  the 5 topics with the highest share (the administrative classification captures them; a scan by class code is sufficient):")
    for _r in _per.sort_values('top_share', ascending=False).head(5).itertuples():
        print(f"    {str(_r.Index)[:42]:44} n={_r.n:>4}  {_r.top_share:.3f}  ({_r.top})")
    print(f"  the 5 topics with the lowest share (those below the baseline are not captured by the administrative classification; see the line above):")
    for _r in _per.sort_values('top_share').head(5).itertuples():
        print(f"    {str(_r.Index)[:42]:44} n={_r.n:>4}  {_r.top_share:.3f}  ({_r.top}; covers {_r.n_ipc} classes)")
    _per.reset_index().to_csv(os.path.join(TBL_DIR, f"e6_top_share_{_col}.csv"), index=False)
    _share_tables[_col] = {"overall": round(_ov, 4), "random_baseline": round(_ov_r, 4),
                         "n_topics": int(len(_per)), "n_ipc": int(_dfm[_col].nunique())}

pd.DataFrame(_share_tables).T.reset_index().rename(columns={"index": "ipc_level"}).to_csv(
    os.path.join(TBL_DIR, "e6_top_share_summary.csv"), index=False)
print("\nsaved -> tables/e6_top_share_main_ipc_3.csv, e6_top_share_main_ipc_4.csv,\n         e6_top_share_summary.csv")

### 7.17  同一份技術地景盤點，用兩種做法各做一次

「誰持有哪些技術」的盤點做兩次：一次用分析師通常會用的欄位（`current_assignee` 與分類號），一次用本資料集的欄位（`effective_assignee` 與主題）。一個主題若其最大類別的占比，低於全語料最大類別的占比，表示它在分類上比隨機分配還分散，用分類號盤點分辨不出它。與這個基準相差 0.03 以內的主題標為臨界。

In [ ]:
# ── 7.17  Usage example: one landscape scan done in two ways (CPU only) ─────
# The same landscape scan is done twice, with only two columns changed, and then the differences are shown:
#   Approach A (current practice)  current_assignee   + patent classification (CPC)
#   Approach B (this data)         effective_assignee + semantic topics
# This section only reports what each approach gives. It evaluates no company and makes no causal claim.
#
# The threshold of the screening rule is not selected by people. It is the random baseline of this corpus: the share of the largest single class in the whole corpus.
# When a topic has "a largest-class share < random baseline", it is more spread than a random assignment,
# and the administrative classification cannot resolve it; otherwise a scan by class code is sufficient, and the semantic layer is not necessary.
import os
import numpy as np, pandas as pd

TOP_N_CO    = 5     # number of companies in the company × topic matrix
TOP_HOLDERS = 3     # number of holders listed for each topic
TOP_N_RANK  = 10    # rows of the assignee ranking comparison
NON_TOPIC   = ["__noise__", "Off-topic / Non-HPC", "Unclassified", ""]

TBL_DIR = os.path.join(SAVE_DIR, "tables"); os.makedirs(TBL_DIR, exist_ok=True)

_clu = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
_clu["patent_id"] = _clu["patent_id"].astype(str)
if "main_ipc_3" not in _clu.columns:
    _clu = add_ipc3(_clu)

_own_path = os.path.join(SAVE_DIR, "ownership_events.csv")
if not os.path.exists(_own_path):
    raise RuntimeError("ownership_events.csv not found; run 7.1 first.")
_own = pd.read_csv(_own_path).fillna("")
_own["patent_id"] = _own["patent_id"].astype(str)
_df = _clu.merge(_own[["patent_id", "effective_assignee", "change_supported"]],
                 on="patent_id", how="left")
_df["effective_assignee"] = (_df["effective_assignee"].fillna("").astype(str).str.strip()
                             .where(lambda s: s != "", _df["current_assignee"]))

# ── Table S1  Top ten assignees: before vs after the correction ─────────────
_cur = _df["current_assignee"].astype(str).str.strip()
_eff = _df["effective_assignee"].astype(str).str.strip()
_a = _cur[_cur != ""].value_counts().head(TOP_N_RANK)
_b = _eff[_eff != ""].value_counts().head(TOP_N_RANK)
_K = min(TOP_N_RANK, len(_a), len(_b))          # no failure when there are fewer distinct assignees than TOP_N_RANK
_a, _b = _a.head(_K), _b.head(_K)
_S1 = pd.DataFrame({
    "rank": range(1, _K + 1),
    "current_assignee": _a.index, "n_before": _a.values,
    "effective_assignee": _b.index, "n_after": _b.values})
_S1["changed"] = np.where(
    _S1["current_assignee"] != _S1["effective_assignee"], "yes", "")
print("Table S1  Top ten assignees: before vs after the correction")
print(_S1.to_string(index=False))
_gone = [x for x in _a.index if x not in set(_b.index)]
_new  = [x for x in _b.index if x not in set(_a.index)]
if _gone or _new:
    print(f"  dropped out of the top {_K} ranks: {_gone}")
    print(f"  entered the top {_K} ranks: {_new}")
_n_changed = int((_cur != _eff).sum())
print(f"\n  patents with a changed assignee: {_n_changed:,} ({_n_changed/len(_df)*100:.1f}% of the whole corpus)")
(_S1.rename(columns={"rank": "rank", "changed": "changed"})
    .replace({"changed": {"yes": "yes", "": "no"}})
    .to_csv(os.path.join(SAVE_DIR, "usage_assignee_before_after.csv"), index=False))

# ── Topic subset ────────────────────────────────────────────────────────────
_t = _df[~_df["merged_topic"].isin(NON_TOPIC)].copy()
_t = _t[_t["main_ipc_3"].astype(str).str.strip() != ""].reset_index(drop=True)
_N = len(_t)
print(f"\nwith topic and class: {_N:,} records (coverage {_N/len(_df)*100:.1f}%) | "
      f"{_t['merged_topic'].nunique()} topics")

# ── Random baseline = share of the largest single class in the whole corpus (equal to the baseline of Table 4 in 7.16) ───
_BASE = {}
for _lvl in ("main_ipc_3", "main_ipc_4"):
    _BASE[_lvl] = _t[_lvl].value_counts().iloc[0] / _N
print(f"random baseline (share of the largest single class in the whole corpus): "
      f"three-character {_BASE['main_ipc_3']:.4f} | four-character {_BASE['main_ipc_4']:.4f}")
print("  (this value is the screening threshold; it equals the random baseline of Table 4 in 7.16)")

# ── Table S2  Topic screening table ─────────────────────────────────────────
_rows = []
for _tp, _g in _t.groupby("merged_topic"):
    _r = {"topic": _tp, "records": len(_g)}
    for _lvl, _tag in (("main_ipc_3", "three-character"), ("main_ipc_4", "four-character")):
        _vc = _g[_lvl].value_counts()
        _r[f"largest {_tag} class"] = _vc.index[0]
        _r[f"{_tag} share"]    = round(_vc.iloc[0] / len(_g), 3)
        _r[f"{_tag} classes"]  = int(_vc.size)
    _hold = _g["effective_assignee"].astype(str).str.strip()
    _hold = _hold[_hold != ""].value_counts().head(TOP_HOLDERS)
    _r["top three holders"] = "; ".join(f"{k[:26]} {v}" for k, v in _hold.items())
    _r["largest holder share"] = round(_hold.iloc[0] / len(_g), 3) if len(_hold) else np.nan
    _rows.append(_r)

# The threshold is a reference point, not a dichotomy: also output the "distance to the baseline" and a borderline mark,
# so that users see the whole ranking and not only the cut point. A topic within MARGIN is borderline and is not a firm conclusion.
MARGIN = 0.03
_S2 = pd.DataFrame(_rows).sort_values("three-character share").reset_index(drop=True)
_S2["distance to baseline"] = (_S2["three-character share"] - _BASE["main_ipc_3"]).round(3)
_S2["not resolved by administrative classification"] = np.where(_S2["distance to baseline"] < 0, "yes", "")
_S2.loc[_S2["distance to baseline"].abs() <= MARGIN, "not resolved by administrative classification"] += " (borderline)"
_S2 = _S2[["topic", "records", "largest three-character class", "three-character share", "distance to baseline", "three-character classes",
           "largest four-character class", "four-character share", "four-character classes",
           "not resolved by administrative classification", "largest holder share", "top three holders"]]
_hit  = _S2["distance to baseline"] < 0
_edge = _S2["distance to baseline"].abs() <= MARGIN
print(f"\nTable S2  Topic screening (threshold = three-character random baseline {_BASE['main_ipc_3']:.4f})")
print(f"  hits: {_hit.sum()} / {len(_S2)} topics, "
      f"covering {_S2.loc[_hit,'records'].sum():,} records ({_S2.loc[_hit,'records'].sum()/_N*100:.1f}%)")
print(f"  of these, borderline topics within ±{MARGIN} of the baseline: {int(_edge.sum())}"
      f" ({', '.join(_S2.loc[_edge, 'topic'].str[:28])})")
print("\n  Hits (the administrative classification cannot resolve them; inspect them with the semantic layer):")
print(_S2[_hit].to_string(index=False))
print("\n  The five most concentrated topics among the non-hits (a scan by class code is sufficient):")
print(_S2[~_hit].nlargest(5, "three-character share").to_string(index=False))
print(f"\n  Full ranking ({len(_S2)} topics; only topic/records/three-character share/distance to baseline):")
print(_S2[["topic", "records", "three-character share", "distance to baseline"]].to_string(index=False))
_S2_EN = _S2.rename(columns={
    "topic": "topic", "records": "n_records", "largest three-character class": "top_ipc3", "three-character share": "top_ipc3_share",
    "distance to baseline": "distance_to_majority_share", "three-character classes": "n_ipc3_classes",
    "largest four-character class": "top_ipc4", "four-character share": "top_ipc4_share", "four-character classes": "n_ipc4_classes",
    "not resolved by administrative classification": "below_majority_share", "largest holder share": "top_holder_share", "top three holders": "top3_holders"})
_S2_EN["below_majority_share"] = _S2_EN["below_majority_share"].map(
    {"yes": "yes", "yes (borderline)": "yes (borderline)", " (borderline)": "no (borderline)", "": "no"})
_S2_EN.to_csv(os.path.join(SAVE_DIR, "usage_landscape_screen.csv"), index=False)

# ── Table S3  Top five companies × topic matrix ─────────────────────────────
_top_co = (_t["effective_assignee"].astype(str).str.strip()
           .replace("", np.nan).dropna().value_counts().head(TOP_N_CO).index.tolist())
_m = _t[_t["effective_assignee"].isin(_top_co)]
_S3 = (_m.pivot_table(index="effective_assignee", columns="merged_topic",
                      values="patent_id", aggfunc="count")
         .reindex(_top_co).fillna(0).astype(int))
_S3 = _S3.loc[:, _S3.sum(axis=0) > 0]
_S3pct = _S3.div(_S3.sum(axis=1), axis=0).mul(100).round(1)

print(f"\nTable S3  Topic portfolios of the top {TOP_N_CO} companies (share within the row in %; only the top three topics of each company)")
for _co in _top_co:
    _r = _S3pct.loc[_co].sort_values(ascending=False).head(3)
    _tot = int(_S3.loc[_co].sum())
    print(f"  {_co[:34]:36} n={_tot:>4}  " +
          " | ".join(f"{k[:30]} {v:.0f}%" for k, v in _r.items()))
_S3.to_csv(os.path.join(SAVE_DIR, "usage_company_topic_matrix.csv"))
_S3pct.to_csv(os.path.join(SAVE_DIR, "usage_company_topic_matrix_pct.csv"))

print("\nsaved -> usage_assignee_before_after.csv, usage_landscape_screen.csv,")
print("         usage_company_topic_matrix.csv, usage_company_topic_matrix_pct.csv")

### 第 7 節結果（釋出版本）

釋出紀錄中有 763 筆（11.7%）的現任權利人與原始申請人不同。其中 417 筆（54.7%）的最新讓與紀錄確實移轉了所有權；180 筆沒有任何讓與紀錄，139 筆的最新紀錄是擔保設定或擔保解除。改用 `effective_assignee` 後，有 404 筆（占語料 6.2%）的權利人改變，前十大名單有兩處不同：Everpure Inc 變成 Pure Storage Inc，EMC Corp 變成 EMC IP Holding Co LLC。

主題與四碼分類的一致程度明顯高於隨機，但遠未完全一致：標準化互資訊 0.456，打亂後為 0.057；調整後蘭德指數 0.146，打亂後為 -0.002。25 個主題中有 14 個，其最大類別占該主題一半以上的紀錄。在三碼層級，有 8 個主題的最大類別占比低於全語料基準 0.513，涵蓋 1,654 筆（32.9%），包括 Advanced Packaging Solutions、Computing Environment Cooling Solutions 與 Memory Architecture Optimization。反過來看，G06F 分散在 24 個主題中，最大的一個主題也只占其紀錄的 20.3%。

使用示範中，語料內部 2,244 條引用有 88.0% 連到同一主題的紀錄。涉訟家族的 25 筆紀錄落在 4 個主題。主題分類器在家族分組切分下的 macro-F1 是 0.957，隨機切分下是 0.961。

## 8  摘要層的技術驗證

以下每一項檢查問的是「資料有沒有錯」，而不是哪一種方法比較好。三個摘要層並列釋出，它們之間的差異當作資料特性報告。除了 8.13 以外，所有量測都用全部的釋出紀錄，避免分群結果反過來挑選評估樣本。

### 8.1  token 預算涵蓋了多少原文

分面預算是人為設定的。這一格在隨機抽出的 1,000 筆上，以合成模型的 tokenizer 計算各段落有多少比例落在預算之內。

In [ ]:
# ── 8.1  How much of each section the token budgets cover (CPU only) ─────────
# The facet budgets (background 1,500, summary 1,500, detailed description 2,000 tokens) are a design choice;
# this cell reports how much of the source text they cover on 1,000 records drawn at random. It makes no claim
# that the budgets are optimal, which would need the facets to be generated again with other budgets.
from transformers import AutoTokenizer as _ATP
_tok_prof = _ATP.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", trust_remote_code=True) \
            if globals().get("USE_LLAMA_TOK_FOR_PROFILE") else \
            _ATP.from_pretrained("Qwen/Qwen2.5-14B-Instruct", trust_remote_code=True)

_SRC = m1 if "m1" in globals() else valid_df
_SAMPLE = _SRC.sample(min(1000, len(_SRC)), random_state=RANDOM_STATE)
_SUMC = "summary_filled" if "summary_filled" in _SAMPLE.columns else "summary"
BUDGETS = [("background",           "background",           1500),
           ("summary",              _SUMC,                  1500),
           ("detailed_description", "detailed_description", 2000)]

rows_p = []
for label, col, budget in BUDGETS:
    if col not in _SAMPLE.columns:
        print(f"skipped {label} (column {col} does not exist)"); continue
    L = np.array([len(_tok_prof.encode(str(t)[:400000], add_special_tokens=False))
                  for t in tqdm(_SAMPLE[col], desc=f"tokenize {label}")])
    rows_p.append({"section": label, "budget_tokens": budget,
                   "median_tokens": int(np.median(L)), "mean_tokens": round(float(L.mean()), 1),
                   "p90_tokens": int(np.percentile(L, 90)), "max_tokens": int(L.max()),
                   "pct_truncated": round(float((L > budget).mean() * 100), 1),
                   "mean_coverage_pct": round(float(np.minimum(L, budget).sum() / max(L.sum(), 1) * 100), 1)})

prof = pd.DataFrame(rows_p)
prof.to_csv(os.path.join(SAVE_DIR, "truncation_profile.csv"), index=False)
print(f"\n  {'section':24}{'budget':>7}{'median':>10}{'share cut':>12}{'coverage':>12}")
for r in rows_p:
    print(f"  {r['section']:24}{r['budget_tokens']:>7}{r['median_tokens']:>10}"
          f"{r['pct_truncated']:>11.1f}%{r['mean_coverage_pct']:>11.1f}%")
print(f"\n(sample of {len(_SAMPLE)} records)")
print("saved -> truncation_profile.csv")

### 8.2–8.7  摘要帶了哪些內容

- 8.2：從分面抽出的詞與從全文抽出的詞之間的 Jaccard 重疊。分面把每個段落壓到約一百個詞，所以數值偏低是壓縮造成的，當作特性報告。
- 8.3：`must_keep` 詞出現在各摘要層的比例。
- 8.4：壓縮率、相對 abstract 的 ROUGE，以及配對檢查：一篇摘要對自己的 abstract 的分數，必須高於對下一筆紀錄的 abstract 的分數。
- 8.5：abstract 以外的內容。候選詞是說明書中出現、但 title 與 abstract 沒有的較長技術詞；摘要含有的候選詞算一次命中。abstract 本身依定義是零。
- 8.6：輸入窗以外的內容。detailed description 第 800 到 2,000 個 token 之間的詞（direct-brief 讀不到），與第 2,000 到 4,000 個 token 之間的對照窗（三個資料層都讀不到）比較。
- 8.7：只有單一資料層含有的詞從哪裡來，以及其中有多少在全文中完全找不到。這個數字是捏造內容的上限，因為改寫與詞形變化也會被算進去；直接證據是人工稽核。

In [ ]:
# ── 8.2  Overlap of the facet terms with the full-text terms (Jaccard) ──────
jaccard_all = []
for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="Jaccard"):
    s1 = stage1_df[stage1_df["patent_id"] == row["patent_id"]]
    if len(s1) == 0:
        continue
    s1 = s1.iloc[0]
    mk_s1  = extract_tech_terms(row["title"], s1["bg_summary"],
                                s1["inv_summary"], s1["tech_summary"])
    mk_raw = extract_tech_terms(row["title"], str(row["background"]),
                                str(row["summary"]), str(row["detailed_description"]))
    t_s1  = set(t.strip().lower() for t in mk_s1.split(",")  if t.strip())
    t_raw = set(t.strip().lower() for t in mk_raw.split(",") if t.strip())
    if t_s1 | t_raw:
        jaccard_all.append(len(t_s1 & t_raw) / len(t_s1 | t_raw))

jac = np.array(jaccard_all)
print(f"\nn = {len(jac)}")
print(f"mean Jaccard   : {jac.mean()*100:.1f}%")
print(f"median Jaccard : {np.median(jac)*100:.1f}%")
print(f"full overlap   : {(jac == 1.0).sum()} | zero overlap: {(jac == 0.0).sum()}")

In [ ]:
# ── 8.3  Retention of the must_keep terms in two summary layers ─────────────
def calc_retention_s1(df, summary_col):
    """Retention of the terms extracted from the facets."""
    hit, total = 0, 0
    for _, row in df.iterrows():
        s1 = stage1_df[stage1_df["patent_id"] == row["patent_id"]]
        if len(s1) == 0:
            continue
        s1 = s1.iloc[0]
        mk = extract_tech_terms(row["title"], s1["bg_summary"],
                                s1["inv_summary"], s1["tech_summary"])
        terms = [t.strip().lower() for t in mk.split(",") if t.strip()]
        hit   += sum(1 for t in terms if t in str(row[summary_col]).lower())
        total += len(terms)
    return hit / total * 100 if total else 0

def calc_retention_raw(df, summary_col):
    """Retention computed with the "terms extracted from the original full text" (a stricter reference)."""
    hit, total = 0, 0
    for _, row in df.iterrows():
        mk = extract_tech_terms(row["title"], str(row["background"]),
                                str(row["summary"]), str(row["detailed_description"]))
        terms = [t.strip().lower() for t in mk.split(",") if t.strip()]
        hit   += sum(1 for t in terms if t in str(row[summary_col]).lower())
        total += len(terms)
    return hit / total * 100 if total else 0

SYSTEMS = [("direct-brief", "baseline_summary"), ("synthesized", "final_summary")]
with timed("8.3  Term retention (summary layers)", n_items=2 * len(valid_df)):
    ret_s1  = {n: calc_retention_s1 (valid_df, c) for n, c in SYSTEMS}
    ret_raw = {n: calc_retention_raw(valid_df, c) for n, c in SYSTEMS}
lens = {n: valid_df[c].str.split().str.len() for n, c in SYSTEMS}

hdr = "".join(f"{n:>13}" for n, _ in SYSTEMS)
print(f"{'':34}{hdr}")
print(f"{'retention (facet terms)':34}" +
      "".join(f"{ret_s1[n]:>12.1f}%" for n, _ in SYSTEMS))
print(f"{'retention (full-text terms)':34}" +
      "".join(f"{ret_raw[n]:>12.1f}%" for n, _ in SYSTEMS))
print(f"{'mean length (words)':34}" +
      "".join(f"{lens[n].mean():>13.0f}" for n, _ in SYSTEMS))

In [ ]:
# ── 8.4  Compression ratio, ROUGE against the abstract, and a pairing check ───
# ROUGE with the abstract as the reference measures how similar a summary is to the abstract. The layers are
# meant to carry content that the abstract does not have, so ROUGE is not evidence of correctness here; its
# values are reported as a property of the layers. ROUGE does serve one check: a summary attached to the
# wrong record, or a degenerate generation, scores no better against its own abstract than against the
# abstract of another record. Each summary is therefore scored against its own abstract and against the
# abstract shifted by one position; records that fail are listed in pairing_integrity_flags.csv.
from rouge_score import rouge_scorer
import numpy as np, pandas as pd, os

_SUM_COL = "summary_filled" if "summary_filled" in valid_df.columns else "summary"
src_words = (valid_df["background"].astype(str) + " " +
             valid_df[_SUM_COL].astype(str) + " " +
             valid_df["detailed_description"].astype(str)).str.split().str.len()

Q_SYSTEMS = [(n, c) for n, c in (("direct-brief", "baseline_summary"),
                                 ("direct-full", "baseline_matched_summary"),
                                 ("synthesized",       "final_summary"))
             if c in valid_df.columns and valid_df[c].astype(str).str.len().gt(0).any()]
print("Systems included:", [n for n, _ in Q_SYSTEMS])

LOW_R1 = 0.10          # anomaly threshold declared in advance
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
abstracts = valid_df["abstract"].astype(str).tolist()
shuffled  = abstracts[1:] + abstracts[:1]        # shift by one position: a deterministic, reproducible control permutation

rows_q, flagged = [], []
for name, col in Q_SYSTEMS:
    hyps = valid_df[col].astype(str).tolist()
    sw   = valid_df[col].astype(str).str.split().str.len()
    comp = sw / src_words.clip(lower=1)
    with timed(f"8.4  ROUGE with shifted control ({name})", n_items=len(valid_df)):
        own1, own2, ownL, shf1 = [], [], [], []
        for i, h in enumerate(hyps):
            s = scorer.score(abstracts[i], h)
            own1.append(s["rouge1"].fmeasure); own2.append(s["rouge2"].fmeasure)
            ownL.append(s["rougeL"].fmeasure)
            shf1.append(scorer.score(shuffled[i], h)["rouge1"].fmeasure)
    own1, shf1 = np.array(own1), np.array(shf1)
    n_low  = int((own1 < LOW_R1).sum())
    n_fail = int((own1 <= shf1).sum())          # not better against its own abstract than against a random one
    print(f"  {name:10} compression {comp.mean()*100:5.2f}% | mean summary {sw.mean():5.0f} words | "
          f"ROUGE-1 {own1.mean():.4f} (shifted control {shf1.mean():.4f}) | "
          f"R-2 {np.mean(own2):.4f} | R-L {np.mean(ownL):.4f}")
    print(f"             pairing integrity: ROUGE-1 < {LOW_R1} in {n_low} records"
          f" ({n_low/len(own1)*100:.2f}%) | not better than the shifted control: {n_fail} records"
          f" ({n_fail/len(own1)*100:.2f}%)")
    rows_q.append({"system": name, "compression_ratio": round(float(comp.mean()), 4),
                   "mean_summary_words": round(float(sw.mean()), 1),
                   "rouge1_f": round(float(own1.mean()), 4),
                   "rouge2_f": round(float(np.mean(own2)), 4),
                   "rougeL_f": round(float(np.mean(ownL)), 4),
                   "rouge1_permuted": round(float(shf1.mean()), 4),
                   "rouge1_lift": round(float(own1.mean() - shf1.mean()), 4),
                   "n_low_rouge1": n_low, "n_not_better_than_permuted": n_fail})
    _bad = valid_df.loc[(own1 < LOW_R1) | (own1 <= shf1), "patent_id"]
    flagged.extend({"system": name, "patent_id": p,
                    "rouge1_own": round(float(own1[j]), 4),
                    "rouge1_permuted": round(float(shf1[j]), 4)}
                   for j, p in zip(np.where((own1 < LOW_R1) | (own1 <= shf1))[0], _bad))

pd.DataFrame(rows_q).to_csv(os.path.join(SAVE_DIR, "summary_metrics.csv"), index=False)
pd.DataFrame(flagged).to_csv(os.path.join(SAVE_DIR, "pairing_integrity_flags.csv"), index=False)
print(f"\nThe lift (own − shifted control) is the evidence of correct pairing; {len(flagged)} records are flagged for inspection")
print("saved -> summary_metrics.csv, pairing_integrity_flags.csv")

In [ ]:
# ── 8.5  Content of the summaries beyond the abstract (CPU only) ─────────────
# Are the summaries only paraphrases of the abstract?
#   candidate = a long technical word that occurs in the specification but not in the title or abstract
#   hit       = a candidate that occurs in the summary
# The abstract itself scores 0 by this definition. The check needs no model and can be repeated in minutes.
import numpy as np, pandas as pd, os, re
for _n in ("HPC_WHITELIST", "TITLE_STOPWORDS", "GENERIC_SUFFIXES"):
    if _n not in globals():
        raise RuntimeError("Run 5.3 first (definition of the term extraction).")

_LONG_AV = re.compile(r"\b[a-zA-Z][a-zA-Z-]{11,}\b")
def _av_terms(text):
    out = set()
    for w in _LONG_AV.findall(str(text)):
        lw = w.lower()
        if lw in HPC_WHITELIST or (lw not in TITLE_STOPWORDS and not GENERIC_SUFFIXES.search(lw)):
            out.add(lw)
    return out

_SUMC = "summary_filled" if "summary_filled" in valid_df.columns else "summary"
AV_SYSTEMS = [(n, c) for n, c in (("direct-brief", "baseline_summary"),
                                  ("direct-full", "baseline_matched_summary"),
                                  ("synthesized",       "final_summary"))
              if c in valid_df.columns and valid_df[c].astype(str).str.len().gt(0).any()]
print("Systems compared:", [n for n, _ in AV_SYSTEMS])

rec = []
with timed("8.5  Content beyond the abstract", n_items=len(valid_df)):
    for _, r in tqdm(valid_df.iterrows(), total=len(valid_df), desc="added-value"):
        abs_terms  = _av_terms(f'{r["title"]} {r["abstract"]}')
        spec_terms = _av_terms(f'{r["background"]} {r[_SUMC]} '
                               f'{str(r["detailed_description"])[:60000]}')
        cand = spec_terms - abs_terms
        if not cand:
            continue
        row = {"patent_id": r["patent_id"], "n_candidate": len(cand)}
        for name, col in AV_SYSTEMS:
            s_ = str(r[col]).lower()
            n_hit = sum(1 for t in cand if t in s_)
            row[f"{name}_added"] = n_hit
            row[f"{name}_rate"]  = n_hit / len(cand)
        rec.append(row)

add_df = pd.DataFrame(rec)
add_df.to_csv(os.path.join(SAVE_DIR, "added_value_beyond_abstract.csv"), index=False)
print(f"\nrecords that can be evaluated {len(add_df):,} / {len(valid_df):,}"
      f" | mean candidates per record {add_df['n_candidate'].mean():.1f}")
print(f"  {'system':16}{'added terms':>14}{'coverage':>10}{'at least one':>14}")
for name, _ in AV_SYSTEMS:
    a = add_df[f"{name}_added"]
    print(f"  {name:16}{a.mean():>14.2f}{add_df[f'{name}_rate'].mean()*100:>9.1f}%"
          f"{(a >= 1).mean()*100:>13.1f}%")
print(f"  {'abstract itself':16}{0.00:>14.2f}{0.0:>9.1f}%{0.0:>13.1f}%   ← doing nothing")

# Paired bootstrap: report first the budget-matched pair (synthesized vs direct-full); otherwise report synthesized vs direct-brief and note the mixture
_hi = "synthesized"
_lo = "direct-full" if any(n == "direct-full" for n, _ in AV_SYSTEMS) else "direct-brief"
_note = ("input budgets matched; the difference can be attributed to the process structure"
         if _lo == "direct-full" else
         "input budgets not matched (direct-brief reads about 2,000 tokens, synthesized about 5,000); the difference mixes the two effects")
rng = np.random.RandomState(RANDOM_STATE)
d  = (add_df[f"{_hi}_added"] - add_df[f"{_lo}_added"]).values
bs = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(1000)])
print(f"\n{_hi} − {_lo}: {d.mean():+.2f} words per record"
      f" | 95% CI [{np.percentile(bs, 2.5):+.2f}, {np.percentile(bs, 97.5):+.2f}]")
print(f"({_note})")
print("saved -> added_value_beyond_abstract.csv")

In [ ]:
# ── 8.6  Coverage beyond the input window: does deep content reach the summary? ──
# Visible range of the three layers (tokens of the synthesis model):
#   direct-brief  background[0,800)   summary[0,400)   detailed[0,800)    about 2,000
#   direct-full   background[0,1500)  summary[0,1500)  detailed[0,2000)   about 5,000
#   synthesized   background[0,1500)  summary[0,1500)  detailed[0,2000)   about 5,000
# The two windows use different visible baselines, so that terms that the specification repeats in its
# visible part do not count:
#   deep zone [800,2000): minus the visible range of direct-brief. direct-brief cannot read it; the other two can.
#                    direct-full − direct-brief = reading-depth effect
#                    synthesized − direct-full       = process-structure effect (both read it; which one brings more of it into the output)
#                    The latter is a descriptive difference, not a hypothesis test.
#   control window [2000,4000): minus the union of the visible ranges of the three. No system reads it, so the survival rate must be near 0.
from transformers import AutoTokenizer as _AT
_tok_cov = _AT.from_pretrained("Qwen/Qwen2.5-14B-Instruct", trust_remote_code=True)

_LONG_COV = re.compile(r"\b[a-zA-Z][a-zA-Z-]{11,}\b")

def _cov_terms(segment):
    terms = set()
    for w in _LONG_COV.findall(segment):
        lw = w.lower()
        if lw in HPC_WHITELIST or (lw not in TITLE_STOPWORDS
                                   and not GENERIC_SUFFIXES.search(lw)):
            terms.add(lw)
    return terms

def _tok_slice(text, lo, hi, max_chars):
    toks = _tok_cov.encode(str(text)[:max_chars], add_special_tokens=False)
    if len(toks) <= lo:
        return None
    return _tok_cov.decode(toks[lo:hi], skip_special_tokens=True)

SYSTEMS = [(n, c) for n, c in (("direct-brief", "baseline_summary"),
                               ("direct-full", "baseline_matched_summary"),
                               ("synthesized",       "final_summary"))
           if c in valid_df.columns and valid_df[c].astype(str).str.len().gt(0).any()]
print("Systems compared:", [n for n, _ in SYSTEMS])

from collections import Counter as _Counter
cov, n_eval = {n: _Counter() for n, _ in SYSTEMS}, _Counter()
with timed("8.6  Coverage beyond the input window (summary layers)", n_items=len(valid_df)):
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="coverage"):
        sum_in = str(row.get("summary_filled", row.get("summary", "")))
        head   = f'{row["title"]} {row["abstract"]} '
        vis_short = (head
                     + (_tok_slice(row["background"], 0, 800, 12000) or "") + " "
                     + (_tok_slice(sum_in,            0, 400, 12000) or "") + " "
                     + (_tok_slice(row["detailed_description"], 0, 800, 12000) or "")).lower()
        vis_union = (head
                     + (_tok_slice(row["background"], 0, 1500, 20000) or "") + " "
                     + (_tok_slice(sum_in,            0, 1500, 20000) or "") + " "
                     + (_tok_slice(row["detailed_description"], 0, 2000, 30000) or "")).lower()
        for wkey, lo, hi, vis in (("deep", 800, 2000, vis_short),
                                  ("control", 2000, 4000, vis_union)):
            seg = _tok_slice(row["detailed_description"], lo, hi, 30000)
            if not seg:
                continue
            terms = {t for t in _cov_terms(seg) if t not in vis}
            if not terms:
                continue
            n_eval[wkey] += 1
            for name, col in SYSTEMS:
                s = str(row[col]).lower()
                cov[name][f"{wkey}_hit"]   += sum(1 for t in terms if t in s)
                cov[name][f"{wkey}_total"] += len(terms)

print(f"\nRecords that can be evaluated: deep zone [800,2000) = {n_eval['deep']}; control [2000,4000) = {n_eval['control']}")
print(f"{'system':12}{'deep zone':>12}{'control':>10}")
rows_c, rate = [], {}
for name, _ in SYSTEMS:
    c = cov[name]
    deep = c["deep_hit"] / c["deep_total"] * 100 if c["deep_total"] else float("nan")
    ctrl = c["control_hit"] / c["control_total"] * 100 if c["control_total"] else float("nan")
    rate[name] = deep
    print(f"{name:12}{deep:>11.2f}%{ctrl:>9.2f}%")
    rows_c.append({"system": name, "deep_survival_pct": round(deep, 2),
                   "control_pct": round(ctrl, 2),
                   "deep_n_terms": int(c["deep_total"]), "control_n_terms": int(c["control_total"])})
pd.DataFrame(rows_c).to_csv(os.path.join(SAVE_DIR, "coverage_beyond_truncation.csv"), index=False)

# ── Decomposition of the two effects (only valid with direct-full) ─────────────
if {"direct-brief", "direct-full", "synthesized"} <= set(rate):
    print(f"\nreading-depth effect      direct-full − direct-brief = {rate['direct-full'] - rate['direct-brief']:+.2f} percentage points")
    print(f"process-structure effect  synthesized       − direct-full = {rate['synthesized'] - rate['direct-full']:+.2f} percentage points"
          f"   ← descriptive difference, not a hypothesis test")
else:
    print("\n(5.11 was not run, so the two effects cannot be separated: the present difference contains both reading depth and process structure)")
print("saved -> coverage_beyond_truncation.csv")

In [ ]:
# ── 8.7  Where the terms that only one layer contains come from (CPU only) ────
# Is the non-overlapping content of the synthesized and the direct-full layer a structural difference or
# generation noise? Both layers use greedy decoding (5.2), so the same input and prompt give the same output;
# a difference can only come from the three recorded design differences: input span, prompt path (three
# section prompts and two calls against one prompt) and the term list, which only the synthesized layer gets.
# Four descriptive predictions follow from these differences:
#   Prediction 1  terms only in synthesized lean toward the deep zone of the detailed description (after token 800)
#   Prediction 2  consensus terms lean toward the front part (title/abstract/background/first 800 tokens of the detailed description)
#   Prediction 3  terms only in synthesized fall in the domain whitelist more often than terms only in the control (the effect of the term guidance)
#   Prediction 4  if it is only noise, none of the three above shows a difference
#   Additional output: the five zones cover only the first 8,000 tokens of the detailed description and do not include the claims; a term that the zones do not find is compared once more against the "full lower-case text"
#   (the six text fields, including the claims): a term still found in the full text counts as elsewhere_in_source; only a term really not in the full text is not_in_source.
#   not_in_source is an upper bound of fabrication (paraphrases and inflections also fall here), not an error rate.
for _fn in ("_tok_cov", "_cov_terms", "_tok_slice"):
    if _fn not in globals():
        raise RuntimeError("Run 8.6 first (this cell uses its tokenizer and term helpers).")

_PAIR_HI = "final_summary"
_PAIR_LO = ("baseline_matched_summary"
            if "baseline_matched_summary" in valid_df.columns
            and valid_df["baseline_matched_summary"].astype(str).str.len().gt(0).any()
            else "baseline_summary")
_LO_NAME = "direct-full" if _PAIR_LO == "baseline_matched_summary" else "direct-brief"
print(f"Pair: synthesized ({_PAIR_HI}) vs {_LO_NAME} ({_PAIR_LO})")
if _LO_NAME == "direct-brief":
    print("Note: 5.11 was not run; the result contains both effects, reading depth and process structure")

BUCKETS = ["front", "background", "dd_0_800", "dd_800_2000", "dd_2000_plus", "elsewhere_in_source", "not_in_source"]
_nis_rows = []   # per record: share of the synthesized-only terms that are not in the full text (edge cases of 8.8)
GROUPS  = ["consensus", "synthesized_only", f"{_LO_NAME}_only"]
from collections import Counter as _C
tally = {g: _C() for g in GROUPS}
wl    = {g: _C() for g in GROUPS}
_SUMC = "summary_filled" if "summary_filled" in valid_df.columns else "summary"

with timed("8.7  Structure of the non-overlapping content", n_items=len(valid_df)):
    for _, row in tqdm(valid_df.iterrows(), total=len(valid_df), desc="overlap"):
        sum_in = str(row.get(_SUMC, ""))
        zones = [
            ("front",        f'{row["title"]} {row["abstract"]}'.lower()),
            ("background",  ((_tok_slice(row["background"], 0, 1500, 20000) or "") + " " +
                             (_tok_slice(sum_in,            0, 1500, 20000) or "")).lower()),
            ("dd_0_800",     (_tok_slice(row["detailed_description"], 0, 800, 12000) or "").lower()),
            ("dd_800_2000",  (_tok_slice(row["detailed_description"], 800, 2000, 30000) or "").lower()),
            ("dd_2000_plus", (_tok_slice(row["detailed_description"], 2000, 8000, 60000) or "").lower()),
        ]
        _full = (f'{row["title"]} {row["abstract"]} {row["background"]} {sum_in} '
                 f'{row["detailed_description"]} {row.get("claims_text", "")}').lower()
        _row_n = _row_nis = 0
        t_hi = _cov_terms(str(row[_PAIR_HI]))
        t_lo = _cov_terms(str(row[_PAIR_LO]))
        for g, terms in (("consensus", t_hi & t_lo), ("synthesized_only", t_hi - t_lo),
                         (f"{_LO_NAME}_only", t_lo - t_hi)):
            for t in terms:
                b = next((z for z, txt in zones if t in txt),
                         "elsewhere_in_source" if t in _full else "not_in_source")
                tally[g][b] += 1
                wl[g]["hit" if t in HPC_WHITELIST else "miss"] += 1
                if g == "synthesized_only":
                    _row_n += 1; _row_nis += (b == "not_in_source")
        _nis_rows.append({"patent_id": row["patent_id"], "n_synth_only_terms": _row_n,
                          "n_not_in_full_text": _row_nis})

print(f"\nSource position distribution of each group (%)")
print(f"  {'group':18}" + "".join(f"{b:>15}" for b in BUCKETS) + f"{'n_terms':>10}")
rows_o = []
for g in GROUPS:
    tot = sum(tally[g].values()) or 1
    print(f"  {g:18}" + "".join(f"{tally[g][b]/tot*100:>14.1f}%" for b in BUCKETS) + f"{tot:>10,}")
    r = {"group": g, "n_terms": tot,
         "whitelist_pct": round(wl[g]["hit"] / max(wl[g]["hit"] + wl[g]["miss"], 1) * 100, 1)}
    r.update({b: round(tally[g][b] / tot * 100, 2) for b in BUCKETS})
    rows_o.append(r)
pd.DataFrame(rows_o).to_csv(os.path.join(SAVE_DIR, "overlap_structure.csv"), index=False)

_deep = {g: tally[g]["dd_800_2000"] + tally[g]["dd_2000_plus"] for g in GROUPS}
_tot  = {g: sum(tally[g].values()) or 1 for g in GROUPS}
_lo_g = f"{_LO_NAME}_only"
print(f"\nFour descriptive comparisons")
print(f"  1  synthesized-only terms in the deep zone : {_deep['synthesized_only']/_tot['synthesized_only']*100:.1f}%"
      f"  vs {_lo_g} {_deep[_lo_g]/_tot[_lo_g]*100:.1f}%"
      f"  → {'agrees' if _deep['synthesized_only']/_tot['synthesized_only'] > _deep[_lo_g]/_tot[_lo_g] else 'does not agree'}")
_front = {g: tally[g]["front"] + tally[g]["background"] + tally[g]["dd_0_800"] for g in GROUPS}
print(f"  2  consensus terms in the front part       : {_front['consensus']/_tot['consensus']*100:.1f}%"
      f"  vs synthesized only {_front['synthesized_only']/_tot['synthesized_only']*100:.1f}%"
      f"  → {'agrees' if _front['consensus']/_tot['consensus'] > _front['synthesized_only']/_tot['synthesized_only'] else 'does not agree'}")
print(f"  3  synthesized-only terms in the whitelist : {rows_o[1]['whitelist_pct']:.1f}%"
      f"  vs {_lo_g} {rows_o[2]['whitelist_pct']:.1f}%"
      f"  → {'agrees' if rows_o[1]['whitelist_pct'] > rows_o[2]['whitelist_pct'] else 'does not agree'}")
print(f"  4  if noise, 1–3 must all show no difference (for comparison; not counted separately)")
print(f"\nShare of long terms not in the full text (upper bound of fabrication; compared against the full text with the claims): "
      + " | ".join(f"{g} {tally[g]['not_in_source']/_tot[g]*100:.1f}%" for g in GROUPS))
print("  ← upper bound, not an error rate; the direct evidence is the human audit (8.8 sample)")
_nis_df = pd.DataFrame(_nis_rows)
_nis_df["not_in_source_share"] = _nis_df["n_not_in_full_text"] / _nis_df["n_synth_only_terms"].replace(0, np.nan)
_nis_df.to_csv(os.path.join(SAVE_DIR, "not_in_full_text_per_record.csv"), index=False)
valid_df = (valid_df.drop(columns=[c for c in ("not_in_source_share",) if c in valid_df.columns])
                    .merge(_nis_df[["patent_id", "not_in_source_share"]], on="patent_id", how="left"))
print("saved -> not_in_full_text_per_record.csv (upper bound per record, for the edge cases of 8.8)")
print("saved -> overlap_structure.csv")

### 8.8  人工稽核的樣本

自動檢查找不出「寫了一件看似合理、但專利裡沒有的事」的句子，所以 synthesized 層另外由人逐句稽核。樣本包括依分類部（G、H、其他）分層抽出的 50 筆，以及依固定規則抽出的 12 筆邊界案例：與分面相似度最低、被配對檢查標記、說明最長、background 被替補，以及全文找不到的詞占比最高。釋出資料集的樣本是 2026-09-12 由這一格抽出的；沿用該次執行時，不會重新抽樣。標註規程與結果隨資料一起釋出（`validation/human_audit/`）。

In [ ]:
# ── 8.8  Human-audit sample: 50 records stratified by class section and 12 edge cases (CPU only) ──
# The sampling rule, the seed and the definition of the edge cases are all in this cell, so the sample is
# reproducible. The annotation protocol is released with the data (validation/human_audit/). The sample of
# the released dataset was drawn on 2026-09-12 and annotated afterwards; when the three sample files exist
# (for example copied by 1.7 from the reused run), no new sample is drawn.
_AUDIT_FILES = [os.path.join(SAVE_DIR, f) for f in ("audit_primary_50.csv", "audit_edge_12.csv", "audit_sentences.csv")]
if all(os.path.exists(p) for p in _AUDIT_FILES):
    print("The audit sample exists; no new sample is drawn:", [os.path.basename(p) for p in _AUDIT_FILES])
else:
    import re as _re
    _src = valid_df.copy()
    _src["ipc_sec"] = _src["main_ipc_4"].astype(str).str[0]
    _src["ipc_sec"] = _src["ipc_sec"].where(_src["ipc_sec"].isin(["G", "H"]), "other")

    # Primary sample: stratified in proportion by CPC section (G, H, other), filled up to 50 in total
    _share = _src["ipc_sec"].value_counts(normalize=True)
    _n = {k: int(round(50 * v)) for k, v in _share.items()}
    _n[_share.idxmax()] += 50 - sum(_n.values())
    _prim = pd.concat([_src[_src["ipc_sec"] == k].sample(n=v, random_state=RANDOM_STATE)
                       for k, v in _n.items() if v > 0])
    _prim = _prim.assign(stratum="primary:" + _prim["ipc_sec"])

    # Edge cases: fixed rules; records already in the primary sample are excluded
    _edge = []
    def _take(mask, k, name):
        _e = _src[mask & ~_src["patent_id"].isin(_prim["patent_id"])
                  & ~_src["patent_id"].isin(pd.concat(_edge)["patent_id"] if _edge else [])].head(k)
        _edge.append(_e.assign(stratum=name))
    _take(_src["patent_id"].isin(["US11321283B2", "US11573920B2", "US11321264B2"]), 3, "edge:low_cosine_to_facets")
    _flag_p = os.path.join(SAVE_DIR, "pairing_integrity_flags.csv")
    if os.path.exists(_flag_p):
        _flag = pd.read_csv(_flag_p)
        _take(_src["patent_id"].isin(_flag.loc[_flag["system"] == "synthesized", "patent_id"]), 3, "edge:pairing_flag")
    _take(_src["detailed_description"].astype(str).str.split().str.len().rank(ascending=False) <= 2, 2, "edge:longest_description")
    _take(_src["background"].astype(str).str.len() < MIN_LEN, 2, "edge:short_background_substituted_what")
    if "not_in_source_share" in _src.columns:      # written back per record by 8.7
        _take(_src["not_in_source_share"].rank(ascending=False, method="first") <= 2, 2, "edge:highest_not_in_full_text")
    _edge = pd.concat(_edge).drop_duplicates("patent_id").head(12)

    _cols = ["patent_id", "stratum", "title", "abstract", "final_summary"]
    _prim_out = _prim[_cols].assign(patent_url="https://patents.google.com/patent/" + _prim["patent_id"].astype(str))
    _edge_out = _edge[_cols].assign(patent_url="https://patents.google.com/patent/" + _edge["patent_id"].astype(str))
    _prim_out.to_csv(os.path.join(SAVE_DIR, "audit_primary_50.csv"), index=False)
    _edge_out.to_csv(os.path.join(SAVE_DIR, "audit_edge_12.csv"), index=False)

    _sent = []
    for _, r in pd.concat([_prim_out, _edge_out]).iterrows():
        for j, sent in enumerate(_re.split(r"(?<=[.!?])\s+", str(r["final_summary"]).strip())):
            if sent:
                _sent.append({"patent_id": r["patent_id"], "stratum": r["stratum"], "sentence_id": j + 1,
                              "sentence": sent, "label": "", "evidence_span": "", "error_type": "", "annotator": ""})
    pd.DataFrame(_sent).to_csv(os.path.join(SAVE_DIR, "audit_sentences.csv"), index=False)
    print(f"primary sample {len(_prim_out)} ({ {k: int(v) for k, v in _prim['ipc_sec'].value_counts().items()} }) | edge {len(_edge_out)}"
          f" ({ {k: int(v) for k, v in _edge_out['stratum'].value_counts().items()} }) | sentences {len(_sent)}")
    print("saved -> audit_primary_50.csv, audit_edge_12.csv, audit_sentences.csv")

### 8.9–8.10  以五種表徵做的使用示範

兩個標準任務示範釋出的向量怎麼接上分類器；兩者都不是在主張哪一種表徵比較好。分類器事先固定：對單位長度的向量做邏輯迴歸，並以家族分組切分。

- 8.9：CPC 代碼。主任務是預測一筆紀錄的所有四碼代碼（多標籤，micro-F1）。次要代碼代表說明書在主要技術以外揭露的主題，常常不在 abstract 裡。表徵之間的差距附上配對 bootstrap 區間（1,000 次）。
- 8.10：原始申請人（`assignee`）。所有權移轉不會改變文字，所以用現任權利人當標籤並不公平。各公司有固定的撰寫習慣，這個任務也有一部分是在辨認用詞。

In [ ]:
# ── 8.9  Usage example: CPC classification from each representation (family-grouped split) ──
# The example shows how the released embeddings plug into a standard classifier; it is not a claim that one
# representation is better than another.
# Main task: multi-label four-character CPC codes (micro-F1). The secondary codes name technical subjects that
# the specification discloses besides the main one, often not in the abstract.
# Secondary tasks: the main four-character code (macro-F1) and the main three-character class (accuracy).
# The classifier is fixed in advance: logistic regression on unit-length embeddings, a linear probe of the
# representation rather than of the classifier.
import ast, numpy as np, pandas as pd, os
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import f1_score, accuracy_score

# MIN_LABEL_N (codes with fewer records are dropped) and N_BOOT (bootstrap draws) are set in 1.3.

# Source of the multi-label labels: ipc_description (the list of all class codes of a record).
# A table in memory with this column is used first; otherwise the two columns are read from the candidate file.
_IPCSRC = None
for _c in ("m1", "corpus_df", "released_df", "valid_df", "df", "df_raw"):
    _o = globals().get(_c)
    if isinstance(_o, pd.DataFrame) and "ipc_description" in _o.columns:
        _IPCSRC = _o[["patent_id", "ipc_description"]]
        print(f"ipc_description source: {_c} in memory")
        break
if _IPCSRC is None:
    if "ipc_description" not in pd.read_csv(CLEAN_PATH, nrows=0).columns:
        raise RuntimeError(f"The candidate file has no ipc_description column: {CLEAN_PATH}")
    with timed("8.9  Read ipc_description"):
        _IPCSRC = pd.read_csv(CLEAN_PATH, usecols=["patent_id", "ipc_description"], dtype=str)
    print(f"ipc_description source: candidate file ({len(_IPCSRC):,} records)")
_IPCSRC = (_IPCSRC.dropna(subset=["patent_id"])
                  .astype({"patent_id": str})
                  .drop_duplicates("patent_id"))

def _ipc4(s):
    try:
        lst = ast.literal_eval(str(s)) if str(s).strip().startswith("[") else []
    except Exception:
        lst = []
    return {d["ipc"][:4] for d in lst if isinstance(d, dict) and d.get("ipc")}

lab = (valid_df[["patent_id", "main_ipc_4"]].astype({"patent_id": str})
       .merge(_IPCSRC, on="patent_id", how="left"))
# When the join key does not match, the multi-label task silently degrades into a single-label task with "only main_ipc_4",
# and the comparison of this section loses its meaning. Thus the coverage is printed, and the run stops when nothing matches.
_miss = int(lab["ipc_description"].isna().sum())
if _miss == len(lab):
    raise RuntimeError("The patent_id of valid_df does not match the ipc_description source at all; "
                       "SAVE_DIR and the candidate file belong to different corpus versions.")
print(f"ipc_description coverage: {len(lab) - _miss:,}/{len(lab):,} "
      f"({(1 - _miss / len(lab)) * 100:.1f}%)")
lab["all4"] = [s | ({m} if isinstance(m, str) and m else set())
               for s, m in zip(lab["ipc_description"].map(_ipc4), lab["main_ipc_4"])]
_cnt = pd.Series([c for s in lab["all4"] for c in s]).value_counts()
_keep = set(_cnt[_cnt >= MIN_LABEL_N].index)
lab["kept"] = [s & _keep for s in lab["all4"]]
mask = np.array([len(s) > 0 for s in lab["kept"]])
sub  = lab[mask].reset_index(drop=True)

mlb = MultiLabelBinarizer(classes=sorted(_keep)); Y = mlb.fit_transform(sub["kept"])
le  = LabelEncoder(); y1 = le.fit_transform(sub["main_ipc_4"].values)
# Three-character classes: about ten times fewer classes than the subclasses, a granularity close to that of the topics
sub = add_ipc3(sub)
le3 = LabelEncoder(); y3 = le3.fit_transform(sub["main_ipc_3"].values)
g   = build_family_groups(sub["patent_id"].values)
ix  = np.arange(len(sub))
tr, te, _, _ = family_train_test_split(ix, y1, g)

print(f"n={len(sub):,}/{len(valid_df):,} | multi-label classes {Y.shape[1]} (threshold >= {MIN_LABEL_N})"
      f" | mean labels per record {Y.sum(1).mean():.2f} | train {len(tr):,} | test {len(te):,} (family-grouped split)")
print(f"### Feasibility check: records with two or more labels {(Y.sum(1) >= 2).mean()*100:.1f}%"
      f"  ← this is the part that the abstract cannot easily answer; if it is too low, this task cannot show differences between representations")

REPS = {k: v for k, v in REPRESENTATIONS.items() if v is not None}
rows, P_ = [], {}
for rep, X in REPS.items():
    Xs = X[mask]
    with timed(f"8.9  {rep}  multi-label", n_items=len(tr), quiet=True) as t:
        m = OneVsRestClassifier(LogisticRegression(max_iter=2000, C=1.0), n_jobs=-1)
        _ = m.fit(Xs[tr], Y[tr]); Yp = m.predict(Xs[te])
    with timed(f"8.9  {rep}  single-label", n_items=len(tr), quiet=True) as t2:
        c = LogisticRegression(max_iter=2000, C=1.0); _ = c.fit(Xs[tr], y1[tr]); yp = c.predict(Xs[te])
    with timed(f"8.9  {rep}  ipc3", n_items=len(tr), quiet=True) as t3:
        c3 = LogisticRegression(max_iter=2000, C=1.0); _ = c3.fit(Xs[tr], y3[tr]); yp3 = c3.predict(Xs[te])
    P_[rep] = Yp
    ev = np.unique(y1[te])
    # Simple metric: the share of test patents with "at least one correct class code"
    _any_hit = float((np.logical_and(Y[te], Yp).sum(1) > 0).mean())
    rows.append({"representation": rep, "dim": int(Xs.shape[1]),
                 "any_label_hit": round(_any_hit, 4),
                 "ml_f1_micro": round(f1_score(Y[te], Yp, average="micro", zero_division=0), 4),
                 "ml_f1_macro": round(f1_score(Y[te], Yp, average="macro", zero_division=0), 4),
                 "single_f1_macro": round(f1_score(y1[te], yp, average="macro",
                                                   labels=ev, zero_division=0), 4),
                 "single_acc": round(accuracy_score(y1[te], yp), 4),
                 "ipc3_acc": round(accuracy_score(y3[te], yp3), 4),
                 "ipc3_f1_macro": round(f1_score(y3[te], yp3, average="macro",
                                                 labels=np.unique(y3[te]), zero_division=0), 4),
                 "n_ipc3_classes": int(len(np.unique(y3))),
                 "n_records": int(len(sub)), "n_labels": int(Y.shape[1]),
                 "n_test": int(len(te)), "fit_sec": round(t.seconds + t2.seconds, 1)})

# Floor: the single-label accuracy when the largest training class is always guessed, without the vectors
_floor  = float(pd.Series(y1[tr]).value_counts(normalize=True).iloc[0])
_floor3 = float(pd.Series(y3[tr]).value_counts(normalize=True).iloc[0])
print(f"\n  Classes: three-character {len(np.unique(y3))} | four-character {len(np.unique(y1))} | multi-label {Y.shape[1]}")
print(f"\n  {'representation':16}{'Acc (three)':>13}{'Acc (four)':>13}{'any correct':>12}"
      f"{'micro (multi)':>14}")
for r in rows:
    print(f"  {r['representation']:16}{r['ipc3_acc']*100:>12.1f}%{r['single_acc']*100:>12.1f}%"
          f"{r['any_label_hit']*100:>11.1f}%{r['ml_f1_micro']:>14.4f}")
print(f"  {'(largest class)':16}{_floor3*100:>12.1f}%{_floor*100:>12.1f}%{'—':>12}{'—':>14}"
      f"   <- floor without the vectors")
pd.DataFrame(rows).to_csv(os.path.join(SAVE_DIR, "ipc_downstream_comparison.csv"), index=False)

# ── Paired bootstrap (micro-F1), resampled indices shared over the test set ───
COMPARISONS = [
    # ── Against the inputs that already exist (the abstract, the cut full text) ───
    ("synthesized", "abstract",
     "better than the abstract alone (a field that the patent page already gives)?"),
    ("synthesized", "fulltext-truncated",
     "better than encoding the joined full text (cut at 512 tokens, out-of-distribution input)?"),
    # ── Descriptive differences between the released layers (CSV only) ───
    ("synthesized", "direct-full",  "release layers only: difference between layers"),
    ("synthesized", "direct-brief", "release layers only: difference between layers"),
    ("direct-full", "direct-brief", "release layers only: difference between layers"),
]
Yte = Y[te]; rng = np.random.RandomState(RANDOM_STATE); n = len(te)
bidx = [rng.randint(0, n, n) for _ in range(N_BOOT)]
boot = []
with timed("8.9  Paired bootstrap (micro-F1)", n_items=len(bidx)):
    for hi, lo, q in COMPARISONS:
        if hi not in P_ or lo not in P_:
            print(f"  [skipped {hi} − {lo}] (representation does not exist)"); continue
        pt = (f1_score(Yte, P_[hi], average="micro", zero_division=0)
              - f1_score(Yte, P_[lo], average="micro", zero_division=0))
        a = np.array([f1_score(Yte[b], P_[hi][b], average="micro", zero_division=0)
                      - f1_score(Yte[b], P_[lo][b], average="micro", zero_division=0) for b in bidx])
        l, h = np.percentile(a, 2.5), np.percentile(a, 97.5)
        v = "interval includes 0; no difference shown" if l <= 0 <= h else ("interval excludes 0; positive" if l > 0 else "interval excludes 0; negative")
        print(f"  [{hi:14} − {lo:14}] Δ {pt:+.4f} | 95% CI [{l:+.4f}, {h:+.4f}] → {v}   ({q})")
        boot.append({"comparison": f"{hi} - {lo}", "question": q, "metric": "micro_f1",
                     "point_estimate": round(float(pt), 4), "ci_low": round(float(l), 4),
                     "ci_high": round(float(h), 4), "n_boot": len(bidx), "verdict": v})
pd.DataFrame(boot).to_csv(os.path.join(SAVE_DIR, "ipc_downstream_bootstrap.csv"), index=False)
print("saved -> ipc_downstream_comparison.csv, ipc_downstream_bootstrap.csv")

In [ ]:
# ── 8.10  Usage example: original assignee from each representation (family-grouped split) ──
# The label is assignee, the original applicant, not current_assignee: a transfer of the patent does not change
# its text, so a current-assignee label would ask for information that the text does not hold. Companies keep
# fixed drafting habits, so the task partly recognises vocabulary; it is an example, not validation evidence.
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

# MIN_ASSIGNEE_N (minimum records per assignee class) is set in 1.3.

_src = (pd.read_csv(CLEAN_PATH, usecols=["patent_id", "assignee"], dtype=str)
          .fillna("").drop_duplicates("patent_id"))
lab = (valid_df[["patent_id"]].astype({"patent_id": str})
       .merge(_src, on="patent_id", how="left").fillna(""))
if len(lab) != len(valid_df):
    raise RuntimeError(f"The row count changed after the merge ({len(lab):,} vs {len(valid_df):,}): "
                       "make sure that patent_id is unique in the source file.")
_hit = int((lab["assignee"] != "").sum())
print(f"assignee coverage: {_hit:,}/{len(lab):,} ({_hit/len(lab)*100:.1f}%)")
if _hit == 0:
    raise RuntimeError("The patent_id of valid_df and the candidate file do not match at all.")

_cnt  = lab.loc[lab["assignee"] != "", "assignee"].value_counts()
_keep = set(_cnt[_cnt >= MIN_ASSIGNEE_N].index)
mask  = lab["assignee"].isin(_keep).values
sub   = lab[mask].reset_index(drop=True)

le = LabelEncoder(); y = le.fit_transform(sub["assignee"].values)
g  = build_family_groups(sub["patent_id"].values)
ix = np.arange(len(sub))
tr, te, y_tr, y_te = family_train_test_split(ix, y, g)

_maj = float(sub["assignee"].value_counts(normalize=True).iloc[0])
print(f"n={len(sub):,}/{len(valid_df):,} | classes {len(_keep)} (threshold >= {MIN_ASSIGNEE_N})"
      f" | train {len(tr):,} | test {len(te):,} (family-grouped split)")
print(f"### Feasibility check: share of the largest class {_maj*100:.1f}%"
      f"  <- an accuracy below this value means that the representation gives no information")

REPS = {k: v for k, v in REPRESENTATIONS.items() if v is not None}
rows = []
for rep, X in REPS.items():
    Xs = X[mask]
    with timed(f"8.10  {rep}  assignee prediction", n_items=len(tr), quiet=True) as _t:
        clf = LogisticRegression(max_iter=2000, C=1.0)
        _ = clf.fit(Xs[tr], y_tr)
        yp = clf.predict(Xs[te])
    rows.append({"representation": rep, "dim": int(Xs.shape[1]),
                 "accuracy": round(accuracy_score(y_te, yp), 4),
                 "f1_macro": round(f1_score(y_te, yp, average="macro",
                                            labels=np.unique(y_te), zero_division=0), 4),
                 "n_records": int(len(sub)), "n_classes": int(len(_keep)),
                 "n_test": int(len(te)), "majority_share": round(_maj, 4),
                 "fit_sec": round(_t.seconds, 1)})

print(f"\n  {'representation':20}{'dim':>6}{'accuracy':>11}{'macro-F1':>11}")
for r in rows:
    print(f"  {r['representation']:20}{r['dim']:>6}{r['accuracy']:>11.4f}{r['f1_macro']:>11.4f}")
print("\n(One table is enough for the example; no interval estimates between representations)")

pd.DataFrame(rows).to_csv(os.path.join(SAVE_DIR, "developer_demo_comparison.csv"), index=False)
pd.DataFrame({"patent_id": sub.loc[te, "patent_id"].values,
              "assignee": le.inverse_transform(y_te),
              "predicted": le.inverse_transform(yp)}
             ).to_csv(os.path.join(SAVE_DIR, "developer_demo_predictions.csv"), index=False)
print("saved -> developer_demo_comparison.csv, developer_demo_predictions.csv")

### 8.11–8.14  流暢度與相對 abstract 的相似度

GPT-2 perplexity 用來篩出三個資料層中破碎的文字。SPECTER2 餘弦相似度與 BERTScore（deberta-xlarge-mnli）把兩個資料層和 abstract 比較。這些資料層本來就要帶進 abstract 沒有的內容，所以相似度較低是預期的結果，照實報告。8.13 報告五種表徵在每個主題內的平均相似度。

In [ ]:
# ── 8.11  GPT-2 perplexity of the three summary layers (distribution and outliers) ──
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

gpt2_tok   = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").eval().to(DEVICE)

def calc_ppl(texts):
    ppls = []
    for text in tqdm(texts, desc="PPL"):
        text = str(text).strip()
        if not text:
            ppls.append(float("nan")); continue
        enc = gpt2_tok(text, return_tensors="pt", truncation=True, max_length=512)
        ids = enc["input_ids"].to(DEVICE)
        with torch.no_grad():
            loss = gpt2_model(ids, labels=ids).loss
        ppls.append(torch.exp(loss).item())
    return np.array(ppls)

PPL_SETS = [(n, c) for n, c in (("direct-brief", "baseline_summary"),
                                ("direct-full",  "baseline_matched_summary"),
                                ("synthesized",  "final_summary"))
            if c in valid_df.columns and valid_df[c].astype(str).str.len().gt(0).any()]
with timed("8.11  GPT-2 perplexity (summary layers)", n_items=len(PPL_SETS) * len(valid_df)):
    ppl = {n: calc_ppl(valid_df[c].tolist()) for n, c in PPL_SETS}

names = list(ppl)
print(f"\n{'':10}" + "".join(f"{n:>14}" for n in names))
for stat, fn in [("mean", np.nanmean), ("median", np.nanmedian),
                 ("p90", lambda a: np.nanpercentile(a, 90)),
                 ("min", np.nanmin), ("max", np.nanmax)]:
    print(f"{stat:10}" + "".join(f"{fn(ppl[n]):>14.2f}" for n in names))

for n in names:
    _anom = int((np.nan_to_num(ppl[n]) > 200).sum())
    print(f"{n} PPL > 200 (manual inspection of each record): {_anom} records")
_out = pd.DataFrame({"patent_id": valid_df["patent_id"]})
for n in names:
    _out[f"ppl_{n.replace('-', '_')}"] = ppl[n]
_out.to_csv(os.path.join(SAVE_DIR, "ppl_summaries.csv"), index=False)
print("saved -> ppl_summaries.csv")

free_gpu("gpt2_model", "gpt2_tok")

In [ ]:
# ── 8.12  Cosine similarity of two summary layers to the abstract (SPECTER2) ──
# cos(summary embedding, abstract embedding); the summary side has no title.
from sklearn.metrics.pairwise import cosine_similarity   # 8.13 uses it
if "embed_batch" not in globals():
    raise RuntimeError("Run 6.12 first (SPECTER2 and embed_batch).")

with timed("8.12  SPECTER2 embedding (direct-brief, without title)", n_items=len(valid_df)):
    E_dir = embed_batch(valid_df["baseline_summary"].astype(str).tolist(),
                        desc="cos_to_abstract [direct-brief]")
with timed("8.12  SPECTER2 embedding (synthesized, without title)", n_items=len(valid_df)):
    E_two = embed_batch(valid_df["final_summary"].astype(str).tolist(),
                        desc="cos_to_abstract [synthesized]")
with timed("8.12  SPECTER2 embedding (abstract)", n_items=len(valid_df)):
    E_abs = embed_batch(valid_df["abstract"].astype(str).tolist(),
                        desc="cos_to_abstract [abstract]")

cos_to_abstract = {"direct-brief": np.einsum("ij,ij->i", E_dir, E_abs),
                   "synthesized":  np.einsum("ij,ij->i", E_two, E_abs)}
names = list(cos_to_abstract)
print(f"\n{'':10}" + "".join(f"{n:>13}" for n in names))
for stat, fn in [("mean", np.mean), ("median", np.median),
                 ("min", np.min), ("max", np.max)]:
    print(f"{stat:10}" + "".join(f"{fn(cos_to_abstract[n]):>13.4f}" for n in names))

pd.DataFrame({"patent_id": valid_df["patent_id"],
              "cos_to_abstract_direct": cos_to_abstract["direct-brief"],
              "cos_to_abstract_twostage": cos_to_abstract["synthesized"]}).to_csv(
    os.path.join(SAVE_DIR, "abstract_cosine.csv"), index=False)
print("saved -> abstract_cosine.csv")

free_gpu("specter_model", "specter_tokenizer")

In [ ]:
# ── 8.13  Mean similarity within each topic, for each representation ────────
# The only measurement of section 8 that uses the topics; it runs on the topic_mask subset of 6.11.
def intra_cluster_sim_per_topic(X, labels):
    topic_sims = {}
    for label in set(labels):
        idx = [i for i, l in enumerate(labels) if l == label]
        if len(idx) < 2:
            continue
        sim = cosine_similarity(X[idx])
        n = len(idx)
        topic_sims[label] = np.mean([sim[i][j] for i in range(n) for j in range(i+1, n)])
    return topic_sims

labels_topic = valid_df.loc[topic_mask, "merged_topic"].values
REP_X = {k: v[topic_mask] for k, v in REPRESENTATIONS.items() if v is not None}
print(f"sample {topic_mask.sum():,} records | topics {len(set(labels_topic))}")
with timed("8.13  Within-topic similarity (representations)", n_items=3 * len(labels_topic)):
    sims = {k: intra_cluster_sim_per_topic(X, labels_topic) for k, X in REP_X.items()}

common = set.intersection(*(set(v) for v in sims.values()))
print("overall mean intra-topic similarity: " +
      " | ".join(f"{k} {np.mean([sims[k][t] for t in common]):.4f}" for k in REP_X))
print()
for t in sorted(common):
    print(f"  {t[:40]:40}" + "".join(f" {sims[k][t]:.4f}" for k in REP_X))
pd.DataFrame([{"merged_topic": t,
               **{k.split()[0].lower(): round(sims[k][t], 4) for k in REP_X}}
              for t in sorted(common)]).to_csv(
    os.path.join(SAVE_DIR, "intra_topic_similarity.csv"), index=False)
print("\nsaved -> intra_topic_similarity.csv")

In [ ]:
# ── 8.14  BERTScore of two summary layers against the abstract ───────────────
from bert_score import score as bert_score

refs = valid_df["abstract"].tolist()
HYPS = [("direct-brief", valid_df["baseline_summary"].tolist()),
        ("synthesized",  valid_df["final_summary"].tolist())]

with timed("8.14  BERTScore (deberta-xlarge-mnli, summary layers)", n_items=2 * len(valid_df)):
    bs = {}
    for name, hyps in HYPS:
        P, R, F = bert_score(hyps, refs, lang="en", batch_size=64, device=DEVICE,
                             model_type="microsoft/deberta-xlarge-mnli", verbose=False)
        bs[name] = (P, R, F)

names = [n for n, _ in HYPS]
print(f"\n{'':12}" + "".join(f"{n:>14}" for n in names))
for row, j in [("Precision", 0), ("Recall", 1), ("F1", 2)]:
    print(f"{row:12}" + "".join(f"{bs[n][j].mean():>14.4f}" for n in names))

### 第 8 節結果（釋出版本）

分面預算涵蓋了大部分較短的段落，對 detailed description 則只涵蓋開頭：background 有 4.7%、summary 有 7.3% 超過預算，detailed description 則是 99.9%，預算平均只涵蓋其原文的 10.2%。

摘要確實帶了 abstract 沒有的內容。每筆紀錄平均有 28.3 個候選詞；synthesized 層平均含 1.71 個，80.9% 的紀錄至少含一個；direct-full 是 0.68 個與 46.9%，direct-brief 是 0.72 個與 49.3%。讀同樣輸入的 synthesized 與 direct-full 相差每筆 1.03 個詞（95% 區間 1.00 到 1.07）。深區詞有 5.28% 進入 synthesized 摘要，direct-full 是 1.42%，direct-brief 是 2.92%；對照窗則維持在 1.9% 到 2.5% 之間。單純讀得更多並不會提高這個比例：direct-full 比 direct-brief 低 1.50 個百分點。只有 synthesized 層含有的長詞中，54.6% 在全文找不到；這是上限，捏造的內容由人工稽核直接量測。

配對檢查顯示摘要確實對應到自己的紀錄：各資料層對自己的 abstract 的 ROUGE-1 F1 在 0.383 到 0.392 之間，對錯位的 abstract 是 0.188 到 0.198。三個資料層合計有 143 篇摘要（其中 synthesized 72 篇）對自己的 abstract 不高於錯位的 abstract，或低於 0.10，列在 `pairing_integrity_flags.csv`。平均 perplexity 在 synthesized 層是 41.9，兩個 direct 層都是 49.7，沒有任何摘要超過 200。CPC 任務中，各表徵的 micro-F1 都在 0.605 到 0.619 之間，synthesized 比 abstract 高 0.014（區間 0.004 到 0.023）。申請人任務的準確率在 0.529 到 0.568 之間，最大類別的占比是 21.3%。

## 9  釋出檔與計算成本

9.1 依本次執行實際使用的物件寫出 prompt、詞表、抽詞函式與三個設定檔，確保釋出的資產就是實際執行的版本。9.2 列出本次執行計時的步驟，並寫出釋出用的成本表；沿用的生成步驟採用沿用執行的 log 中該步驟的完整生成時間，沒有時才用第一次完整生成（2026-08-16）的實測時間。9.3 把釋出檔寫到 `SAVE_DIR/release/`，分成三層：來源與規則衍生的檔、模型生成與衍生的檔、流程紀錄。9.4 列出 `SAVE_DIR` 裡的所有檔案。

In [ ]:
# ── 9.1  Release assets: prompts, term lists and settings, written from the objects of this run ──
import json, os, re

ASSET_DIR = os.path.join(SAVE_DIR, "release_assets")
for _d in ("prompts", "terminology", "config"):
    os.makedirs(os.path.join(ASSET_DIR, _d), exist_ok=True)

_written, _skipped = [], []

def _put(rel, text, why="not defined (the section was not run)"):
    """Write a text to its own file; text None means that the object is not yet defined."""
    if text is None:
        _skipped.append((rel, why)); return
    p = os.path.join(ASSET_DIR, rel)
    with open(p, "w", encoding="utf-8") as f:
        f.write(text if str(text).endswith("\n") else str(text) + "\n")
    _written.append((rel, len(str(text))))

_G = globals()
_sp = _G.get("SECTION_PROMPTS") or {}

# ── prompts: one file each; {placeholder} kept as it is ──────────────────
_put("prompts/stage1_background.txt",            _sp.get("BACKGROUND"))
_put("prompts/stage1_summary.txt",               _sp.get("SUMMARY"))
_put("prompts/stage1_detailed_description.txt",  _sp.get("DETAILED DESCRIPTION"))
_put("prompts/stage2_cot_reasoning.txt",         _G.get("COT_REASONING_PROMPT"))
_put("prompts/stage2_final_synthesis.txt",       _G.get("FINAL_SYNTHESIS_PROMPT_V2"))
_put("prompts/stage2_compression_optional.txt",  _G.get("COMPRESSION_PROMPT"))
_put("prompts/single_pass_baseline.txt",         _G.get("DIRECT_PROMPT"))
_put("prompts/cluster_labeling.txt",             _G.get("CLUSTER_LABEL_PROMPT"))

# ── terminology: the three term lists + the extractor ───────────────────
_wl = _G.get("HPC_WHITELIST")
_sw = _G.get("TITLE_STOPWORDS")
_gs = _G.get("GENERIC_SUFFIXES")
_put("terminology/hpc_whitelist.txt",
     None if _wl is None else
     ("# High-performance computing whitelist, tier 1 of the must-keep term strategy.\n"
      f"# {len(_wl)} terms, one per line, lower-case, matched case-insensitively.\n"
      + "\n".join(sorted(_wl))))
_put("terminology/title_stopwords.txt",
     None if _sw is None else
     ("# Words excluded when tier 2 harvests informative terms from the patent title.\n"
      f"# {len(_sw)} terms, one per line.\n" + "\n".join(sorted(_sw))))
_put("terminology/generic_suffixes.txt",
     None if _gs is None else
     ("# Regular expression (case-insensitive) rejecting morphologically generic words\n"
      "# in tiers 2 and 3. Anchored at the end of the token.\n" + _gs.pattern))

# Extractor: the source code of the function itself, so that it is the definition of 5.3
_why_fn = "not defined (the section was not run)"
try:
    import inspect
    _fn_src = inspect.getsource(_G["extract_tech_terms"])
except KeyError:
    _fn_src = None
except Exception as _e:                       # the function exists, but its source code is not available
    _fn_src, _why_fn = None, (
        f"The function is defined, but inspect cannot get its source code ({type(_e).__name__}). "
        "It is not rewritten from constants, so that the released extractor does not differ from the version that actually ran: "
        "rerun this cell in a normal notebook session.")
_put("terminology/extract_tech_terms.py",
     None if _fn_src is None else
     ('"""Must-keep term extractor (tier 1 whitelist > tier 2 title > tier 3 long words), cap 10.\n'
      'Standalone copy of the single definition in section 5.3 of the pipeline notebook.\n'
      'Run from this directory; the three lists are loaded from the files beside it.\n"""\n'
      "import re\n\n"
      "HPC_WHITELIST = {w.strip() for w in open('hpc_whitelist.txt') if w.strip() and not w.startswith('#')}\n"
      "TITLE_STOPWORDS = {w.strip() for w in open('title_stopwords.txt') if w.strip() and not w.startswith('#')}\n"
      "GENERIC_SUFFIXES = re.compile(\n"
      "    [l for l in open('generic_suffixes.txt') if not l.startswith('#')][0].strip(), re.I)\n\n"
      + _fn_src), why=_why_fn)

# ── config: generation ─────────────────────────────────────────────────
_matched = _G.get("MATCHED_BUDGETS") or {"background": 1500, "summary": 1500, "detail": 2000}
_generation = {
    "random_state": _G.get("RANDOM_STATE"),
    "min_section_chars": _G.get("MIN_LEN"),
    "models": {
        "stage1_facets":   {"id": S1_MODEL_ID, "precision": "4-bit NF4" if S1_4BIT else str(GEN_DTYPE).replace("torch.", ""),
                            "batch": _G.get("S1_GPU_BATCH")},
        "stage2_condense": {"id": S2_MODEL_NAME,
                            "precision": str(GEN_DTYPE).replace("torch.", ""), "batch": _G.get("S2_GPU_BATCH")},
        "single_pass":     {"id": S2_MODEL_NAME,
                            "precision": str(GEN_DTYPE).replace("torch.", ""), "batch": _G.get("BASE_BATCH")},
        "embedding":       {"id": _G.get("SPECTER2_MODEL", "allenai/specter2_base"),
                            "adapter": "allenai/specter2 (proximity)",
                            "dim": 768, "max_input_tokens": 512},
    },
    "decoding": {
        "do_sample": False, "repetition_penalty": 1.15,
        "max_new_tokens": {"stage1": 200, "stage2": 300, "single_pass": 300, "cluster_label": 100},
        "note": "greedy for every generated set, so no difference between sets is sampling noise",
    },
    "input_budgets_tokens": {
        "synthesized_facets": {"background": 1500, "summary": 1500,
                               "detailed_description": 2000, "total_cap_approx": 5000},
        "direct_brief":       {"background": 800, "summary": 400,
                               "detailed_description": 800, "total_cap_approx": 2000},
        "direct_full":        {"background": _matched["background"], "summary": _matched["summary"],
                               "detailed_description": _matched["detail"], "total_cap_approx": 5000,
                               "realised_prompt_tokens_median": _G.get("DIRECT_FULL_PROMPT_MEDIAN")},
    },
    "set_differences": {
        "direct_full_vs_direct_brief": "input budget only",
        "synthesized_vs_direct_full": "a separate facet step (Llama-3.1-8B-Instruct, 4-bit; inputs counted in Llama tokens), a reasoning call (up to 350 new tokens) before the final call, a different prompt template, and up to ten terms to keep in both calls",
        "held_constant_in_final_call": ["model", "precision", "decoding", "repetition_penalty",
                          "max_new_tokens", "target length 5-7 sentences"],
        "note": "the three sets are released as parallel data layers; no set is ranked above another",
    },
    "must_keep_terms": {"tiers": ["whitelist", "title", "long words (>=12 chars)"],
                        "cap_per_patent": 10, "applied_to": ["synthesized only"]},
    "summary_fallback": {"field": "summary",
                         "rule": "first 2000 characters of detailed_description when summary has fewer than 100 characters and detailed_description has at least 100 characters",
                         "affected_share": round(float(m1["summary_is_substituted"].mean()), 3)},
}
_put("config/generation.json", json.dumps(_generation, indent=2, ensure_ascii=False))

# ── config: clustering (read the live objects when they are available) ───
_cl = _G.get("clusterer")
_um = _G.get("umap_3d")
_put("config/clustering.json", json.dumps({
    "umap": {"n_components": getattr(_um, "n_components", 3),
             "n_neighbors": getattr(_um, "n_neighbors", 15),
             "min_dist": getattr(_um, "min_dist", 0.0),
             "metric": getattr(_um, "metric", "cosine"),
             "random_state": _G.get("RANDOM_STATE"),
             "low_memory": getattr(_um, "low_memory", False)},
    "hdbscan": {"min_cluster_size": getattr(_cl, "min_cluster_size", 30),
                "min_samples": getattr(_cl, "min_samples", 10),
                "metric": getattr(_cl, "metric", "euclidean"),
                "cluster_selection_method": getattr(_cl, "cluster_selection_method", "eom"),
                "prediction_data": bool(getattr(_cl, "prediction_data", True)),
                "gen_min_span_tree": bool(getattr(_cl, "gen_min_span_tree", True))},
    "selection_rule": ("the only setting in the released six-point sweep with DBCV > 0.3; "
                       "Silhouette and noise share are secondary criteria"),
    "reported_thresholds": {"silhouette": "> 0.3", "dbcv": "> 0.3", "noise_share": "< 0.20"},
    "sweep_file": "hdbscan_param_sweep.csv",
    "topic_merge": {"method": "average-linkage agglomerative clustering of the cluster centroids on cosine distance (section 6.9)",
                    "n_topics": _G.get("N_TARGET"), "min_topic_records": _G.get("MIN_CLASS_N"),
                    "small_topic_rule": "a topic with fewer than min_topic_records records joins the topic with the most similar centroid",
                    "label_rule": "the usable label of the largest member cluster that a larger topic has not taken",
                    "renamed_topics": len(_G.get("RENAME") or {}),
                    "worksheet": "merge_map_worksheet.csv",
                    "note": "rule-based; the worksheet lists the merge so that users can audit or replace it"},
}, indent=2, ensure_ascii=False))

# ── config: corpus rule ────────────────────────────────────────────────
_re_obj = _G.get("EXPANSION_RE")
_put("config/corpus_rule.json", json.dumps({
    "inclusion_regex": _re_obj.pattern if _re_obj is not None
                       else r"high[\s\-]{0,2}performance[\s\-]{0,2}comput",
    "regex_flags": "IGNORECASE",
    "scanned_fields": _G.get("TEXT_FIELDS", ["title", "abstract", "background", "summary",
                                             "detailed_description", "claims_text"]),
    "required_non_empty_fields": ["title", "abstract", "background",
                                  "detailed_description", "claims_text"],
    "exempt_field": {"summary": "not a required component of a granted specification"},
    "funnel": [{"stage": "input file (query term HPC; records using the abbreviation in unrelated senses removed)", "n": N_INPUT},
               {"stage": "five-field completeness", "n": N_COMPLETE},
               {"stage": "expansion-phrase rule", "n": N_RELEASED},
               {"stage": "excluded by the rule, released in full", "n": N_EXCLUDED}],
    "portability_note": ("the abbreviation / expansion-phrase / field-completeness funnel "
                         "transfers to any technical domain delimited by an abbreviation; "
                         "replace the regex and the required-field list"),
}, indent=2, ensure_ascii=False))

print(f"release_assets -> {ASSET_DIR}")
for rel, sz in _written:
    print(f"  {rel:46} {sz:>6} chars")
if _skipped:
    print("\n  Skipped:")
    for rel, why in _skipped:
        print(f"    {rel:44} {why}")
else:
    print(f"\n  All {len(_written)} files written")

In [ ]:
# ── 9.2  Run-time tables: this run, and the cost table of the release ─────────
# runtime_log.csv holds the timed steps of this run. For the release table, a generation step that this run
# did not perform (its checkpoint was reused) takes the full generation of that step recorded in the log of the
# reused run (6,535 records, at least 60 s), or else the first full generation run on one A100-SXM4-80GB
# (2026-08-16); every other step keeps the time measured in this run.
rt = pd.read_csv(RUNTIME_PATH)
rt["_sec"] = rt["step"].astype(str).str.split().str[0]
rt["note"] = rt["note"].astype(object)

_FIRST_PASS = {"4.3": 17075, "5.6": 14243, "5.10": 13253, "5.11": 28524}   # seconds; 4:44:35, 3:57:23, 3:40:53, 7:55:24
_GEN_NAME   = {"4.3": "4.3  Facet generation (Llama-3.1-8B-Instruct)",
               "5.6": "5.6  Synthesized summary generation (Qwen2.5-14B-Instruct)",
               "5.10": "5.10  direct-brief generation (Qwen2.5-14B-Instruct, single prompt)",
               "5.11": "5.11  direct-full generation (Qwen2.5-14B-Instruct, facet budgets)"}
# Full generations in the log of the reused run. A log of the earlier notebook numbers these steps
# 3.3, 4.6, 7.1b and 7.1b′; it is recognised by its section 7.1b.
_measured = {}
_reused_rt_path = os.path.join(SAVE_DIR, "reused_runtime_log.csv")
if os.path.exists(_reused_rt_path):
    _old = pd.read_csv(_reused_rt_path)
    _old_sec = _old["step"].astype(str).str.split().str[0]
    if _old_sec.isin(["7.1b", "7.1b′"]).any():
        _old_sec = _old_sec.replace({"3.3": "4.3", "4.6": "5.6", "7.1b": "5.10", "7.1b′": "5.11"})
    _old_n = pd.to_numeric(_old["n_items"], errors="coerce").fillna(0)
    for _s in _FIRST_PASS:
        _full = _old[(_old_sec == _s) & (_old_n >= N_RELEASED) & (_old["seconds"] >= 60)]
        if len(_full):
            _measured[_s] = _full.iloc[-1]

_n_it = pd.to_numeric(rt["n_items"], errors="coerce").fillna(0)
rel = rt.copy()
if PROFILE == "full":
    for _s, _secs in _FIRST_PASS.items():
        _m = (rel["_sec"] == _s) & (_n_it == 0)          # the step ran, but generated nothing (reused checkpoint)
        if not _m.any():
            continue
        if _s in _measured:
            _r = _measured[_s]
            _vals = [float(_r["seconds"]), _hms(_r["seconds"]), int(float(_r["n_items"])), float(_r["ms_per_item"]),
                     str(_r["hardware"]), "full generation recorded in the log of the reused run; reused in this run"]
        else:
            _vals = [_secs, _hms(_secs), N_RELEASED, round(_secs / N_RELEASED * 1000, 1),
                     "GPU NVIDIA A100-SXM4-80GB 85GB", "first full generation run (2026-08-16); reused in this run"]
        rel.loc[_m, ["step", "seconds", "hms", "n_items", "ms_per_item", "hardware", "note"]] = [_GEN_NAME[_s]] + _vals
_by_sec = rel.groupby("_sec")["seconds"].sum()
_syn = float(_by_sec.get("4.3", 0) + _by_sec.get("5.6", 0))
_dir = float(_by_sec.get("5.10", 0) + _by_sec.get("5.11", 0))
_all = float(rel["seconds"].sum()); _oth = _all - _syn - _dir
print(f"Cost of the release: synthesized path {_hms(_syn + _oth)} ({(_syn + _oth) / N_RELEASED:.1f} s per record); "
      f"all three summary layers {_hms(_all)} ({_all / N_RELEASED:.1f} s per record); other steps {_hms(_oth)}")
rel = rel.drop(columns="_sec").reset_index(drop=True)
rel.to_csv(os.path.join(SAVE_DIR, "runtime_log_release.csv"), index=False)
print("saved -> runtime_log_release.csv (the working runtime_log.csv is not changed)\n")

rt = rt.drop(columns="_sec")
rt["hours"] = (rt["seconds"] / 3600).round(3)
print("Timed steps of this run:")
print(f"{'step':58} {'hh:mm:ss':>10} {'n':>9} {'ms/item':>11}")
print("-" * 92)
for _, r in rt.iterrows():
    n   = "" if pd.isna(r["n_items"])     else f"{int(r['n_items']):,}"
    mpi = "" if pd.isna(r["ms_per_item"]) else f"{r['ms_per_item']:,.1f}"
    print(f"{str(r['step'])[:58]:58} {r['hms']:>10} {n:>9} {mpi:>11}")
print("-" * 92)
print(f"{'total of the timed steps':58} {_hms(rt['seconds'].sum()):>10}")
for hw in rt["hardware"].unique():
    sub = rt[rt["hardware"] == hw]
    print(f"  {hw}: {sub['seconds'].sum()/3600:.2f} h ({len(sub)} steps)")

# The same release table in Markdown, for a README or a report
print("\n| step | wall-clock | items | ms/item | hardware |")
print("|---|---|---|---|---|")
for _, r in rel.iterrows():
    n   = "-" if pd.isna(r["n_items"])     else f"{int(r['n_items']):,}"
    mpi = "-" if pd.isna(r["ms_per_item"]) else f"{r['ms_per_item']:,.1f}"
    print(f"| {r['step']} | {r['hms']} | {n} | {mpi} | {r['hardware']} |")

In [ ]:
# ── 9.3  Write the release files to SAVE_DIR/release/ (working files are not changed) ──
# Three layers:
#   Layer 1 source and rule-derived: hpc_patenttrace_corpus.csv, excluded_no_expansion.csv, released_corpus_ids.csv,
#                          evidence_flags.csv, ownership_events.csv, litigation_family.csv
#   Layer 2 model-generated and derived: facets.csv, summaries_synthesized.csv, summaries_direct_brief.csv,
#                          summaries_direct_full.csv, embeddings/, cluster_results.csv, topic_summary.csv, merge_map_worksheet.csv
#   Layer 3 workflow assets and records: release_assets/, runtime_log_release.csv, tables/, usage_*.csv, audit_*.csv
# Normalisation: list columns become JSON; the topic is split into topic_id and topic_label; the facets get
# what_facet_source, sections_usable and must_keep_terms. The family and litigation notes are written in English
# by section 2; the replacements below only convert files of earlier runs. Column definitions: the data dictionary.
import os, re, json, ast, shutil, glob
import numpy as np, pandas as pd

RELEASE_DIR = os.path.join(SAVE_DIR, "release")
for _d in ("embeddings", "tables"):
    os.makedirs(os.path.join(RELEASE_DIR, _d), exist_ok=True)
_manifest = []

def _save(frame, name):
    p = os.path.join(RELEASE_DIR, name)
    frame.to_csv(p, index=False)
    _manifest.append({"file": name, "rows": len(frame), "columns": len(frame.columns),
                      "MB": round(os.path.getsize(p) / 1e6, 2)})
    print(f"  {name:40} {len(frame):>6,} × {len(frame.columns):>3}  {os.path.getsize(p) / 1e6:8.2f} MB")

LIST_COLS = ["ipc_description", "citations", "applications_claiming_priority", "legal_events"]
_LIT_EN   = [("IPR\u8b49\u66f8", "IPR certificate"), ("\u518d\u5be9\u67e5", "reexamination"), ("\u5730\u9662", "district court "), ("\u4ef6(", " cases (")]

def _to_json(x):
    x = "" if pd.isna(x) else str(x)
    if not x.strip():
        return ""
    try:
        return json.dumps(ast.literal_eval(x), ensure_ascii=False)
    except Exception:
        return x                       # already JSON or cannot be parsed: kept as it is

def _normalise(frame):
    out = frame.copy()
    for c in LIST_COLS:
        if c in out.columns:
            out[c] = out[c].map(_to_json)
    if "family" in out.columns:
        out["family"] = out["family"].fillna("").astype(str).str.replace(r"\((\d+)\u4ef6\)", r"(n=\1)", regex=True)
    if "litigation" in out.columns:
        _l = out["litigation"].fillna("").astype(str)
        for k, v in _LIT_EN:
            _l = _l.str.replace(k, v, regex=False)
        out["litigation"] = _l
    return out

def _ipc3(frame):
    return add_ipc3(frame) if "add_ipc3" in globals() else frame.assign(main_ipc_3=frame["main_ipc_4"].astype(str).str[:3])

print("Layer 1 ─ source and rule-derived")
_ids = pd.read_csv(os.path.join(SAVE_DIR, "released_corpus_ids.csv"), dtype=str)["patent_id"]
_idset = set(_ids)
_raw = pd.read_csv(CLEAN_PATH, dtype=str, keep_default_na=False, low_memory=False).drop_duplicates("patent_id")
_main = _raw[_raw["patent_id"].isin(_idset)].reset_index(drop=True)
assert len(_main) == len(_idset) == N_RELEASED, f"the main file has {len(_main):,} records and the identifier list {len(_idset):,}; they do not agree"
_save(_normalise(_ipc3(_main)), "hpc_patenttrace_corpus.csv")

_REQ = ["title", "abstract", "background", "detailed_description", "claims_text"]
_complete = np.logical_and.reduce([_raw[c].str.strip().str.len().gt(0).values for c in _REQ])
_excl = _raw[_complete & ~_raw["patent_id"].isin(_idset)].reset_index(drop=True)
assert len(_excl) == N_EXCLUDED, f"excluded records {len(_excl):,} differ from 3.5 ({N_EXCLUDED:,})"
_save(_normalise(_ipc3(_excl)), "excluded_no_expansion.csv")
del _raw

_save(pd.DataFrame({"patent_id": _ids}), "released_corpus_ids.csv")
for _f in ("evidence_flags.csv", "ownership_events.csv"):
    _t = pd.read_csv(os.path.join(SAVE_DIR, _f), dtype=str, keep_default_na=False)
    assert set(_t["patent_id"]) == _idset, f"the identifiers of {_f} do not agree with the main file"
    _save(_t, _f)
_lf = os.path.join(SAVE_DIR, "litigation_family.csv")
if os.path.exists(_lf):
    _save(_normalise(_ipc3(pd.read_csv(_lf, dtype=str, keep_default_na=False))), "litigation_family.csv")

print("Layer 2 ─ model-generated and derived")
_s1 = pd.read_csv(os.path.join(SAVE_DIR, "stage1_summaries.csv"), dtype=str, keep_default_na=False)
_fp = pd.read_csv(os.path.join(SAVE_DIR, "patent_full_pipeline.csv"), dtype=str, keep_default_na=False,
                  usecols=lambda c: c in {"patent_id", "title", "summary_is_substituted", "sections_usable",
                                          "cot_reasoning", "final_summary"})
assert set(_s1["patent_id"]) == _idset and set(_fp["patent_id"]) == _idset, "the identifiers of the facet or synthesis file do not agree with the main file"
_fac = _s1[["patent_id", "bg_summary", "inv_summary", "tech_summary"]].merge(
    _fp[["patent_id", "title", "summary_is_substituted", "sections_usable"]], on="patent_id", how="left")
_fac["what_facet_source"] = np.where(_fac["summary_is_substituted"].str.lower().eq("true"),
                                     "detailed_description_first_2000_chars", "summary_section")
if "extract_tech_terms" in globals():
    _fac["must_keep_terms"] = [extract_tech_terms(t, b, i, h) for t, b, i, h in
                               zip(_fac["title"], _fac["bg_summary"], _fac["inv_summary"], _fac["tech_summary"])]
else:
    _fac["must_keep_terms"] = ""
    print("  (extract_tech_terms not defined: must_keep_terms left empty; rerun this cell in a session that ran 4.3)")
_save(_fac[["patent_id", "bg_summary", "inv_summary", "tech_summary",
            "what_facet_source", "sections_usable", "must_keep_terms"]], "facets.csv")
_save(_fp[["patent_id", "cot_reasoning", "final_summary"]], "summaries_synthesized.csv")
_db = pd.read_csv(os.path.join(SAVE_DIR, "baseline_direct.csv"), dtype=str, keep_default_na=False) \
        .rename(columns={"baseline_summary": "direct_brief_summary"})
_save(_db[["patent_id", "direct_brief_summary"]], "summaries_direct_brief.csv")
_dfu = pd.read_csv(os.path.join(SAVE_DIR, "baseline_direct_matched.csv"), dtype=str, keep_default_na=False) \
        .rename(columns={"baseline_matched_summary": "direct_full_summary"})
_save(_dfu[["patent_id", "direct_full_summary"]], "summaries_direct_full.csv")

_cl = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv"), dtype=str, keep_default_na=False)
_SENT = globals().get("NON_TOPIC_SENTINEL", "__noise__")
_RN = globals().get("RENAME", {}) or {}
_lab = _cl["merged_topic"].replace(_RN)
_sizes = _lab[_lab != _SENT].value_counts()                 # sorted by record count -> topic_id 0, 1, ...
_tid = {t: i for i, t in enumerate(_sizes.index)}
_cl["topic_id"] = _lab.map(_tid).fillna(-1).astype(int)
_cl["topic_label"] = _lab
_save(_cl, "cluster_results.csv")
_ts = pd.read_csv(os.path.join(SAVE_DIR, "topic_summary.csv"), dtype=str, keep_default_na=False)
_ts["topic_label"] = _ts["merged_topic"].replace(_RN)
_ts["topic_id"] = _ts["topic_label"].map(_tid).fillna(-1).astype(int)
_save(_ts[["topic_id", "topic_label"] + [c for c in _ts.columns if c not in ("topic_id", "topic_label", "merged_topic")]],
      "topic_summary.csv")
_mw = pd.read_csv(os.path.join(SAVE_DIR, "merge_map_worksheet.csv"), dtype=str, keep_default_na=False)
_mw["topic_label"] = _mw["auto_merged_topic"].replace(_RN)
_mw["topic_id"] = _mw["topic_label"].map(_tid).fillna(-1).astype(int)
_save(_mw, "merge_map_worksheet.csv")
for _p in sorted(glob.glob(os.path.join(SAVE_DIR, "embeddings", "*"))):
    _ = shutil.copy2(_p, os.path.join(RELEASE_DIR, "embeddings", os.path.basename(_p)))
    _manifest.append({"file": "embeddings/" + os.path.basename(_p), "rows": N_RELEASED if _p.endswith(".parquet") else "",
                      "columns": 769 if _p.endswith(".parquet") else "", "MB": round(os.path.getsize(_p) / 1e6, 2)})
print(f"  embeddings/ copied {len(glob.glob(os.path.join(RELEASE_DIR, 'embeddings', '*')))} files")

print("Layer 3 ─ workflow assets and records")
for _p in sorted(glob.glob(os.path.join(SAVE_DIR, "tables", "*.csv"))):
    _ = shutil.copy2(_p, os.path.join(RELEASE_DIR, "tables", os.path.basename(_p)))
print(f"  tables/ copied {len(glob.glob(os.path.join(RELEASE_DIR, 'tables', '*.csv')))} files")
for _pat in ("usage_*.csv", "audit_*.csv", "runtime_log_release.csv", "hdbscan_param_sweep.csv",
             "differential_exclusion.csv", "ipc_downstream_comparison.csv", "ipc_downstream_bootstrap.csv",
             "summary_metrics.csv", "pairing_integrity_flags.csv", "added_value_beyond_abstract.csv",
             "coverage_beyond_truncation.csv", "overlap_structure.csv", "not_in_full_text_per_record.csv",
             "ppl_summaries.csv", "truncation_profile.csv", "topic_classifier_benchmark.csv",
             "developer_demo_comparison.csv", "intra_topic_similarity.csv"):
    for _p in sorted(glob.glob(os.path.join(SAVE_DIR, _pat))):
        _ = shutil.copy2(_p, os.path.join(RELEASE_DIR, os.path.basename(_p)))
        _manifest.append({"file": os.path.basename(_p), "rows": "", "columns": "", "MB": round(os.path.getsize(_p) / 1e6, 2)})
if not os.path.exists(os.path.join(RELEASE_DIR, "runtime_log_release.csv")):
    print("  Note: runtime_log_release.csv does not exist; run 9.2 first, then this cell again")
_assets = os.path.join(SAVE_DIR, "release_assets")
if os.path.isdir(_assets):
    _ = shutil.copytree(_assets, os.path.join(RELEASE_DIR, "release_assets"), dirs_exist_ok=True)
    print("  release_assets/ copied (prompts, terminology, config)")
else:
    print("  Note: release_assets/ does not exist; run 9.1 first, then this cell again")

pd.DataFrame(_manifest).to_csv(os.path.join(RELEASE_DIR, "MANIFEST_release.csv"), index=False)
print(f"\nrelease -> {RELEASE_DIR}  ({len(_manifest)} items, list in MANIFEST_release.csv)")

In [ ]:
# ── 9.4  Files written to SAVE_DIR ──────────────────────────────────────────
_rows = []
for _root, _dirs, _files in os.walk(SAVE_DIR):
    for _f in _files:
        _p = os.path.join(_root, _f)
        _rows.append({"folder": os.path.relpath(_root, SAVE_DIR), "file": _f, "MB": round(os.path.getsize(_p) / 1e6, 2)})
files_df = pd.DataFrame(_rows).sort_values(["folder", "file"]).reset_index(drop=True)
print(f"{SAVE_DIR}: {len(files_df):,} files, {files_df['MB'].sum()/1e3:.2f} GB")
files_df.groupby("folder").agg(files=("file", "size"), MB=("MB", "sum")).round(1)
files_df[files_df["folder"] == "."]

## 10  執行情況報告

報告列出執行環境、沿用了哪些檔、每一節的實際耗時（取自 1.6 的逐格紀錄），以及有錯誤的格。完整版會把 17 個關鍵數字和 2026-10-11 的釋出資料集逐項比對，標出相同或不同。第 2、3 節的筆數由規則決定，必須完全相同；沿用生成結果時，分群、主題與稽核的數字也應該相同。

In [ ]:
# ── 10.1  Run report: what ran, what was reused, how long each section took ──
import importlib.metadata, sklearn
from IPython.display import display
def _ver(dist):   # version of an installed distribution (hdbscan has no __version__ attribute)
    try:
        return importlib.metadata.version(dist)
    except Exception:
        return "-"

env_df = pd.DataFrame([
    ("profile", PROFILE + (f" (SAMPLE_N = {SAMPLE_N})" if SAMPLE_N else "")),
    ("generated layers", "reused from " + REUSE_DIR if REUSE_DIR else "generated in this run"),
    ("models", f"{S1_MODEL_ID} | {S2_MODEL_NAME}" + (" (small stand-in model)" if SMALL_MODELS else "")),
    ("hardware", HARDWARE),
    ("python / torch / transformers", f"{sys.version.split()[0]} / {torch.__version__} / {_ver('transformers')}"),
    ("pandas / scikit-learn / umap / hdbscan", f"{pd.__version__} / {sklearn.__version__} / {_ver('umap-learn')} / {_ver('hdbscan')}"),
    ("input file", f"{DATA_PATH} ({N_FILE:,} records)"),
    ("output folder", SAVE_DIR),
], columns=["item", "value"])
env_df

# Wall time by section, from the per-cell log that 1.6 starts (cells 1.1-1.5 are not in it; the last run of each cell counts)
cells_df = pd.read_csv(CELL_LOG_PATH)
cells_df["section"] = cells_df["cell"].astype(str).str.extract(r"^(\d+)\.", expand=False)
cells_df = cells_df.drop_duplicates("cell", keep="last")
by_section = (cells_df.dropna(subset=["section"])
              .assign(section=lambda d: d["section"].astype(int))
              .groupby("section").agg(cells=("cell", "size"), seconds=("seconds", "sum"))
              .assign(hh_mm_ss=lambda d: d["seconds"].map(_hms)))
print(f"Total wall time of the executed cells: {_hms(cells_df['seconds'].sum())}")
by_section
_failed = cells_df[~cells_df["ok"].astype(bool)]
print(f"Cells that ended with an error: {len(_failed)}")
if len(_failed):
    display(_failed)

# Key numbers of this run; for the full profile next to the released dataset (2026-10-11)
_clu = pd.read_csv(os.path.join(SAVE_DIR, "cluster_results.csv")).fillna("")
_topic = _clu["merged_topic"][~_clu["merged_topic"].isin(["__noise__", "Off-topic / Non-HPC", "Unclassified", ""])]
_ev = pd.read_csv(os.path.join(SAVE_DIR, "evidence_flags.csv"))
_nmi = pd.read_csv(os.path.join(TBL_DIR, "e6_topic_ipc_metrics.csv")).iloc[0]["value"]
_audit = [os.path.join(SAVE_DIR, f) for f in ("audit_primary_50.csv", "audit_edge_12.csv", "audit_sentences.csv")]
key_df = pd.DataFrame([
    ("records in the input file",              N_FILE,                                21947),
    ("records used in this run",               N_RAW,                                 21947),
    ("removed: HPC has another sense",         N_RAW - N_CLEAN,                       5513),
    ("candidates",                             N_CLEAN,                               16434),
    ("five complete fields",                   N_COMPLETE,                            7158),
    ("released corpus",                        N_RELEASED,                            6535),
    ("excluded by the inclusion rule",         N_EXCLUDED,                            623),
    ("records that define HPC (sense_basis)",  int((_ev["sense_basis"] == "definition").sum()), 2831),
    ("litigated-family subset",                len(pd.read_csv(os.path.join(SAVE_DIR, "litigation_family.csv"))), 25),
    ("clusters (HDBSCAN)",                     int(_clu.loc[_clu["cluster_id"] != -1, "cluster_id"].nunique()), 51),
    ("noise records",                          int((_clu["cluster_id"] == -1).sum()), 1502),
    ("merged topics",                          int(_topic.nunique()),                 25),
    ("records with a topic",                   int(len(_topic)),                      5033),
    ("NMI, topics vs CPC subclasses",          round(float(_nmi), 4),                 0.4558),
    ("pairing flags (8.4)",                    len(pd.read_csv(os.path.join(SAVE_DIR, "pairing_integrity_flags.csv"))), 143),
    ("human-audit sample: records",            sum(len(pd.read_csv(p)) for p in _audit[:2]), 62),
    ("human-audit sample: sentences",          len(pd.read_csv(_audit[2])),           349),
], columns=["quantity", "this_run", "released_2026_10_11"])
if PROFILE == "full":
    key_df["same"] = np.where(np.isclose(key_df["this_run"].astype(float), key_df["released_2026_10_11"].astype(float)), "yes", "NO")
else:
    key_df = key_df.drop(columns="released_2026_10_11")
_fmt = lambda v: f"{v:,.4f}" if not float(v).is_integer() else f"{int(v):,}"
for _c in ("this_run", "released_2026_10_11"):
    if _c in key_df.columns:
        key_df[_c] = key_df[_c].astype(object).map(_fmt)
key_df.to_csv(os.path.join(SAVE_DIR, "run_report_key_numbers.csv"), index=False)
key_df

if EMBEDDING_CHECK:
    print(f"SPECTER2 check of the reused embeddings ({EMBEDDING_CHECK['records']} records encoded again on {HARDWARE}): "
          f"max |difference| {EMBEDDING_CHECK['max_abs_diff']:.2e}, minimum cosine {EMBEDDING_CHECK['min_cosine']:.6f}")
print("Figures:", ", ".join(sorted(os.listdir(FIG_DIR))))

### 附註

- 2026-10-11 的釋出資料集是由先前的兩份 notebook（撞字篩選與主流程）產生的，判定規則、prompt 與模型設定都和本 notebook 相同；已發布的數字來自該次執行，其工作檔就在 `REUSE_DIR`。
- 在其他硬體或套件版本上全部重新生成，即使用貪婪解碼，文字也可能不同。人工稽核對應的是釋出版本的文字，重新生成的語料需要重新稽核。
- 分類欄位（`main_ipc_4`、`main_ipc_3`）取自 Google Patents 頁面上的第一個 CPC 路徑，欄名沿用「ipc」。